# Integrated internal audit and model freezing

**Public role:** Core model-freezing and internal audit notebook.

**Relation to the manuscript:** Supports the frozen model panel and separation between development and external reliability assessment.

**Execution status of this curated copy:** Stored outputs and execution counters were removed to prevent disclosure of local paths, embedded dataset images, stale errors, and large widget state. The code was not re-executed after this mechanical curation. Reference outputs are distributed separately under `results/`.

**Path configuration:** Set `AQPR_PROJECT_ROOT` to the working directory containing `project_data/`, `models/`, and `results/`. Set `AQPR_DATA_ROOT` to the external dataset root. The default values target a Google Drive/Colab workflow and can be changed without editing the notebook logic.

**Original source notebook:** `12_integrated_internal_model_audit_and_freezing.ipynb`


In [ ]:
import os
# Cell 1: Setup, artifact audit, candidate registry, and XAI target registry

from google.colab import drive
drive.mount("/content/drive")

import json
import platform
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path(os.environ.get("AQPR_PROJECT_ROOT", "/content/drive/MyDrive/AQPR_Glaucoma"))

PROJECT_DATA_DIR = PROJECT_ROOT / "project_data"
RESULTS_DIR = PROJECT_ROOT / "results"
MODELS_DIR = PROJECT_ROOT / "models"

RESULTS_07_DIR = RESULTS_DIR / "07_hygd_glaucoma_classifier_benchmark"
RESULTS_08_DIR = RESULTS_DIR / "08_anatomical_explainability_validation"
RESULTS_09_DIR = RESULTS_DIR / "09_anatomy_ablation_and_shortcut_controls"
RESULTS_10_DIR = RESULTS_DIR / "10_anatomy_constrained_classifier_training"
RESULTS_11_DIR = RESULTS_DIR / "11_hybrid_anatomy_aware_classifier_training"
RESULTS_12_DIR = RESULTS_DIR / "12_integrated_internal_model_audit_and_freezing"

RESULTS_12_DIR.mkdir(parents=True, exist_ok=True)

NOTEBOOK07_FINAL_STATUS_JSON = RESULTS_07_DIR / "notebook07_final_status.json"

NOTEBOOK08_XAI_METHOD_REGISTRY_CSV = (
    RESULTS_08_DIR
    / "notebook08_xai_method_registry.csv"
)

NOTEBOOK08_HEATMAP_PATIENT_STATS_CSV = (
    RESULTS_08_DIR
    / "notebook08_heatmap_xai_primary_test_patient_level_statistics.csv"
)

NOTEBOOK08_OCCLUSION_PATIENT_STATS_CSV = (
    RESULTS_08_DIR
    / "notebook08_occlusion_primary_test_patient_level_statistics.csv"
)

NOTEBOOK08_DEITSMALL_XAI_STATS_CSV = (
    RESULTS_08_DIR
    / "notebook08_deitsmall_transformer_xai_patient_level_statistics.csv"
)

NOTEBOOK09_FIGURE_MANIFEST_CSV = (
    RESULTS_09_DIR
    / "notebook09_ablation_figure_manifest.csv"
)

NOTEBOOK09_FIGURE_SUMMARY_JSON = (
    RESULTS_09_DIR
    / "notebook09_ablation_figure_summary.json"
)

NOTEBOOK10_FINAL_STATUS_JSON = RESULTS_10_DIR / "notebook10_final_status.json"
NOTEBOOK10_FINAL_SYNTHESIS_CSV = RESULTS_10_DIR / "notebook10_final_synthesis.csv"
NOTEBOOK10_FINAL_MODEL_ROLES_CSV = RESULTS_10_DIR / "notebook10_final_model_roles.csv"
NOTEBOOK10_FINAL_CHECKS_CSV = RESULTS_10_DIR / "notebook10_final_status_checks.csv"

NOTEBOOK11_FINAL_STATUS_JSON = RESULTS_11_DIR / "notebook11_final_status.json"
NOTEBOOK11_FINAL_SYNTHESIS_CSV = RESULTS_11_DIR / "notebook11_final_synthesis.csv"
NOTEBOOK11_FINAL_BRANCH_SUMMARY_CSV = RESULTS_11_DIR / "notebook11_final_branch_contribution_summary.csv"
NOTEBOOK11_FINAL_CHECKS_CSV = RESULTS_11_DIR / "notebook11_final_status_checks.csv"

NOTEBOOK11_PRIMARY_SUMMARY_CSV = (
    RESULTS_11_DIR
    / "notebook11_primary_hybrid_vs_internal_baselines_summary.csv"
)

NOTEBOOK11_PRIMARY_DELTAS_CSV = (
    RESULTS_11_DIR
    / "notebook11_primary_hybrid_deltas_vs_internal_baselines.csv"
)

NOTEBOOK11_HYBRID_COMPARISON_FIGURE_SUMMARY_JSON = (
    RESULTS_11_DIR
    / "notebook11_hybrid_comparison_figure_summary.json"
)

NOTEBOOK11_BRANCH_FIGURE_SUMMARY_JSON = (
    RESULTS_11_DIR
    / "notebook11_branch_contribution_figure_summary.json"
)

INPUT_ARTIFACT_AUDIT_CSV = (
    RESULTS_12_DIR
    / "notebook12_input_artifact_audit.csv"
)

INTERNAL_CANDIDATE_REGISTRY_CSV = (
    RESULTS_12_DIR
    / "notebook12_internal_candidate_registry.csv"
)

INTERNAL_CANDIDATE_REGISTRY_JSON = (
    RESULTS_12_DIR
    / "notebook12_internal_candidate_registry.json"
)

EVIDENCE_DOMAIN_PLAN_CSV = (
    RESULTS_12_DIR
    / "notebook12_evidence_domain_plan.csv"
)

XAI_TARGET_REGISTRY_CSV = (
    RESULTS_12_DIR
    / "notebook12_xai_target_registry.csv"
)

XAI_TARGET_REGISTRY_JSON = (
    RESULTS_12_DIR
    / "notebook12_xai_target_registry.json"
)

SETUP_SUMMARY_JSON = (
    RESULTS_12_DIR
    / "notebook12_setup_summary.json"
)

required_artifacts = [
    NOTEBOOK07_FINAL_STATUS_JSON,
    NOTEBOOK10_FINAL_STATUS_JSON,
    NOTEBOOK10_FINAL_SYNTHESIS_CSV,
    NOTEBOOK10_FINAL_MODEL_ROLES_CSV,
    NOTEBOOK10_FINAL_CHECKS_CSV,
    NOTEBOOK11_FINAL_STATUS_JSON,
    NOTEBOOK11_FINAL_SYNTHESIS_CSV,
    NOTEBOOK11_FINAL_BRANCH_SUMMARY_CSV,
    NOTEBOOK11_FINAL_CHECKS_CSV,
    NOTEBOOK11_PRIMARY_SUMMARY_CSV,
    NOTEBOOK11_PRIMARY_DELTAS_CSV,
    NOTEBOOK11_HYBRID_COMPARISON_FIGURE_SUMMARY_JSON,
    NOTEBOOK11_BRANCH_FIGURE_SUMMARY_JSON,
]

optional_artifacts = [
    NOTEBOOK08_XAI_METHOD_REGISTRY_CSV,
    NOTEBOOK08_HEATMAP_PATIENT_STATS_CSV,
    NOTEBOOK08_OCCLUSION_PATIENT_STATS_CSV,
    NOTEBOOK08_DEITSMALL_XAI_STATS_CSV,
    NOTEBOOK09_FIGURE_MANIFEST_CSV,
    NOTEBOOK09_FIGURE_SUMMARY_JSON,
]

artifact_rows = []

for path in required_artifacts:
    artifact_rows.append(
        {
            "artifact": path.name,
            "path": str(path),
            "required": True,
            "exists": path.exists(),
        }
    )

for path in optional_artifacts:
    artifact_rows.append(
        {
            "artifact": path.name,
            "path": str(path),
            "required": False,
            "exists": path.exists(),
        }
    )

artifact_audit = pd.DataFrame(artifact_rows)
artifact_audit.to_csv(INPUT_ARTIFACT_AUDIT_CSV, index=False)

missing_required = artifact_audit[
    artifact_audit["required"].astype(bool)
    & (~artifact_audit["exists"].astype(bool))
]["path"].tolist()

if missing_required:
    print(artifact_audit.to_string(index=False))
    raise FileNotFoundError(f"Missing required Notebook 12 inputs: {missing_required}")

with open(NOTEBOOK07_FINAL_STATUS_JSON, "r", encoding="utf-8") as f:
    notebook07_status = json.load(f)

with open(NOTEBOOK10_FINAL_STATUS_JSON, "r", encoding="utf-8") as f:
    notebook10_status = json.load(f)

with open(NOTEBOOK11_FINAL_STATUS_JSON, "r", encoding="utf-8") as f:
    notebook11_status = json.load(f)

notebook10_checks = pd.read_csv(NOTEBOOK10_FINAL_CHECKS_CSV)
notebook11_checks = pd.read_csv(NOTEBOOK11_FINAL_CHECKS_CSV)

notebook10_synthesis = pd.read_csv(NOTEBOOK10_FINAL_SYNTHESIS_CSV)
notebook10_model_roles = pd.read_csv(NOTEBOOK10_FINAL_MODEL_ROLES_CSV)

notebook11_synthesis = pd.read_csv(NOTEBOOK11_FINAL_SYNTHESIS_CSV)
notebook11_branch_summary = pd.read_csv(NOTEBOOK11_FINAL_BRANCH_SUMMARY_CSV)

primary_summary = pd.read_csv(NOTEBOOK11_PRIMARY_SUMMARY_CSV)
primary_deltas = pd.read_csv(NOTEBOOK11_PRIMARY_DELTAS_CSV)

if not notebook10_checks["status"].astype(bool).all():
    raise RuntimeError("Notebook 10 final checks are not all PASS.")

if not notebook11_checks["status"].astype(bool).all():
    raise RuntimeError("Notebook 11 final checks are not all PASS.")

EXPECTED_MODEL_ORDER = [
    "full_image__EfficientNetB0",
    "full_image__MobileNetV3Large",
    "full_image__DeiTSmall",
    "anatomy__od_centered_crop_2p5x",
    "anatomy__od_centered_crop_4p0x",
    "anatomy__od_only_context_suppressed",
    "anatomy__outside_od_only_trained_control",
    "hybrid__full_image_plus_od_centered_2p5x_effb0",
]

MODEL_REGISTRY = {
    "full_image__EfficientNetB0": {
        "model_family": "full_image",
        "model_role": "architecture_matched_full_image_reference",
        "clinical_candidate_status": "comparator",
        "external_validation_role": "internal_reference_comparator",
        "xai_status": "available_from_notebook08_partial",
        "interpretation": "Full-image EfficientNet-B0 reference used for architecture-matched comparison.",
    },
    "full_image__MobileNetV3Large": {
        "model_family": "full_image",
        "model_role": "lightweight_full_image_finalist",
        "clinical_candidate_status": "comparator",
        "external_validation_role": "full_image_architecture_comparator",
        "xai_status": "secondary_optional",
        "interpretation": "Lightweight full-image finalist.",
    },
    "full_image__DeiTSmall": {
        "model_family": "full_image",
        "model_role": "transformer_full_image_finalist",
        "clinical_candidate_status": "comparator",
        "external_validation_role": "full_image_architecture_comparator",
        "xai_status": "available_from_notebook08_transformer_specific",
        "interpretation": "Transformer full-image finalist with transformer-specific XAI evidence.",
    },
    "anatomy__od_centered_crop_2p5x": {
        "model_family": "anatomy_constrained",
        "model_role": "primary_anatomically_parsimonious_candidate",
        "clinical_candidate_status": "candidate",
        "external_validation_role": "primary_anatomical_comparator",
        "xai_status": "pending_notebook12",
        "interpretation": "Primary OD-centered anatomically parsimonious model.",
    },
    "anatomy__od_centered_crop_4p0x": {
        "model_family": "anatomy_constrained",
        "model_role": "wider_peripapillary_context_sensitivity_model",
        "clinical_candidate_status": "candidate",
        "external_validation_role": "anatomical_context_sensitivity_comparator",
        "xai_status": "secondary_optional_notebook12",
        "interpretation": "Wider OD-centered context sensitivity model.",
    },
    "anatomy__od_only_context_suppressed": {
        "model_family": "anatomy_stress_test",
        "model_role": "trained_od_only_context_suppressed_stress_test",
        "clinical_candidate_status": "stress_test_not_primary_clinical_model",
        "external_validation_role": "stress_test_optional",
        "xai_status": "secondary_optional_notebook12",
        "interpretation": "Stress-test model trained with outside-OD context suppressed.",
    },
    "anatomy__outside_od_only_trained_control": {
        "model_family": "shortcut_control",
        "model_role": "trained_outside_od_shortcut_control",
        "clinical_candidate_status": "control_not_clinical_candidate",
        "external_validation_role": "shortcut_control",
        "xai_status": "pending_notebook12",
        "interpretation": "Control model testing whether extra-papillary signal alone is predictive.",
    },
    "hybrid__full_image_plus_od_centered_2p5x_effb0": {
        "model_family": "hybrid_anatomy_aware",
        "model_role": "primary_hybrid_candidate",
        "clinical_candidate_status": "candidate",
        "external_validation_role": "primary_hybrid_candidate",
        "xai_status": "pending_notebook12",
        "interpretation": "Hybrid model combining full-image and OD-centered branches by late feature fusion.",
    },
}


def metric_row(model_id, metric):
    block = primary_summary[
        (primary_summary["model_id"].astype(str) == model_id)
        & (primary_summary["metric"].astype(str) == metric)
    ]

    if len(block) != 1:
        raise RuntimeError(f"Metric row not found uniquely: {model_id} | {metric}")

    return block.iloc[0]


candidate_rows = []

for model_id in EXPECTED_MODEL_ORDER:
    registry = MODEL_REGISTRY[model_id]

    auroc_row = metric_row(model_id, "auroc")
    bacc_row = metric_row(model_id, "balanced_accuracy")
    brier_row = metric_row(model_id, "brier_score")

    candidate_rows.append(
        {
            "model_id": model_id,
            "model_family": registry["model_family"],
            "model_role": registry["model_role"],
            "clinical_candidate_status": registry["clinical_candidate_status"],
            "external_validation_role": registry["external_validation_role"],
            "xai_status": registry["xai_status"],
            "patient_auroc": float(auroc_row["observed"]),
            "patient_auroc_ci95_low": float(auroc_row["ci95_low"]),
            "patient_auroc_ci95_high": float(auroc_row["ci95_high"]),
            "patient_balanced_accuracy": float(bacc_row["observed"]),
            "patient_balanced_accuracy_ci95_low": float(bacc_row["ci95_low"]),
            "patient_balanced_accuracy_ci95_high": float(bacc_row["ci95_high"]),
            "patient_brier_score": float(brier_row["observed"]),
            "patient_brier_score_ci95_low": float(brier_row["ci95_low"]),
            "patient_brier_score_ci95_high": float(brier_row["ci95_high"]),
            "interpretation": registry["interpretation"],
        }
    )

candidate_registry = pd.DataFrame(candidate_rows)
candidate_registry.to_csv(INTERNAL_CANDIDATE_REGISTRY_CSV, index=False)

with open(INTERNAL_CANDIDATE_REGISTRY_JSON, "w", encoding="utf-8") as f:
    json.dump(candidate_registry.to_dict(orient="records"), f, indent=2)

evidence_domain_plan = pd.DataFrame(
    [
        {
            "evidence_domain": "discrimination",
            "primary_outputs": "patient-level AUROC, AUPRC, balanced accuracy, sensitivity, specificity",
            "main_sources": "Notebooks 07, 10, 11",
            "purpose_for_freezing": "Confirm that candidate models preserve diagnostic performance under patient-level evaluation.",
            "status": "available",
        },
        {
            "evidence_domain": "probabilistic_reliability",
            "primary_outputs": "Brier score, calibration curves, expected calibration error, reliability summaries",
            "main_sources": "Notebook 11 available; calibration curves pending in Notebook 12",
            "purpose_for_freezing": "Select models that provide reliable probabilities rather than only high AUROC.",
            "status": "partially_available_pending_calibration_curves",
        },
        {
            "evidence_domain": "anatomical_plausibility_xai",
            "primary_outputs": "OD/OC heatmap enrichment, random-control enrichment, branch-specific hybrid XAI, XAI method agreement",
            "main_sources": "Notebook 08 partially available; hybrid and anatomy-constrained XAI pending in Notebook 12",
            "purpose_for_freezing": "Verify whether high-performing models rely on anatomically plausible regions.",
            "status": "partially_available_pending_hybrid_and_anatomy_xai",
        },
        {
            "evidence_domain": "shortcut_robustness",
            "primary_outputs": "OD/OC ablation, outside-OD only retention, outside-OD trained control",
            "main_sources": "Notebooks 09 and 10",
            "purpose_for_freezing": "Identify whether performance persists using non-OD signal, indicating possible shortcut or dataset-specific signal.",
            "status": "available",
        },
        {
            "evidence_domain": "branch_contribution",
            "primary_outputs": "hybrid original vs full-branch-only vs OD-branch-only vs both-neutralized",
            "main_sources": "Notebook 11",
            "purpose_for_freezing": "Verify whether the hybrid uses both global and anatomical branches.",
            "status": "available",
        },
        {
            "evidence_domain": "patient_consistency",
            "primary_outputs": "multi-image patient aggregation, within-patient prediction dispersion, disagreement cases",
            "main_sources": "Notebooks 07, 10, 11; integrated analysis pending",
            "purpose_for_freezing": "Prefer models that are stable across multiple images from the same patient.",
            "status": "pending_integrated_analysis",
        },
        {
            "evidence_domain": "quality_robustness",
            "primary_outputs": "quality-stratified performance and probability stability",
            "main_sources": "HYGD quality score; integrated analysis pending",
            "purpose_for_freezing": "Assess whether model performance depends on image quality strata.",
            "status": "pending_integrated_analysis",
        },
        {
            "evidence_domain": "external_validation_readiness",
            "primary_outputs": "frozen model list, frozen thresholds, preprocessing contracts, final external dataset selection",
            "main_sources": "Notebook 12 final outputs",
            "purpose_for_freezing": "Prevent model or threshold selection using external validation data.",
            "status": "pending",
        },
    ]
)

evidence_domain_plan.to_csv(EVIDENCE_DOMAIN_PLAN_CSV, index=False)

xai_target_registry = pd.DataFrame(
    [
        {
            "model_id": "hybrid__full_image_plus_od_centered_2p5x_effb0",
            "xai_priority": "primary_required",
            "xai_scope": "branch_specific_hybrid_xai",
            "methods": "GradCAMPlusPlus; IntegratedGradients; AnatomicalOcclusion; BranchNeutralizationAlreadyAvailable",
            "target_regions": "OD; OC; Rim; OD_plus_OC; outside_OD; peripapillary_context",
            "analysis_unit": "locked_test_primary_strict_patients_and_images",
            "expected_outputs": "full-branch XAI; OD-centered-branch XAI; branch-specific anatomical enrichment; branch-specific occlusion",
            "reason": "Primary hybrid candidate must be anatomically audited before model freezing.",
        },
        {
            "model_id": "anatomy__od_centered_crop_2p5x",
            "xai_priority": "primary_required",
            "xai_scope": "od_centered_crop_xai",
            "methods": "GradCAMPlusPlus; IntegratedGradients; AnatomicalOcclusion",
            "target_regions": "OD; OC; Rim; peripapillary_crop_context",
            "analysis_unit": "locked_test_primary_strict_patients_and_images",
            "expected_outputs": "crop-space anatomical enrichment; crop-space occlusion; comparison against hybrid OD branch",
            "reason": "Primary anatomically parsimonious comparator.",
        },
        {
            "model_id": "anatomy__outside_od_only_trained_control",
            "xai_priority": "primary_required_control",
            "xai_scope": "shortcut_control_xai",
            "methods": "GradCAMPlusPlus; IntegratedGradients; region heatmap audit",
            "target_regions": "outside_OD; background; retinal periphery; artifacts",
            "analysis_unit": "locked_test_primary_strict_patients_and_images",
            "expected_outputs": "extra-papillary attribution map; shortcut-region audit",
            "reason": "Strong outside-OD performance requires explanation as control evidence.",
        },
        {
            "model_id": "full_image__EfficientNetB0",
            "xai_priority": "already_available_primary_reference",
            "xai_scope": "full_image_reference_xai",
            "methods": "GradCAM; GradCAMPlusPlus; IntegratedGradients; AnatomicalOcclusion",
            "target_regions": "OD; OC; Rim; OD_plus_OC; outside_OD",
            "analysis_unit": "locked_test_primary_strict_patients_and_images",
            "expected_outputs": "reuse Notebook 08 where available; add only if missing for final comparison",
            "reason": "Architecture-matched full-image reference.",
        },
        {
            "model_id": "full_image__DeiTSmall",
            "xai_priority": "already_available_transformer_reference",
            "xai_scope": "transformer_specific_xai",
            "methods": "IntegratedGradients; AttentionRollout; AnatomicalOcclusion",
            "target_regions": "OD; OC; Rim; OD_plus_OC; outside_OD",
            "analysis_unit": "locked_test_primary_strict_patients_and_images",
            "expected_outputs": "reuse Notebook 08 transformer-specific XAI",
            "reason": "Transformer full-image comparator with strong internal performance.",
        },
        {
            "model_id": "anatomy__od_centered_crop_4p0x",
            "xai_priority": "secondary_sensitivity",
            "xai_scope": "wider_od_centered_crop_xai",
            "methods": "GradCAMPlusPlus; IntegratedGradients",
            "target_regions": "OD; OC; Rim; wider_peripapillary_context",
            "analysis_unit": "locked_test_primary_strict_patients_and_images_or_subset",
            "expected_outputs": "sensitivity comparison against OD-centered 2.5x",
            "reason": "Wider anatomical context sensitivity model.",
        },
        {
            "model_id": "anatomy__od_only_context_suppressed",
            "xai_priority": "secondary_stress_test",
            "xai_scope": "od_only_stress_test_xai",
            "methods": "GradCAMPlusPlus; IntegratedGradients",
            "target_regions": "visible_OD_region",
            "analysis_unit": "locked_test_primary_strict_patients_and_images_or_subset",
            "expected_outputs": "sanity-check heatmap concentration",
            "reason": "Stress-test model; not a primary clinical candidate.",
        },
        {
            "model_id": "full_image__MobileNetV3Large",
            "xai_priority": "secondary_optional",
            "xai_scope": "lightweight_full_image_xai",
            "methods": "GradCAMPlusPlus; IntegratedGradients",
            "target_regions": "OD; OC; Rim; OD_plus_OC; outside_OD",
            "analysis_unit": "optional subset if needed",
            "expected_outputs": "secondary architecture sensitivity only",
            "reason": "Comparator; lower priority than hybrid, OD-centered, and shortcut-control models.",
        },
    ]
)

xai_target_registry.to_csv(XAI_TARGET_REGISTRY_CSV, index=False)

with open(XAI_TARGET_REGISTRY_JSON, "w", encoding="utf-8") as f:
    json.dump(xai_target_registry.to_dict(orient="records"), f, indent=2)

setup_summary = {
    "notebook": "12_integrated_internal_model_audit_and_freezing",
    "status": "PASS",
    "objective": (
        "Integrate internal evidence across full-image, anatomy-constrained, shortcut-control, and hybrid models before freezing the model set for external validation."
    ),
    "python": platform.python_version(),
    "project_root": str(PROJECT_ROOT),
    "notebook07_status": notebook07_status.get("status", "unknown"),
    "notebook10_status": notebook10_status.get("status", "unknown"),
    "notebook11_status": notebook11_status.get("status", "unknown"),
    "n_required_artifacts": int(len(required_artifacts)),
    "n_optional_artifacts": int(len(optional_artifacts)),
    "n_optional_artifacts_found": int(
        artifact_audit.loc[
            ~artifact_audit["required"].astype(bool),
            "exists",
        ].astype(bool).sum()
    ),
    "n_internal_candidate_models": int(len(candidate_registry)),
    "candidate_models": candidate_registry["model_id"].tolist(),
    "evidence_domains": evidence_domain_plan["evidence_domain"].tolist(),
    "primary_xai_targets": xai_target_registry[
        xai_target_registry["xai_priority"].astype(str).str.contains("primary")
    ]["model_id"].tolist(),
    "next_cell": "Build integrated performance, calibration and patient-consistency tables.",
    "outputs": {
        "input_artifact_audit_csv": str(INPUT_ARTIFACT_AUDIT_CSV),
        "internal_candidate_registry_csv": str(INTERNAL_CANDIDATE_REGISTRY_CSV),
        "internal_candidate_registry_json": str(INTERNAL_CANDIDATE_REGISTRY_JSON),
        "evidence_domain_plan_csv": str(EVIDENCE_DOMAIN_PLAN_CSV),
        "xai_target_registry_csv": str(XAI_TARGET_REGISTRY_CSV),
        "xai_target_registry_json": str(XAI_TARGET_REGISTRY_JSON),
        "setup_summary_json": str(SETUP_SUMMARY_JSON),
    },
}

with open(SETUP_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(setup_summary, f, indent=2)

print("Notebook 12 setup")
print(f"Notebook 07 status: {setup_summary['notebook07_status']}")
print(f"Notebook 10 status: {setup_summary['notebook10_status']}")
print(f"Notebook 11 status: {setup_summary['notebook11_status']}")
print(f"Required artifacts found: {len(required_artifacts)}/{len(required_artifacts)}")
print(f"Optional artifacts found: {setup_summary['n_optional_artifacts_found']}/{len(optional_artifacts)}")

print("\nInternal candidate registry:")
print(
    candidate_registry[
        [
            "model_id",
            "model_family",
            "model_role",
            "clinical_candidate_status",
            "xai_status",
            "patient_auroc",
            "patient_balanced_accuracy",
            "patient_brier_score",
        ]
    ].to_string(index=False)
)

print("\nEvidence domain plan:")
print(evidence_domain_plan.to_string(index=False))

print("\nXAI target registry:")
print(
    xai_target_registry[
        [
            "model_id",
            "xai_priority",
            "xai_scope",
            "methods",
            "reason",
        ]
    ].to_string(index=False)
)

print(f"\nInput artifact audit CSV: {INPUT_ARTIFACT_AUDIT_CSV}")
print(f"Internal candidate registry CSV: {INTERNAL_CANDIDATE_REGISTRY_CSV}")
print(f"Internal candidate registry JSON: {INTERNAL_CANDIDATE_REGISTRY_JSON}")
print(f"Evidence domain plan CSV: {EVIDENCE_DOMAIN_PLAN_CSV}")
print(f"XAI target registry CSV: {XAI_TARGET_REGISTRY_CSV}")
print(f"XAI target registry JSON: {XAI_TARGET_REGISTRY_JSON}")
print(f"Setup summary JSON: {SETUP_SUMMARY_JSON}")

if len(candidate_registry) != len(EXPECTED_MODEL_ORDER):
    raise RuntimeError(
        f"Unexpected number of candidate models: {len(candidate_registry)} != {len(EXPECTED_MODEL_ORDER)}"
    )

if not set(EXPECTED_MODEL_ORDER).issubset(set(candidate_registry["model_id"].astype(str))):
    raise RuntimeError("Candidate registry is missing expected models.")

if not notebook10_checks["status"].astype(bool).all():
    raise RuntimeError("Notebook 10 final checks are not all PASS.")

if not notebook11_checks["status"].astype(bool).all():
    raise RuntimeError("Notebook 11 final checks are not all PASS.")

primary_required_xai = xai_target_registry[
    xai_target_registry["xai_priority"].isin(
        [
            "primary_required",
            "primary_required_control",
        ]
    )
]

if len(primary_required_xai) < 3:
    raise RuntimeError("Primary XAI target registry is incomplete.")

print("Status: PASS")

In [ ]:
# Cell 2: Integrated performance, calibration, patient consistency, and quality robustness tables

import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    brier_score_loss,
    log_loss,
)

COMPARISON_ROOT_11 = RESULTS_11_DIR / "hybrid_vs_internal_baselines_comparison"

COMBINED_PATIENT_PREDICTIONS_CSV = (
    COMPARISON_ROOT_11
    / "notebook11_combined_internal_and_hybrid_patient_predictions.csv"
)

ANALYSIS_MANIFEST_CSV = (
    PROJECT_DATA_DIR
    / "hygd_final_od_oc_proxy"
    / "hygd_final_od_oc_proxy_analysis_manifest.csv"
)

SPLIT_MANIFEST_CSV = (
    PROJECT_DATA_DIR
    / "hygd_glaucoma_classifier_splits"
    / "hygd_image_level_split_manifest_seed42.csv"
)

ANATOMY_IMAGE_PREDICTIONS_CSV = (
    RESULTS_10_DIR
    / "anatomy_constrained_locked_test_evaluation"
    / "notebook10_anatomy_constrained_test_image_predictions_all_experiments.csv"
)

HYBRID_IMAGE_PREDICTIONS_CSV = (
    RESULTS_11_DIR
    / "hybrid_locked_test_evaluation"
    / "notebook11_test_image_predictions_hybrid_full_image_plus_od_centered_2p5x_effb0.csv"
)

INTEGRATED_PATIENT_PREDICTIONS_CSV = (
    RESULTS_12_DIR
    / "notebook12_integrated_patient_predictions.csv"
)

INTEGRATED_PERFORMANCE_SUMMARY_CSV = (
    RESULTS_12_DIR
    / "notebook12_integrated_patient_level_performance_summary.csv"
)

INTEGRATED_CALIBRATION_SUMMARY_CSV = (
    RESULTS_12_DIR
    / "notebook12_integrated_calibration_summary.csv"
)

INTEGRATED_CALIBRATION_BINS_CSV = (
    RESULTS_12_DIR
    / "notebook12_integrated_calibration_bins.csv"
)

INTEGRATED_QUALITY_ROBUSTNESS_CSV = (
    RESULTS_12_DIR
    / "notebook12_integrated_quality_robustness_summary.csv"
)

INTEGRATED_IMAGE_PREDICTIONS_CSV = (
    RESULTS_12_DIR
    / "notebook12_integrated_image_predictions_available_models.csv"
)

INTEGRATED_PATIENT_CONSISTENCY_CSV = (
    RESULTS_12_DIR
    / "notebook12_integrated_patient_consistency_summary.csv"
)

INTEGRATED_PATIENT_CONSISTENCY_DETAIL_CSV = (
    RESULTS_12_DIR
    / "notebook12_integrated_patient_consistency_detail.csv"
)

INTEGRATED_EVIDENCE_TABLE_CSV = (
    RESULTS_12_DIR
    / "notebook12_integrated_internal_evidence_table.csv"
)

CELL2_SUMMARY_JSON = (
    RESULTS_12_DIR
    / "notebook12_integrated_performance_calibration_consistency_summary.json"
)

required_inputs = [
    INTERNAL_CANDIDATE_REGISTRY_CSV,
    COMBINED_PATIENT_PREDICTIONS_CSV,
    ANALYSIS_MANIFEST_CSV,
    SPLIT_MANIFEST_CSV,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not Path(path).exists()
]

if missing_inputs:
    raise FileNotFoundError(f"Missing Cell 2 inputs: {missing_inputs}")

candidate_registry = pd.read_csv(INTERNAL_CANDIDATE_REGISTRY_CSV)
patient_predictions = pd.read_csv(COMBINED_PATIENT_PREDICTIONS_CSV)
analysis_manifest = pd.read_csv(ANALYSIS_MANIFEST_CSV)
split_manifest = pd.read_csv(SPLIT_MANIFEST_CSV)

candidate_registry["model_id"] = candidate_registry["model_id"].astype(str)
patient_predictions["model_id"] = patient_predictions["model_id"].astype(str)
patient_predictions["patient_id"] = patient_predictions["patient_id"].astype(str)
patient_predictions["y_true"] = pd.to_numeric(patient_predictions["y_true"], errors="coerce").astype(int)
patient_predictions["y_prob"] = pd.to_numeric(patient_predictions["y_prob"], errors="coerce")

if "threshold" not in patient_predictions.columns:
    patient_predictions["threshold"] = 0.5

patient_predictions["threshold"] = pd.to_numeric(
    patient_predictions["threshold"],
    errors="coerce",
).fillna(0.5)

analysis_manifest["sample_id"] = analysis_manifest["sample_id"].astype(str)
analysis_manifest["patient_id"] = analysis_manifest["patient_id"].astype(str)

split_manifest["sample_id"] = split_manifest["sample_id"].astype(str)
split_manifest["patient_id"] = split_manifest["patient_id"].astype(str)
split_manifest["glaucoma_split"] = split_manifest["glaucoma_split"].astype(str)

if "quality_score" not in analysis_manifest.columns:
    raise ValueError("Missing quality_score in analysis manifest.")

if "gon_binary" not in analysis_manifest.columns:
    raise ValueError("Missing gon_binary in analysis manifest.")

analysis_manifest["quality_score"] = pd.to_numeric(
    analysis_manifest["quality_score"],
    errors="coerce",
)

analysis_manifest["gon_binary"] = pd.to_numeric(
    analysis_manifest["gon_binary"],
    errors="coerce",
).astype(int)

test_samples = split_manifest[
    split_manifest["glaucoma_split"] == "test"
][
    [
        "sample_id",
        "patient_id",
        "glaucoma_split",
    ]
].drop_duplicates().copy()

test_image_metadata = test_samples.merge(
    analysis_manifest[
        [
            "sample_id",
            "patient_id",
            "quality_score",
            "gon_binary",
        ]
    ],
    on=["sample_id", "patient_id"],
    how="left",
    validate="one_to_one",
)

if test_image_metadata["quality_score"].isna().any():
    missing_quality = int(test_image_metadata["quality_score"].isna().sum())
    raise RuntimeError(f"Missing test-image quality scores: {missing_quality}")

test_patient_metadata = (
    test_image_metadata
    .groupby("patient_id")
    .agg(
        n_test_images=("sample_id", "nunique"),
        y_true_from_metadata=("gon_binary", "max"),
        patient_quality_mean=("quality_score", "mean"),
        patient_quality_min=("quality_score", "min"),
        patient_quality_max=("quality_score", "max"),
        patient_quality_std=("quality_score", "std"),
    )
    .reset_index()
)

test_patient_metadata["patient_quality_std"] = (
    test_patient_metadata["patient_quality_std"].fillna(0.0)
)

quality_ranks = test_patient_metadata["patient_quality_mean"].rank(method="first")

test_patient_metadata["quality_tertile"] = pd.qcut(
    quality_ranks,
    q=3,
    labels=["low", "medium", "high"],
)

patient_predictions = patient_predictions.merge(
    test_patient_metadata,
    on="patient_id",
    how="left",
    validate="many_to_one",
)

if patient_predictions["patient_quality_mean"].isna().any():
    missing_patient_quality = int(patient_predictions["patient_quality_mean"].isna().sum())
    raise RuntimeError(f"Missing patient-level quality metadata after merge: {missing_patient_quality}")

patient_predictions["y_pred"] = (
    patient_predictions["y_prob"].astype(float)
    >= patient_predictions["threshold"].astype(float)
).astype(int)

patient_predictions["correct"] = (
    patient_predictions["y_pred"].astype(int)
    == patient_predictions["y_true"].astype(int)
)

patient_predictions.to_csv(INTEGRATED_PATIENT_PREDICTIONS_CSV, index=False)


def safe_metric_bundle(y_true, y_prob, threshold):
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)
    threshold = float(threshold)

    valid = np.isfinite(y_prob)
    y_true = y_true[valid]
    y_prob = y_prob[valid]

    if len(y_true) == 0:
        return {
            "n_samples": 0,
            "n_positive": 0,
            "n_negative": 0,
            "threshold": threshold,
            "auroc": np.nan,
            "auprc": np.nan,
            "accuracy": np.nan,
            "balanced_accuracy": np.nan,
            "precision": np.nan,
            "sensitivity": np.nan,
            "specificity": np.nan,
            "f1": np.nan,
            "brier_score": np.nan,
            "log_loss": np.nan,
            "tn": 0,
            "fp": 0,
            "fn": 0,
            "tp": 0,
        }

    y_pred = (y_prob >= threshold).astype(int)

    if len(np.unique(y_true)) == 2:
        auroc = float(roc_auc_score(y_true, y_prob))
        auprc = float(average_precision_score(y_true, y_prob))
        logloss = float(
            log_loss(
                y_true,
                np.clip(y_prob, 1e-7, 1.0 - 1e-7),
                labels=[0, 1],
            )
        )
    else:
        auroc = np.nan
        auprc = np.nan
        logloss = np.nan

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan

    return {
        "n_samples": int(len(y_true)),
        "n_positive": int(np.sum(y_true == 1)),
        "n_negative": int(np.sum(y_true == 0)),
        "threshold": threshold,
        "auroc": auroc,
        "auprc": auprc,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "sensitivity": float(sensitivity),
        "specificity": float(specificity),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "brier_score": float(brier_score_loss(y_true, y_prob)),
        "log_loss": logloss,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def calibration_summary_and_bins(model_df, n_bins=5):
    y_true = model_df["y_true"].astype(int).values
    y_prob = model_df["y_prob"].astype(float).values

    valid = np.isfinite(y_prob)
    y_true = y_true[valid]
    y_prob = y_prob[valid]

    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    bin_rows = []

    ece = 0.0
    mce = 0.0

    for bin_idx in range(n_bins):
        lower = bin_edges[bin_idx]
        upper = bin_edges[bin_idx + 1]

        if bin_idx == n_bins - 1:
            in_bin = (y_prob >= lower) & (y_prob <= upper)
        else:
            in_bin = (y_prob >= lower) & (y_prob < upper)

        n_bin = int(in_bin.sum())

        if n_bin > 0:
            mean_confidence = float(np.mean(y_prob[in_bin]))
            observed_rate = float(np.mean(y_true[in_bin]))
            abs_gap = abs(mean_confidence - observed_rate)
            weight = n_bin / len(y_true)
            ece += weight * abs_gap
            mce = max(mce, abs_gap)
        else:
            mean_confidence = np.nan
            observed_rate = np.nan
            abs_gap = np.nan
            weight = 0.0

        bin_rows.append(
            {
                "bin_index": bin_idx,
                "bin_lower": float(lower),
                "bin_upper": float(upper),
                "n_bin": n_bin,
                "bin_weight": float(weight),
                "mean_confidence": mean_confidence,
                "observed_positive_rate": observed_rate,
                "absolute_calibration_gap": abs_gap,
            }
        )

    summary = {
        "ece_equal_width_5bin": float(ece),
        "mce_equal_width_5bin": float(mce),
        "mean_predicted_probability": float(np.mean(y_prob)),
        "observed_prevalence": float(np.mean(y_true)),
        "calibration_intercept_proxy": float(np.mean(y_prob) - np.mean(y_true)),
    }

    return summary, pd.DataFrame(bin_rows)


performance_rows = []
calibration_rows = []
calibration_bin_frames = []

for model_id, model_df in patient_predictions.groupby("model_id"):
    threshold = float(model_df["threshold"].iloc[0])

    metrics = safe_metric_bundle(
        y_true=model_df["y_true"].values,
        y_prob=model_df["y_prob"].values,
        threshold=threshold,
    )

    calibration_summary, calibration_bins = calibration_summary_and_bins(
        model_df,
        n_bins=5,
    )

    registry_block = candidate_registry[
        candidate_registry["model_id"].astype(str) == str(model_id)
    ]

    if len(registry_block) == 1:
        registry_row = registry_block.iloc[0].to_dict()
    else:
        registry_row = {}

    performance_rows.append(
        {
            "model_id": str(model_id),
            "model_family": registry_row.get("model_family", "unknown"),
            "model_role": registry_row.get("model_role", "unknown"),
            "clinical_candidate_status": registry_row.get("clinical_candidate_status", "unknown"),
            **metrics,
        }
    )

    calibration_rows.append(
        {
            "model_id": str(model_id),
            "model_family": registry_row.get("model_family", "unknown"),
            "model_role": registry_row.get("model_role", "unknown"),
            **calibration_summary,
        }
    )

    calibration_bins["model_id"] = str(model_id)
    calibration_bin_frames.append(calibration_bins)

performance_summary = pd.DataFrame(performance_rows)
calibration_summary = pd.DataFrame(calibration_rows)
calibration_bins = pd.concat(calibration_bin_frames, axis=0, ignore_index=True)

performance_summary.to_csv(INTEGRATED_PERFORMANCE_SUMMARY_CSV, index=False)
calibration_summary.to_csv(INTEGRATED_CALIBRATION_SUMMARY_CSV, index=False)
calibration_bins.to_csv(INTEGRATED_CALIBRATION_BINS_CSV, index=False)


quality_rows = []

for model_id, model_df in patient_predictions.groupby("model_id"):
    for quality_tertile, block in model_df.groupby("quality_tertile"):
        threshold = float(block["threshold"].iloc[0])

        metrics = safe_metric_bundle(
            y_true=block["y_true"].values,
            y_prob=block["y_prob"].values,
            threshold=threshold,
        )

        quality_rows.append(
            {
                "model_id": str(model_id),
                "quality_tertile": str(quality_tertile),
                "patient_quality_mean_min": float(block["patient_quality_mean"].min()),
                "patient_quality_mean_max": float(block["patient_quality_mean"].max()),
                **metrics,
            }
        )

quality_robustness = pd.DataFrame(quality_rows)
quality_robustness.to_csv(INTEGRATED_QUALITY_ROBUSTNESS_CSV, index=False)


def standardize_image_predictions_from_anatomy(path):
    if not Path(path).exists():
        return pd.DataFrame([])

    df = pd.read_csv(path)

    required_cols = ["experiment_id", "sample_id", "patient_id", "y_true", "y_prob"]

    if not set(required_cols).issubset(df.columns):
        return pd.DataFrame([])

    output = df.copy()
    output["model_id"] = "anatomy__" + output["experiment_id"].astype(str)
    output["sample_id"] = output["sample_id"].astype(str)
    output["patient_id"] = output["patient_id"].astype(str)
    output["y_true"] = pd.to_numeric(output["y_true"], errors="coerce").astype(int)
    output["y_prob"] = pd.to_numeric(output["y_prob"], errors="coerce")
    output["image_prediction_source"] = str(path)

    return output[
        [
            "model_id",
            "sample_id",
            "patient_id",
            "y_true",
            "y_prob",
            "image_prediction_source",
        ]
    ]


def standardize_image_predictions_from_hybrid(path):
    if not Path(path).exists():
        return pd.DataFrame([])

    df = pd.read_csv(path)

    required_cols = ["sample_id", "patient_id", "y_true", "y_prob"]

    if not set(required_cols).issubset(df.columns):
        return pd.DataFrame([])

    output = df.copy()
    output["model_id"] = "hybrid__full_image_plus_od_centered_2p5x_effb0"
    output["sample_id"] = output["sample_id"].astype(str)
    output["patient_id"] = output["patient_id"].astype(str)
    output["y_true"] = pd.to_numeric(output["y_true"], errors="coerce").astype(int)
    output["y_prob"] = pd.to_numeric(output["y_prob"], errors="coerce")
    output["image_prediction_source"] = str(path)

    return output[
        [
            "model_id",
            "sample_id",
            "patient_id",
            "y_true",
            "y_prob",
            "image_prediction_source",
        ]
    ]


def standardize_possible_full_image_predictions(path):
    try:
        df = pd.read_csv(path)
    except Exception:
        return pd.DataFrame([])

    columns = set(df.columns)

    if not {"sample_id", "patient_id"}.issubset(columns):
        return pd.DataFrame([])

    prob_col = None
    for candidate in ["y_prob", "probability", "pred_probability", "pred_prob", "mean_y_prob"]:
        if candidate in columns:
            prob_col = candidate
            break

    if prob_col is None:
        return pd.DataFrame([])

    label_col = None
    for candidate in ["y_true", "gon_binary", "label", "target"]:
        if candidate in columns:
            label_col = candidate
            break

    if label_col is None:
        return pd.DataFrame([])

    model_id = None

    if "model_id" in columns:
        unique_model_ids = df["model_id"].dropna().astype(str).unique().tolist()
        if len(unique_model_ids) == 1:
            model_id = unique_model_ids[0]

    if model_id is None and "model_name" in columns:
        unique_model_names = df["model_name"].dropna().astype(str).unique().tolist()
        if len(unique_model_names) == 1:
            model_name = unique_model_names[0]
            model_map = {
                "EfficientNetB0": "full_image__EfficientNetB0",
                "MobileNetV3Large": "full_image__MobileNetV3Large",
                "DeiTSmall": "full_image__DeiTSmall",
            }
            model_id = model_map.get(model_name, None)

    if model_id is None:
        path_text = str(path)
        if "EfficientNetB0" in path_text:
            model_id = "full_image__EfficientNetB0"
        elif "MobileNetV3Large" in path_text:
            model_id = "full_image__MobileNetV3Large"
        elif "DeiTSmall" in path_text:
            model_id = "full_image__DeiTSmall"

    if model_id is None:
        return pd.DataFrame([])

    if not str(model_id).startswith("full_image__"):
        return pd.DataFrame([])

    output = pd.DataFrame(
        {
            "model_id": str(model_id),
            "sample_id": df["sample_id"].astype(str),
            "patient_id": df["patient_id"].astype(str),
            "y_true": pd.to_numeric(df[label_col], errors="coerce").astype(int),
            "y_prob": pd.to_numeric(df[prob_col], errors="coerce"),
            "image_prediction_source": str(path),
        }
    )

    return output


image_prediction_frames = []

anatomy_image_predictions = standardize_image_predictions_from_anatomy(
    ANATOMY_IMAGE_PREDICTIONS_CSV
)

if len(anatomy_image_predictions) > 0:
    image_prediction_frames.append(anatomy_image_predictions)

hybrid_image_predictions = standardize_image_predictions_from_hybrid(
    HYBRID_IMAGE_PREDICTIONS_CSV
)

if len(hybrid_image_predictions) > 0:
    image_prediction_frames.append(hybrid_image_predictions)

possible_full_image_files = sorted(
    set(
        list(RESULTS_07_DIR.rglob("*test*image*prediction*.csv"))
        + list(RESULTS_07_DIR.rglob("*test_image_predictions*.csv"))
        + list(RESULTS_07_DIR.rglob("*image_level*pred*.csv"))
    )
)

full_image_frames = []

for path in possible_full_image_files:
    standardized = standardize_possible_full_image_predictions(path)

    if len(standardized) > 0:
        full_image_frames.append(standardized)

if len(full_image_frames) > 0:
    full_image_predictions = pd.concat(full_image_frames, axis=0, ignore_index=True)
    full_image_predictions = full_image_predictions.drop_duplicates(
        subset=["model_id", "sample_id", "patient_id"],
        keep="last",
    )

    image_prediction_frames.append(full_image_predictions)

if len(image_prediction_frames) > 0:
    image_predictions = pd.concat(image_prediction_frames, axis=0, ignore_index=True)

    threshold_map = (
        patient_predictions
        .groupby("model_id")["threshold"]
        .first()
        .to_dict()
    )

    image_predictions["threshold"] = image_predictions["model_id"].map(threshold_map)
    image_predictions["threshold"] = image_predictions["threshold"].fillna(0.5)

    image_predictions["image_pred"] = (
        image_predictions["y_prob"].astype(float)
        >= image_predictions["threshold"].astype(float)
    ).astype(int)

    image_predictions.to_csv(INTEGRATED_IMAGE_PREDICTIONS_CSV, index=False)

    consistency_detail = (
        image_predictions
        .groupby(["model_id", "patient_id"])
        .agg(
            n_images=("sample_id", "nunique"),
            y_true=("y_true", "max"),
            y_prob_mean=("y_prob", "mean"),
            y_prob_std=("y_prob", "std"),
            y_prob_min=("y_prob", "min"),
            y_prob_max=("y_prob", "max"),
            image_pred_positive_fraction=("image_pred", "mean"),
            image_pred_n_unique=("image_pred", "nunique"),
        )
        .reset_index()
    )

    consistency_detail["y_prob_std"] = consistency_detail["y_prob_std"].fillna(0.0)
    consistency_detail["y_prob_range"] = (
        consistency_detail["y_prob_max"] - consistency_detail["y_prob_min"]
    )

    consistency_detail["has_image_prediction_disagreement"] = (
        consistency_detail["image_pred_n_unique"] > 1
    )

    consistency_summary = (
        consistency_detail
        .groupby("model_id")
        .agg(
            n_patients=("patient_id", "nunique"),
            n_multi_image_patients=("n_images", lambda x: int((x > 1).sum())),
            mean_within_patient_prob_std=("y_prob_std", "mean"),
            median_within_patient_prob_std=("y_prob_std", "median"),
            mean_within_patient_prob_range=("y_prob_range", "mean"),
            median_within_patient_prob_range=("y_prob_range", "median"),
            n_patients_with_image_prediction_disagreement=("has_image_prediction_disagreement", "sum"),
        )
        .reset_index()
    )

    consistency_summary["fraction_patients_with_image_prediction_disagreement"] = (
        consistency_summary["n_patients_with_image_prediction_disagreement"]
        / consistency_summary["n_patients"]
    )

else:
    image_predictions = pd.DataFrame([])
    consistency_detail = pd.DataFrame([])
    consistency_summary = pd.DataFrame(
        columns=[
            "model_id",
            "n_patients",
            "n_multi_image_patients",
            "mean_within_patient_prob_std",
            "median_within_patient_prob_std",
            "mean_within_patient_prob_range",
            "median_within_patient_prob_range",
            "n_patients_with_image_prediction_disagreement",
            "fraction_patients_with_image_prediction_disagreement",
        ]
    )

    image_predictions.to_csv(INTEGRATED_IMAGE_PREDICTIONS_CSV, index=False)

consistency_detail.to_csv(INTEGRATED_PATIENT_CONSISTENCY_DETAIL_CSV, index=False)
consistency_summary.to_csv(INTEGRATED_PATIENT_CONSISTENCY_CSV, index=False)

evidence_table = candidate_registry.merge(
    performance_summary[
        [
            "model_id",
            "auroc",
            "auprc",
            "balanced_accuracy",
            "sensitivity",
            "specificity",
            "brier_score",
            "log_loss",
        ]
    ].rename(
        columns={
            "auroc": "integrated_patient_auroc",
            "auprc": "integrated_patient_auprc",
            "balanced_accuracy": "integrated_patient_balanced_accuracy",
            "sensitivity": "integrated_patient_sensitivity",
            "specificity": "integrated_patient_specificity",
            "brier_score": "integrated_patient_brier_score",
            "log_loss": "integrated_patient_log_loss",
        }
    ),
    on="model_id",
    how="left",
    validate="one_to_one",
)

evidence_table = evidence_table.merge(
    calibration_summary[
        [
            "model_id",
            "ece_equal_width_5bin",
            "mce_equal_width_5bin",
            "mean_predicted_probability",
            "observed_prevalence",
            "calibration_intercept_proxy",
        ]
    ],
    on="model_id",
    how="left",
    validate="one_to_one",
)

evidence_table = evidence_table.merge(
    consistency_summary,
    on="model_id",
    how="left",
    validate="one_to_one",
)

evidence_table.to_csv(INTEGRATED_EVIDENCE_TABLE_CSV, index=False)

summary_payload = {
    "notebook": "12_integrated_internal_model_audit_and_freezing",
    "cell": "02_integrated_performance_calibration_patient_consistency_quality",
    "status": "PASS",
    "n_models_in_patient_predictions": int(patient_predictions["model_id"].nunique()),
    "n_patient_prediction_rows": int(len(patient_predictions)),
    "n_test_patients": int(test_patient_metadata["patient_id"].nunique()),
    "n_quality_tertiles": int(test_patient_metadata["quality_tertile"].nunique()),
    "n_models_with_image_level_predictions": int(image_predictions["model_id"].nunique()) if len(image_predictions) > 0 else 0,
    "models_with_image_level_predictions": sorted(image_predictions["model_id"].unique().tolist()) if len(image_predictions) > 0 else [],
    "n_discovered_full_image_prediction_files": int(len(possible_full_image_files)),
    "outputs": {
        "integrated_patient_predictions_csv": str(INTEGRATED_PATIENT_PREDICTIONS_CSV),
        "integrated_performance_summary_csv": str(INTEGRATED_PERFORMANCE_SUMMARY_CSV),
        "integrated_calibration_summary_csv": str(INTEGRATED_CALIBRATION_SUMMARY_CSV),
        "integrated_calibration_bins_csv": str(INTEGRATED_CALIBRATION_BINS_CSV),
        "integrated_quality_robustness_csv": str(INTEGRATED_QUALITY_ROBUSTNESS_CSV),
        "integrated_image_predictions_csv": str(INTEGRATED_IMAGE_PREDICTIONS_CSV),
        "integrated_patient_consistency_csv": str(INTEGRATED_PATIENT_CONSISTENCY_CSV),
        "integrated_patient_consistency_detail_csv": str(INTEGRATED_PATIENT_CONSISTENCY_DETAIL_CSV),
        "integrated_evidence_table_csv": str(INTEGRATED_EVIDENCE_TABLE_CSV),
    },
}

with open(CELL2_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary_payload, f, indent=2)

print("Integrated patient-level performance summary:")
print(
    performance_summary[
        [
            "model_id",
            "model_family",
            "auroc",
            "auprc",
            "balanced_accuracy",
            "sensitivity",
            "specificity",
            "brier_score",
            "log_loss",
        ]
    ].sort_values(
        ["brier_score", "auroc"],
        ascending=[True, False],
    ).to_string(index=False)
)

print("\nIntegrated calibration summary:")
print(
    calibration_summary[
        [
            "model_id",
            "ece_equal_width_5bin",
            "mce_equal_width_5bin",
            "mean_predicted_probability",
            "observed_prevalence",
            "calibration_intercept_proxy",
        ]
    ].sort_values("ece_equal_width_5bin").to_string(index=False)
)

print("\nQuality-stratified robustness summary:")
print(
    quality_robustness[
        [
            "model_id",
            "quality_tertile",
            "n_samples",
            "auroc",
            "balanced_accuracy",
            "brier_score",
            "sensitivity",
            "specificity",
        ]
    ].sort_values(["model_id", "quality_tertile"]).to_string(index=False)
)

print("\nPatient consistency summary for available image-level predictions:")
if len(consistency_summary) == 0:
    print("No image-level prediction files were available for consistency analysis.")
else:
    print(
        consistency_summary.sort_values(
            "mean_within_patient_prob_std",
            ascending=True,
        ).to_string(index=False)
    )

print("\nIntegrated evidence table:")
print(
    evidence_table[
        [
            "model_id",
            "model_family",
            "clinical_candidate_status",
            "integrated_patient_auroc",
            "integrated_patient_balanced_accuracy",
            "integrated_patient_brier_score",
            "ece_equal_width_5bin",
            "mean_within_patient_prob_std",
            "fraction_patients_with_image_prediction_disagreement",
        ]
    ].to_string(index=False)
)

print(f"\nIntegrated patient predictions CSV: {INTEGRATED_PATIENT_PREDICTIONS_CSV}")
print(f"Performance summary CSV: {INTEGRATED_PERFORMANCE_SUMMARY_CSV}")
print(f"Calibration summary CSV: {INTEGRATED_CALIBRATION_SUMMARY_CSV}")
print(f"Calibration bins CSV: {INTEGRATED_CALIBRATION_BINS_CSV}")
print(f"Quality robustness CSV: {INTEGRATED_QUALITY_ROBUSTNESS_CSV}")
print(f"Image predictions CSV: {INTEGRATED_IMAGE_PREDICTIONS_CSV}")
print(f"Patient consistency CSV: {INTEGRATED_PATIENT_CONSISTENCY_CSV}")
print(f"Patient consistency detail CSV: {INTEGRATED_PATIENT_CONSISTENCY_DETAIL_CSV}")
print(f"Integrated evidence table CSV: {INTEGRATED_EVIDENCE_TABLE_CSV}")
print(f"Cell 2 summary JSON: {CELL2_SUMMARY_JSON}")

required_outputs = [
    INTEGRATED_PATIENT_PREDICTIONS_CSV,
    INTEGRATED_PERFORMANCE_SUMMARY_CSV,
    INTEGRATED_CALIBRATION_SUMMARY_CSV,
    INTEGRATED_CALIBRATION_BINS_CSV,
    INTEGRATED_QUALITY_ROBUSTNESS_CSV,
    INTEGRATED_IMAGE_PREDICTIONS_CSV,
    INTEGRATED_PATIENT_CONSISTENCY_CSV,
    INTEGRATED_PATIENT_CONSISTENCY_DETAIL_CSV,
    INTEGRATED_EVIDENCE_TABLE_CSV,
    CELL2_SUMMARY_JSON,
]

missing_outputs = [
    str(path)
    for path in required_outputs
    if not Path(path).exists()
]

if missing_outputs:
    raise RuntimeError(f"Missing Cell 2 outputs: {missing_outputs}")

if patient_predictions["model_id"].nunique() != len(EXPECTED_MODEL_ORDER):
    raise RuntimeError(
        f"Unexpected number of models in patient predictions: {patient_predictions['model_id'].nunique()}"
    )

if len(performance_summary) != len(EXPECTED_MODEL_ORDER):
    raise RuntimeError("Integrated performance summary does not include all expected models.")

if len(calibration_summary) != len(EXPECTED_MODEL_ORDER):
    raise RuntimeError("Integrated calibration summary does not include all expected models.")

print("Status: PASS")

In [ ]:
# Cell 3: Generate integrated internal evidence figure

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

INTEGRATED_EVIDENCE_TABLE_CSV = (
    RESULTS_12_DIR
    / "notebook12_integrated_internal_evidence_table.csv"
)

INTEGRATED_QUALITY_ROBUSTNESS_CSV = (
    RESULTS_12_DIR
    / "notebook12_integrated_quality_robustness_summary.csv"
)

INTEGRATED_CALIBRATION_BINS_CSV = (
    RESULTS_12_DIR
    / "notebook12_integrated_calibration_bins.csv"
)

FIGURE_ROOT = RESULTS_12_DIR / "figures"
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)

INTEGRATED_EVIDENCE_FIGURE_PNG = (
    FIGURE_ROOT
    / "notebook12_integrated_internal_evidence_figure.png"
)

INTEGRATED_EVIDENCE_FIGURE_PDF = (
    FIGURE_ROOT
    / "notebook12_integrated_internal_evidence_figure.pdf"
)

INTEGRATED_EVIDENCE_FIGURE_SVG = (
    FIGURE_ROOT
    / "notebook12_integrated_internal_evidence_figure.svg"
)

INTEGRATED_EVIDENCE_FIGURE_MANIFEST_CSV = (
    RESULTS_12_DIR
    / "notebook12_integrated_internal_evidence_figure_manifest.csv"
)

INTEGRATED_EVIDENCE_FIGURE_SUMMARY_JSON = (
    RESULTS_12_DIR
    / "notebook12_integrated_internal_evidence_figure_summary.json"
)

required_inputs = [
    INTEGRATED_EVIDENCE_TABLE_CSV,
    INTEGRATED_QUALITY_ROBUSTNESS_CSV,
    INTEGRATED_CALIBRATION_BINS_CSV,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not Path(path).exists()
]

if missing_inputs:
    raise FileNotFoundError(f"Missing Cell 3 inputs: {missing_inputs}")

evidence_table = pd.read_csv(INTEGRATED_EVIDENCE_TABLE_CSV)
quality_robustness = pd.read_csv(INTEGRATED_QUALITY_ROBUSTNESS_CSV)
calibration_bins = pd.read_csv(INTEGRATED_CALIBRATION_BINS_CSV)

MODEL_ORDER = [
    "full_image__EfficientNetB0",
    "full_image__MobileNetV3Large",
    "full_image__DeiTSmall",
    "anatomy__od_centered_crop_2p5x",
    "anatomy__od_centered_crop_4p0x",
    "anatomy__od_only_context_suppressed",
    "anatomy__outside_od_only_trained_control",
    "hybrid__full_image_plus_od_centered_2p5x_effb0",
]

MODEL_LABELS = {
    "full_image__EfficientNetB0": "Full image\nEffNet-B0",
    "full_image__MobileNetV3Large": "Full image\nMobileNetV3",
    "full_image__DeiTSmall": "Full image\nDeiT-Small",
    "anatomy__od_centered_crop_2p5x": "OD-centered\n2.5x",
    "anatomy__od_centered_crop_4p0x": "OD-centered\n4.0x",
    "anatomy__od_only_context_suppressed": "OD-only\nstress",
    "anatomy__outside_od_only_trained_control": "Outside-OD\ncontrol",
    "hybrid__full_image_plus_od_centered_2p5x_effb0": "Hybrid\nFull + OD",
}

MODEL_GROUPS = {
    "full_image__EfficientNetB0": "Full-image",
    "full_image__MobileNetV3Large": "Full-image",
    "full_image__DeiTSmall": "Full-image",
    "anatomy__od_centered_crop_2p5x": "Anatomy-constrained",
    "anatomy__od_centered_crop_4p0x": "Anatomy-constrained",
    "anatomy__od_only_context_suppressed": "Stress test",
    "anatomy__outside_od_only_trained_control": "Shortcut-control",
    "hybrid__full_image_plus_od_centered_2p5x_effb0": "Hybrid",
}

MODEL_COLORS = {
    "Full-image": "#4C78A8",
    "Anatomy-constrained": "#54A24B",
    "Stress test": "#F58518",
    "Shortcut-control": "#B279A2",
    "Hybrid": "#E45756",
}

available_models = set(evidence_table["model_id"].astype(str))

missing_models = [
    model_id
    for model_id in MODEL_ORDER
    if model_id not in available_models
]

if missing_models:
    raise RuntimeError(f"Missing expected models in evidence table: {missing_models}")

plot_table = evidence_table.set_index("model_id").loc[MODEL_ORDER].reset_index()
x = np.arange(len(MODEL_ORDER))
bar_colors = [MODEL_COLORS[MODEL_GROUPS[m]] for m in MODEL_ORDER]


def add_panel_label(ax, label):
    ax.text(
        -0.12,
        1.08,
        label,
        transform=ax.transAxes,
        fontsize=13,
        fontweight="bold",
        va="top",
        ha="left",
    )


def style_axis(ax, grid_axis="y"):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis=grid_axis, alpha=0.25, linewidth=0.8)
    ax.tick_params(axis="both", labelsize=9)


fig = plt.figure(figsize=(19.5, 12.0))

gs = gridspec.GridSpec(
    2,
    3,
    figure=fig,
    height_ratios=[1.0, 1.0],
    width_ratios=[1.0, 1.0, 1.0],
    hspace=0.52,
    wspace=0.36,
)

axes = [
    fig.add_subplot(gs[0, 0]),
    fig.add_subplot(gs[0, 1]),
    fig.add_subplot(gs[0, 2]),
    fig.add_subplot(gs[1, 0]),
    fig.add_subplot(gs[1, 1]),
    fig.add_subplot(gs[1, 2]),
]

# A: Patient balanced accuracy
ax = axes[0]
values = plot_table["integrated_patient_balanced_accuracy"].astype(float).values

ax.bar(x, values, color=bar_colors, alpha=0.88)
ax.set_xticks(x)
ax.set_xticklabels([MODEL_LABELS[m] for m in MODEL_ORDER], rotation=35, ha="right")
ax.set_ylabel("Patient balanced accuracy")
ax.set_title("Locked-test patient-level balanced accuracy")
ax.set_ylim(0.75, 1.03)
style_axis(ax)
add_panel_label(ax, "A")

# B: Brier score
ax = axes[1]
values = plot_table["integrated_patient_brier_score"].astype(float).values

ax.bar(x, values, color=bar_colors, alpha=0.88)
ax.set_xticks(x)
ax.set_xticklabels([MODEL_LABELS[m] for m in MODEL_ORDER], rotation=35, ha="right")
ax.set_ylabel("Patient Brier score")
ax.set_title("Patient-level probabilistic error")
ax.set_ylim(0.0, max(values) * 1.35)
style_axis(ax)
add_panel_label(ax, "B")

# C: ECE
ax = axes[2]
values = plot_table["ece_equal_width_5bin"].astype(float).values

ax.bar(x, values, color=bar_colors, alpha=0.88)
ax.set_xticks(x)
ax.set_xticklabels([MODEL_LABELS[m] for m in MODEL_ORDER], rotation=35, ha="right")
ax.set_ylabel("ECE, 5 equal-width bins")
ax.set_title("Calibration error")
ax.set_ylim(0.0, max(values) * 1.35)
style_axis(ax)
add_panel_label(ax, "C")

# D: Patient consistency probability std
ax = axes[3]
values = plot_table["mean_within_patient_prob_std"].astype(float).values

ax.bar(x, values, color=bar_colors, alpha=0.88)
ax.set_xticks(x)
ax.set_xticklabels([MODEL_LABELS[m] for m in MODEL_ORDER], rotation=35, ha="right")
ax.set_ylabel("Mean within-patient probability SD")
ax.set_title("Multi-image patient consistency")
ax.set_ylim(0.0, max(values) * 1.35)
style_axis(ax)
add_panel_label(ax, "D")

# E: Prediction disagreement fraction
ax = axes[4]
values = plot_table["fraction_patients_with_image_prediction_disagreement"].astype(float).values

ax.bar(x, values, color=bar_colors, alpha=0.88)
ax.set_xticks(x)
ax.set_xticklabels([MODEL_LABELS[m] for m in MODEL_ORDER], rotation=35, ha="right")
ax.set_ylabel("Fraction of patients with image-level disagreement")
ax.set_title("Within-patient classification disagreement")
ax.set_ylim(0.0, max(values) * 1.35)
style_axis(ax)
add_panel_label(ax, "E")

# F: Quality-stratified Brier score for selected candidates
ax = axes[5]

SELECTED_FOR_QUALITY = [
    "full_image__DeiTSmall",
    "anatomy__od_centered_crop_2p5x",
    "anatomy__od_centered_crop_4p0x",
    "anatomy__outside_od_only_trained_control",
    "hybrid__full_image_plus_od_centered_2p5x_effb0",
]

quality_order = ["low", "medium", "high"]
width = 0.14
base_x = np.arange(len(quality_order))

for idx, model_id in enumerate(SELECTED_FOR_QUALITY):
    block = quality_robustness[
        quality_robustness["model_id"].astype(str) == model_id
    ].copy()

    block["quality_tertile"] = block["quality_tertile"].astype(str)
    block = block.set_index("quality_tertile").loc[quality_order].reset_index()

    offset = (idx - (len(SELECTED_FOR_QUALITY) - 1) / 2.0) * width

    ax.bar(
        base_x + offset,
        block["brier_score"].astype(float).values,
        width=width,
        alpha=0.88,
        label=MODEL_LABELS[model_id].replace("\n", " "),
        color=MODEL_COLORS[MODEL_GROUPS[model_id]],
    )

ax.set_xticks(base_x)
ax.set_xticklabels(["Low", "Medium", "High"])
ax.set_ylabel("Patient Brier score")
ax.set_title("Quality-stratified Brier score")
ax.legend(fontsize=7, frameon=False, loc="upper right")
style_axis(ax)
add_panel_label(ax, "F")

legend_handles = []

for group_name, color in MODEL_COLORS.items():
    if group_name in set(MODEL_GROUPS[m] for m in MODEL_ORDER):
        legend_handles.append(
            plt.Line2D(
                [0],
                [0],
                marker="s",
                color="w",
                label=group_name,
                markerfacecolor=color,
                markersize=10,
            )
        )

fig.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.01),
    ncol=5,
    frameon=False,
    fontsize=10,
)

fig.suptitle(
    "Integrated internal evidence before model freezing",
    fontsize=15,
    fontweight="bold",
    y=1.055,
)

plt.savefig(INTEGRATED_EVIDENCE_FIGURE_PNG, dpi=300, bbox_inches="tight")
plt.savefig(INTEGRATED_EVIDENCE_FIGURE_PDF, bbox_inches="tight")
plt.savefig(INTEGRATED_EVIDENCE_FIGURE_SVG, bbox_inches="tight")
plt.show()

figure_manifest = pd.DataFrame(
    [
        {
            "figure_name": "integrated_internal_evidence_figure",
            "format": "png",
            "figure_path": str(INTEGRATED_EVIDENCE_FIGURE_PNG),
            "exists": INTEGRATED_EVIDENCE_FIGURE_PNG.exists(),
        },
        {
            "figure_name": "integrated_internal_evidence_figure",
            "format": "pdf",
            "figure_path": str(INTEGRATED_EVIDENCE_FIGURE_PDF),
            "exists": INTEGRATED_EVIDENCE_FIGURE_PDF.exists(),
        },
        {
            "figure_name": "integrated_internal_evidence_figure",
            "format": "svg",
            "figure_path": str(INTEGRATED_EVIDENCE_FIGURE_SVG),
            "exists": INTEGRATED_EVIDENCE_FIGURE_SVG.exists(),
        },
    ]
)

figure_manifest.to_csv(INTEGRATED_EVIDENCE_FIGURE_MANIFEST_CSV, index=False)

hybrid_row = plot_table[
    plot_table["model_id"] == "hybrid__full_image_plus_od_centered_2p5x_effb0"
].iloc[0]

outside_control_row = plot_table[
    plot_table["model_id"] == "anatomy__outside_od_only_trained_control"
].iloc[0]

od25_row = plot_table[
    plot_table["model_id"] == "anatomy__od_centered_crop_2p5x"
].iloc[0]

deit_row = plot_table[
    plot_table["model_id"] == "full_image__DeiTSmall"
].iloc[0]

figure_summary = {
    "notebook": "12_integrated_internal_model_audit_and_freezing",
    "cell": "03_generate_integrated_internal_evidence_figure",
    "status": "PASS",
    "main_figure_png": str(INTEGRATED_EVIDENCE_FIGURE_PNG),
    "main_figure_pdf": str(INTEGRATED_EVIDENCE_FIGURE_PDF),
    "main_figure_svg": str(INTEGRATED_EVIDENCE_FIGURE_SVG),
    "figure_manifest_csv": str(INTEGRATED_EVIDENCE_FIGURE_MANIFEST_CSV),
    "models_included": MODEL_ORDER,
    "main_interpretation": (
        "The hybrid model preserves ceiling-level patient discrimination with low Brier score and stable patient-level behavior. "
        "However, the outside-OD shortcut-control remains strong, and the OD-only stress-test has excellent internal probabilistic metrics under an artificial input distribution. "
        "Therefore, anatomical XAI and shortcut-aware interpretation remain required before model freezing."
    ),
    "key_values": {
        "hybrid_patient_brier_score": float(hybrid_row["integrated_patient_brier_score"]),
        "hybrid_ece": float(hybrid_row["ece_equal_width_5bin"]),
        "hybrid_mean_within_patient_prob_std": float(hybrid_row["mean_within_patient_prob_std"]),
        "outside_od_control_patient_brier_score": float(outside_control_row["integrated_patient_brier_score"]),
        "od_centered_2p5x_patient_brier_score": float(od25_row["integrated_patient_brier_score"]),
        "deitsmall_patient_brier_score": float(deit_row["integrated_patient_brier_score"]),
    },
    "next_step": (
        "Proceed to XAI preparation for the hybrid, OD-centered 2.5x model, and outside-OD shortcut-control."
    ),
}

with open(INTEGRATED_EVIDENCE_FIGURE_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(figure_summary, f, indent=2)

print("Integrated evidence figure manifest:")
print(figure_manifest.to_string(index=False))

print("\nFigure summary:")
print(json.dumps(figure_summary, indent=2))

print(f"\nIntegrated evidence figure PNG: {INTEGRATED_EVIDENCE_FIGURE_PNG}")
print(f"Integrated evidence figure PDF: {INTEGRATED_EVIDENCE_FIGURE_PDF}")
print(f"Integrated evidence figure SVG: {INTEGRATED_EVIDENCE_FIGURE_SVG}")
print(f"Figure manifest CSV: {INTEGRATED_EVIDENCE_FIGURE_MANIFEST_CSV}")
print(f"Figure summary JSON: {INTEGRATED_EVIDENCE_FIGURE_SUMMARY_JSON}")

if not figure_manifest["exists"].astype(bool).all():
    missing_figures = figure_manifest.loc[
        ~figure_manifest["exists"].astype(bool),
        "figure_path",
    ].tolist()
    raise RuntimeError(f"Missing integrated evidence figures: {missing_figures}")

required_outputs = [
    INTEGRATED_EVIDENCE_FIGURE_PNG,
    INTEGRATED_EVIDENCE_FIGURE_PDF,
    INTEGRATED_EVIDENCE_FIGURE_SVG,
    INTEGRATED_EVIDENCE_FIGURE_MANIFEST_CSV,
    INTEGRATED_EVIDENCE_FIGURE_SUMMARY_JSON,
]

missing_outputs = [
    str(path)
    for path in required_outputs
    if not Path(path).exists()
]

if missing_outputs:
    raise RuntimeError(f"Missing Cell 3 outputs: {missing_outputs}")

print("Status: PASS")

In [ ]:
# Cell 4: XAI runtime registry and smoke test for priority models

import json
import time
import warnings
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import timm

warnings.filterwarnings("ignore")

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

XAI_RUNTIME_ROOT = RESULTS_12_DIR / "xai_runtime_priority_models"
XAI_RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)

XAI_TARGET_REGISTRY_CSV = (
    RESULTS_12_DIR
    / "notebook12_xai_target_registry.csv"
)

HYBRID_SELECTION_JSON = (
    RESULTS_11_DIR
    / "notebook11_validation_selected_hybrid_model.json"
)

HYBRID_PAIRED_INPUT_MANIFEST_CSV = (
    PROJECT_DATA_DIR
    / "hygd_hybrid_anatomy_aware_classifier_inputs"
    / "notebook11_hybrid_full_image_od25x_paired_input_manifest.csv"
)

ANATOMY_STRICT_INPUT_MANIFEST_CSV = (
    PROJECT_DATA_DIR
    / "hygd_anatomy_constrained_classifier_inputs"
    / "notebook10_strict_anatomy_constrained_input_manifest.csv"
)

OUTSIDE_OD_INPUT_MANIFEST_CSV = (
    PROJECT_DATA_DIR
    / "hygd_anatomy_constrained_classifier_inputs"
    / "notebook10_outside_od_only_trained_control_input_manifest.csv"
)

SPLIT_MANIFEST_CSV = (
    PROJECT_DATA_DIR
    / "hygd_glaucoma_classifier_splits"
    / "hygd_image_level_split_manifest_seed42.csv"
)

ANALYSIS_MANIFEST_CSV = (
    PROJECT_DATA_DIR
    / "hygd_final_od_oc_proxy"
    / "hygd_final_od_oc_proxy_analysis_manifest.csv"
)

LOCAL_HYGD_IMAGE_CACHE = Path("/content/HYGD_local/Images")

XAI_PRIORITY_MODEL_RUNTIME_REGISTRY_CSV = (
    XAI_RUNTIME_ROOT
    / "notebook12_xai_priority_model_runtime_registry.csv"
)

XAI_PRIORITY_MODEL_RUNTIME_REGISTRY_JSON = (
    XAI_RUNTIME_ROOT
    / "notebook12_xai_priority_model_runtime_registry.json"
)

XAI_SMOKE_TEST_CASES_CSV = (
    XAI_RUNTIME_ROOT
    / "notebook12_xai_priority_smoke_test_cases.csv"
)

XAI_SMOKE_TEST_RESULTS_CSV = (
    XAI_RUNTIME_ROOT
    / "notebook12_xai_priority_smoke_test_results.csv"
)

XAI_TARGET_LAYER_AUDIT_CSV = (
    XAI_RUNTIME_ROOT
    / "notebook12_xai_target_layer_audit.csv"
)

XAI_RUNTIME_CONFIG_JSON = (
    XAI_RUNTIME_ROOT
    / "notebook12_xai_runtime_config.json"
)

XAI_INPUT_SIZE = 384
XAI_SMOKE_CASES_PER_MODEL = 2

required_inputs = [
    XAI_TARGET_REGISTRY_CSV,
    HYBRID_SELECTION_JSON,
    HYBRID_PAIRED_INPUT_MANIFEST_CSV,
    ANATOMY_STRICT_INPUT_MANIFEST_CSV,
    OUTSIDE_OD_INPUT_MANIFEST_CSV,
    SPLIT_MANIFEST_CSV,
    ANALYSIS_MANIFEST_CSV,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not Path(path).exists()
]

if missing_inputs:
    raise FileNotFoundError(f"Missing Cell 4 inputs: {missing_inputs}")

xai_target_registry = pd.read_csv(XAI_TARGET_REGISTRY_CSV)
split_manifest = pd.read_csv(SPLIT_MANIFEST_CSV)
analysis_manifest = pd.read_csv(ANALYSIS_MANIFEST_CSV)

with open(HYBRID_SELECTION_JSON, "r", encoding="utf-8") as f:
    hybrid_selection = json.load(f)

hybrid_manifest = pd.read_csv(HYBRID_PAIRED_INPUT_MANIFEST_CSV)
anatomy_manifest = pd.read_csv(ANATOMY_STRICT_INPUT_MANIFEST_CSV)
outside_od_manifest = pd.read_csv(OUTSIDE_OD_INPUT_MANIFEST_CSV)

for df in [
    split_manifest,
    analysis_manifest,
    hybrid_manifest,
    anatomy_manifest,
    outside_od_manifest,
]:
    if "sample_id" in df.columns:
        df["sample_id"] = df["sample_id"].astype(str)
    if "patient_id" in df.columns:
        df["patient_id"] = df["patient_id"].astype(str)

split_manifest["glaucoma_split"] = split_manifest["glaucoma_split"].astype(str)

test_samples = split_manifest[
    split_manifest["glaucoma_split"] == "test"
][
    [
        "sample_id",
        "patient_id",
        "glaucoma_split",
    ]
].drop_duplicates().copy()

metadata_cols = [
    "sample_id",
    "patient_id",
    "gon_binary",
    "gon_label",
    "quality_score",
    "area_cdr_proxy",
]

available_metadata_cols = [
    col for col in metadata_cols
    if col in analysis_manifest.columns
]

test_metadata = test_samples.merge(
    analysis_manifest[available_metadata_cols],
    on=["sample_id", "patient_id"],
    how="left",
    validate="one_to_one",
)

if "gon_binary" not in test_metadata.columns and "gon_label" not in test_metadata.columns:
    raise ValueError("Missing gon_binary/gon_label after merging test metadata.")

if "gon_binary" in test_metadata.columns:
    test_metadata["gon_binary"] = pd.to_numeric(
        test_metadata["gon_binary"],
        errors="coerce",
    )

if "quality_score" not in test_metadata.columns:
    test_metadata["quality_score"] = np.nan

if "area_cdr_proxy" not in test_metadata.columns:
    test_metadata["area_cdr_proxy"] = np.nan

test_metadata["quality_score"] = pd.to_numeric(
    test_metadata["quality_score"],
    errors="coerce",
)

test_metadata["area_cdr_proxy"] = pd.to_numeric(
    test_metadata["area_cdr_proxy"],
    errors="coerce",
)


def safe_exists(path):
    try:
        return Path(str(path)).exists()
    except Exception:
        return False


def candidate_full_image_paths(image_path):
    image_path = str(image_path)
    candidates = []

    local_candidate = LOCAL_HYGD_IMAGE_CACHE / Path(image_path).name
    candidates.append(str(local_candidate))
    candidates.append(image_path)

    unique = []
    seen = set()

    for item in candidates:
        if item not in seen:
            unique.append(item)
            seen.add(item)

    return unique


def read_rgb_image_robust(
    image_path,
    use_local_candidate=False,
    n_attempts=3,
    sleep_sec=0.20,
):
    if use_local_candidate:
        candidates = candidate_full_image_paths(image_path)
    else:
        candidates = [str(image_path)]

    last_error = ""

    for candidate in candidates:
        for _ in range(n_attempts):
            try:
                image_bgr = cv2.imread(str(candidate), cv2.IMREAD_COLOR)

                if image_bgr is not None:
                    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
                    return image_rgb, str(candidate)

                last_error = f"cv2.imread returned None for {candidate}"

            except Exception as exc:
                last_error = repr(exc)

            time.sleep(sleep_sec)

    raise FileNotFoundError(
        f"Could not read image from candidates for {image_path}. Last error: {last_error}"
    )


def detect_path_column(df, preferred_columns=None):
    if preferred_columns is None:
        preferred_columns = []

    for col in preferred_columns:
        if col in df.columns:
            non_null = df[col].dropna().astype(str)
            if len(non_null) > 0:
                return col

    path_like_cols = [
        col for col in df.columns
        if "path" in str(col).lower()
    ]

    if len(path_like_cols) == 0:
        raise ValueError("No path-like columns found.")

    existence_scores = []

    sample_df = df.head(20).copy()

    for col in path_like_cols:
        values = sample_df[col].dropna().astype(str).tolist()
        n_existing = 0

        for value in values:
            if safe_exists(value):
                n_existing += 1

        existence_scores.append(
            {
                "column": col,
                "n_existing_in_head20": n_existing,
                "n_non_null_in_head20": len(values),
            }
        )

    scores_df = pd.DataFrame(existence_scores)

    scores_df = scores_df.sort_values(
        ["n_existing_in_head20", "n_non_null_in_head20"],
        ascending=[False, False],
    )

    return str(scores_df.iloc[0]["column"])


def get_merged_value(row, base_name, default=np.nan):
    candidate_columns = [
        base_name,
        f"{base_name}_x",
        f"{base_name}_y",
        f"{base_name}_metadata",
        f"{base_name}_manifest",
    ]

    for col in candidate_columns:
        if col in row.index:
            value = row[col]
            if pd.notna(value):
                return value

    return default


def get_binary_label_from_row(row):
    value = get_merged_value(row, "gon_binary", default=np.nan)

    if pd.notna(value):
        return int(float(value))

    label_value = get_merged_value(row, "gon_label", default="")
    label_value = str(label_value).strip().upper()

    if label_value in ["GON+", "1", "POSITIVE", "GLAUCOMA"]:
        return 1

    if label_value in ["GON-", "0", "NEGATIVE", "NORMAL"]:
        return 0

    raise KeyError(
        "Could not resolve binary label from merged row. "
        f"Available columns: {list(row.index)}"
    )


def get_quality_from_row(row):
    value = get_merged_value(row, "quality_score", default=np.nan)

    if pd.notna(value):
        return float(value)

    return np.nan


def find_last_conv2d_layer(model):
    last_name = None
    last_module = None

    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            last_name = name
            last_module = module

    if last_module is None:
        raise RuntimeError("No Conv2d layer found for CAM target.")

    return last_name, last_module


def strip_state_dict_prefixes(state_dict):
    cleaned = {}

    for key, value in state_dict.items():
        new_key = str(key)

        for prefix in ["module.", "model."]:
            if new_key.startswith(prefix):
                new_key = new_key[len(prefix):]

        cleaned[new_key] = value

    return cleaned


def extract_state_dict(checkpoint):
    if isinstance(checkpoint, dict):
        for key in ["model_state_dict", "state_dict", "model"]:
            if key in checkpoint and isinstance(checkpoint[key], dict):
                return strip_state_dict_prefixes(checkpoint[key])

    if isinstance(checkpoint, dict):
        return strip_state_dict_prefixes(checkpoint)

    raise RuntimeError("Unsupported checkpoint format.")


class HybridEfficientNetB0Classifier(nn.Module):
    def __init__(
        self,
        full_backbone_name="efficientnet_b0",
        anatomy_backbone_name="efficientnet_b0",
        pretrained=False,
        dropout=0.35,
        hidden_dim=512,
    ):
        super().__init__()

        self.full_branch = timm.create_model(
            full_backbone_name,
            pretrained=pretrained,
            num_classes=0,
            global_pool="avg",
        )

        self.anatomy_branch = timm.create_model(
            anatomy_backbone_name,
            pretrained=pretrained,
            num_classes=0,
            global_pool="avg",
        )

        full_dim = int(self.full_branch.num_features)
        anatomy_dim = int(self.anatomy_branch.num_features)
        fusion_dim = full_dim + anatomy_dim

        self.full_dim = full_dim
        self.anatomy_dim = anatomy_dim
        self.fusion_dim = fusion_dim

        self.fusion_head = nn.Sequential(
            nn.LayerNorm(fusion_dim),
            nn.Dropout(dropout),
            nn.Linear(fusion_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, full_image, anatomy_image, return_features=False):
        full_features = self.full_branch(full_image)
        anatomy_features = self.anatomy_branch(anatomy_image)

        fused_features = torch.cat(
            [
                full_features,
                anatomy_features,
            ],
            dim=1,
        )

        logits = self.fusion_head(fused_features).squeeze(1)

        if return_features:
            return {
                "logits": logits,
                "full_features": full_features,
                "anatomy_features": anatomy_features,
                "fused_features": fused_features,
            }

        return logits


def create_binary_efficientnet_b0():
    model = timm.create_model(
        "efficientnet_b0",
        pretrained=False,
        num_classes=1,
    )

    return model


def load_binary_model(checkpoint_path):
    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=False,
    )

    state_dict = extract_state_dict(checkpoint)

    model = create_binary_efficientnet_b0()

    load_result = model.load_state_dict(state_dict, strict=False)

    model = model.to(DEVICE)
    model.eval()

    return model, checkpoint, load_result


def load_hybrid_model(checkpoint_path):
    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=False,
    )

    state_dict = extract_state_dict(checkpoint)

    model = HybridEfficientNetB0Classifier(pretrained=False)

    load_result = model.load_state_dict(state_dict, strict=False)

    model = model.to(DEVICE)
    model.eval()

    return model, checkpoint, load_result


xai_transform = transforms.Compose(
    [
        transforms.Resize((XAI_INPUT_SIZE, XAI_INPUT_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ]
)


def image_to_tensor(image_path, use_local_candidate=False):
    image_rgb, resolved_path = read_rgb_image_robust(
        image_path,
        use_local_candidate=use_local_candidate,
    )

    pil_image = Image.fromarray(image_rgb)
    tensor = xai_transform(pil_image).unsqueeze(0)

    return tensor, resolved_path


def sigmoid_np(logits):
    return torch.sigmoid(logits.detach()).cpu().numpy().reshape(-1)


hybrid_checkpoint = Path(hybrid_selection["selected_best_checkpoint"])

od25_checkpoint = (
    MODELS_DIR
    / "hygd_anatomy_constrained_classifiers"
    / "training_runs"
    / "od_centered_crop_2p5x"
    / "od_centered_crop_2p5x_seed42"
    / "best_checkpoint.pt"
)

outside_od_checkpoint = (
    MODELS_DIR
    / "hygd_anatomy_constrained_classifiers"
    / "training_runs"
    / "outside_od_only_trained_control"
    / "outside_od_only_trained_control_seed42"
    / "best_checkpoint.pt"
)

priority_model_specs = [
    {
        "model_id": "hybrid__full_image_plus_od_centered_2p5x_effb0",
        "runtime_model_type": "hybrid_two_branch_efficientnet_b0",
        "checkpoint_path": str(hybrid_checkpoint),
        "input_contract": "full_image_path + od_centered_2p5x_image_path",
        "xai_priority": "primary_required",
    },
    {
        "model_id": "anatomy__od_centered_crop_2p5x",
        "runtime_model_type": "single_branch_efficientnet_b0",
        "checkpoint_path": str(od25_checkpoint),
        "input_contract": "od_centered_2p5x_constrained_image_path",
        "xai_priority": "primary_required",
    },
    {
        "model_id": "anatomy__outside_od_only_trained_control",
        "runtime_model_type": "single_branch_efficientnet_b0",
        "checkpoint_path": str(outside_od_checkpoint),
        "input_contract": "outside_od_only_trained_control_image_path",
        "xai_priority": "primary_required_control",
    },
]

runtime_registry_rows = []

for spec in priority_model_specs:
    runtime_registry_rows.append(
        {
            **spec,
            "checkpoint_exists": Path(spec["checkpoint_path"]).exists(),
        }
    )

runtime_registry = pd.DataFrame(runtime_registry_rows)

if not runtime_registry["checkpoint_exists"].astype(bool).all():
    print(runtime_registry.to_string(index=False))
    missing_checkpoints = runtime_registry.loc[
        ~runtime_registry["checkpoint_exists"].astype(bool),
        "checkpoint_path",
    ].tolist()
    raise FileNotFoundError(f"Missing XAI priority checkpoints: {missing_checkpoints}")

runtime_registry.to_csv(XAI_PRIORITY_MODEL_RUNTIME_REGISTRY_CSV, index=False)

with open(XAI_PRIORITY_MODEL_RUNTIME_REGISTRY_JSON, "w", encoding="utf-8") as f:
    json.dump(runtime_registry.to_dict(orient="records"), f, indent=2)

hybrid_test = hybrid_manifest.merge(
    test_metadata,
    on=["sample_id", "patient_id"],
    how="inner",
    validate="one_to_one",
)

split_columns = [
    col for col in hybrid_test.columns
    if col.startswith("glaucoma_split")
]

if len(split_columns) > 0:
    hybrid_test = hybrid_test[
        hybrid_test[split_columns].astype(str).eq("test").any(axis=1)
    ].copy()

if "glaucoma_split" not in hybrid_test.columns:
    for col in ["glaucoma_split_x", "glaucoma_split_y"]:
        if col in hybrid_test.columns:
            hybrid_test["glaucoma_split"] = hybrid_test[col].astype(str)
            break

od25_test = anatomy_manifest[
    anatomy_manifest["experiment_id"].astype(str).eq("od_centered_crop_2p5x")
].copy()

od25_test = od25_test.merge(
    test_metadata,
    on=["sample_id", "patient_id"],
    how="inner",
    validate="one_to_one",
)

outside_test = outside_od_manifest.merge(
    test_metadata,
    on=["sample_id", "patient_id"],
    how="inner",
    validate="one_to_one",
)

if len(hybrid_test) == 0:
    raise RuntimeError("Hybrid test manifest is empty.")

if len(od25_test) == 0:
    raise RuntimeError("OD-centered 2.5x test manifest is empty.")

if len(outside_test) == 0:
    raise RuntimeError("Outside-OD control test manifest is empty.")

od25_path_col = detect_path_column(
    od25_test,
    preferred_columns=[
        "constrained_image_path",
        "od_centered_2p5x_image_path",
        "input_image_path",
        "output_image_path",
        "image_path",
    ],
)

outside_path_col = detect_path_column(
    outside_test,
    preferred_columns=[
        "constrained_image_path",
        "outside_od_image_path",
        "outside_od_only_image_path",
        "input_image_path",
        "output_image_path",
        "image_path",
    ],
)

hybrid_required_cols = [
    "full_image_path",
    "od_centered_2p5x_image_path",
]

missing_hybrid_cols = [
    col for col in hybrid_required_cols
    if col not in hybrid_test.columns
]

if missing_hybrid_cols:
    raise ValueError(f"Missing hybrid manifest columns: {missing_hybrid_cols}")

smoke_cases = []

for _, row in hybrid_test.head(XAI_SMOKE_CASES_PER_MODEL).iterrows():
    smoke_cases.append(
        {
            "model_id": "hybrid__full_image_plus_od_centered_2p5x_effb0",
            "sample_id": str(row["sample_id"]),
            "patient_id": str(row["patient_id"]),
            "y_true": get_binary_label_from_row(row),
            "quality_score": get_quality_from_row(row),
            "full_image_path": str(row["full_image_path"]),
            "anatomy_image_path": str(row["od_centered_2p5x_image_path"]),
        }
    )

for _, row in od25_test.head(XAI_SMOKE_CASES_PER_MODEL).iterrows():
    smoke_cases.append(
        {
            "model_id": "anatomy__od_centered_crop_2p5x",
            "sample_id": str(row["sample_id"]),
            "patient_id": str(row["patient_id"]),
            "y_true": get_binary_label_from_row(row),
            "quality_score": get_quality_from_row(row),
            "single_image_path": str(row[od25_path_col]),
        }
    )

for _, row in outside_test.head(XAI_SMOKE_CASES_PER_MODEL).iterrows():
    smoke_cases.append(
        {
            "model_id": "anatomy__outside_od_only_trained_control",
            "sample_id": str(row["sample_id"]),
            "patient_id": str(row["patient_id"]),
            "y_true": get_binary_label_from_row(row),
            "quality_score": get_quality_from_row(row),
            "single_image_path": str(row[outside_path_col]),
        }
    )

smoke_cases_df = pd.DataFrame(smoke_cases)
smoke_cases_df.to_csv(XAI_SMOKE_TEST_CASES_CSV, index=False)

target_layer_rows = []
smoke_result_rows = []

hybrid_model, hybrid_checkpoint_payload, hybrid_load_result = load_hybrid_model(hybrid_checkpoint)
od25_model, od25_checkpoint_payload, od25_load_result = load_binary_model(od25_checkpoint)
outside_model, outside_checkpoint_payload, outside_load_result = load_binary_model(outside_od_checkpoint)

hybrid_full_layer_name, hybrid_full_layer = find_last_conv2d_layer(hybrid_model.full_branch)
hybrid_od_layer_name, hybrid_od_layer = find_last_conv2d_layer(hybrid_model.anatomy_branch)
od25_layer_name, od25_layer = find_last_conv2d_layer(od25_model)
outside_layer_name, outside_layer = find_last_conv2d_layer(outside_model)

target_layer_rows.extend(
    [
        {
            "model_id": "hybrid__full_image_plus_od_centered_2p5x_effb0",
            "branch": "full_image_branch",
            "target_layer_name": f"full_branch.{hybrid_full_layer_name}",
            "target_layer_type": type(hybrid_full_layer).__name__,
        },
        {
            "model_id": "hybrid__full_image_plus_od_centered_2p5x_effb0",
            "branch": "od_centered_branch",
            "target_layer_name": f"anatomy_branch.{hybrid_od_layer_name}",
            "target_layer_type": type(hybrid_od_layer).__name__,
        },
        {
            "model_id": "anatomy__od_centered_crop_2p5x",
            "branch": "single_branch",
            "target_layer_name": od25_layer_name,
            "target_layer_type": type(od25_layer).__name__,
        },
        {
            "model_id": "anatomy__outside_od_only_trained_control",
            "branch": "single_branch",
            "target_layer_name": outside_layer_name,
            "target_layer_type": type(outside_layer).__name__,
        },
    ]
)

target_layer_audit = pd.DataFrame(target_layer_rows)
target_layer_audit.to_csv(XAI_TARGET_LAYER_AUDIT_CSV, index=False)

for _, row in smoke_cases_df.iterrows():
    model_id = str(row["model_id"])

    try:
        if model_id == "hybrid__full_image_plus_od_centered_2p5x_effb0":
            full_tensor, full_resolved_path = image_to_tensor(
                row["full_image_path"],
                use_local_candidate=True,
            )

            od_tensor, od_resolved_path = image_to_tensor(
                row["anatomy_image_path"],
                use_local_candidate=False,
            )

            full_tensor = full_tensor.to(DEVICE)
            od_tensor = od_tensor.to(DEVICE)

            with torch.no_grad():
                logits = hybrid_model(full_tensor, od_tensor)
                prob = float(sigmoid_np(logits)[0])

            smoke_result_rows.append(
                {
                    "model_id": model_id,
                    "sample_id": str(row["sample_id"]),
                    "patient_id": str(row["patient_id"]),
                    "y_true": int(row["y_true"]),
                    "probability": prob,
                    "logit": float(logits.detach().cpu().numpy().reshape(-1)[0]),
                    "full_resolved_path": full_resolved_path,
                    "anatomy_resolved_path": od_resolved_path,
                    "single_resolved_path": "",
                    "status": "pass",
                    "error": "",
                }
            )

        elif model_id == "anatomy__od_centered_crop_2p5x":
            tensor, resolved_path = image_to_tensor(
                row["single_image_path"],
                use_local_candidate=False,
            )

            tensor = tensor.to(DEVICE)

            with torch.no_grad():
                logits = od25_model(tensor)
                prob = float(sigmoid_np(logits)[0])

            smoke_result_rows.append(
                {
                    "model_id": model_id,
                    "sample_id": str(row["sample_id"]),
                    "patient_id": str(row["patient_id"]),
                    "y_true": int(row["y_true"]),
                    "probability": prob,
                    "logit": float(logits.detach().cpu().numpy().reshape(-1)[0]),
                    "full_resolved_path": "",
                    "anatomy_resolved_path": "",
                    "single_resolved_path": resolved_path,
                    "status": "pass",
                    "error": "",
                }
            )

        elif model_id == "anatomy__outside_od_only_trained_control":
            tensor, resolved_path = image_to_tensor(
                row["single_image_path"],
                use_local_candidate=False,
            )

            tensor = tensor.to(DEVICE)

            with torch.no_grad():
                logits = outside_model(tensor)
                prob = float(sigmoid_np(logits)[0])

            smoke_result_rows.append(
                {
                    "model_id": model_id,
                    "sample_id": str(row["sample_id"]),
                    "patient_id": str(row["patient_id"]),
                    "y_true": int(row["y_true"]),
                    "probability": prob,
                    "logit": float(logits.detach().cpu().numpy().reshape(-1)[0]),
                    "full_resolved_path": "",
                    "anatomy_resolved_path": "",
                    "single_resolved_path": resolved_path,
                    "status": "pass",
                    "error": "",
                }
            )

        else:
            raise ValueError(f"Unknown smoke-test model: {model_id}")

    except Exception as exc:
        smoke_result_rows.append(
            {
                "model_id": model_id,
                "sample_id": str(row.get("sample_id", "")),
                "patient_id": str(row.get("patient_id", "")),
                "y_true": int(row.get("y_true", -1)),
                "probability": np.nan,
                "logit": np.nan,
                "full_resolved_path": "",
                "anatomy_resolved_path": "",
                "single_resolved_path": "",
                "status": "fail",
                "error": repr(exc),
            }
        )

smoke_results = pd.DataFrame(smoke_result_rows)
smoke_results.to_csv(XAI_SMOKE_TEST_RESULTS_CSV, index=False)

del hybrid_model, od25_model, outside_model

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

if not smoke_results["status"].eq("pass").all():
    print(smoke_results.to_string(index=False))
    raise RuntimeError("At least one XAI priority smoke test failed.")

runtime_config = {
    "notebook": "12_integrated_internal_model_audit_and_freezing",
    "cell": "04_xai_runtime_registry_and_smoke_test",
    "status": "PASS",
    "device": str(DEVICE),
    "input_size": int(XAI_INPUT_SIZE),
    "smoke_cases_per_model": int(XAI_SMOKE_CASES_PER_MODEL),
    "priority_models": runtime_registry.to_dict(orient="records"),
    "detected_path_columns": {
        "od_centered_crop_2p5x": od25_path_col,
        "outside_od_only_trained_control": outside_path_col,
    },
    "target_layers": target_layer_audit.to_dict(orient="records"),
    "outputs": {
        "runtime_registry_csv": str(XAI_PRIORITY_MODEL_RUNTIME_REGISTRY_CSV),
        "runtime_registry_json": str(XAI_PRIORITY_MODEL_RUNTIME_REGISTRY_JSON),
        "smoke_test_cases_csv": str(XAI_SMOKE_TEST_CASES_CSV),
        "smoke_test_results_csv": str(XAI_SMOKE_TEST_RESULTS_CSV),
        "target_layer_audit_csv": str(XAI_TARGET_LAYER_AUDIT_CSV),
        "runtime_config_json": str(XAI_RUNTIME_CONFIG_JSON),
    },
}

with open(XAI_RUNTIME_CONFIG_JSON, "w", encoding="utf-8") as f:
    json.dump(runtime_config, f, indent=2)

print("XAI runtime registry:")
print(runtime_registry.to_string(index=False))

print("\nDetected input path columns:")
print(f"OD-centered 2.5x: {od25_path_col}")
print(f"Outside-OD control: {outside_path_col}")

print("\nXAI target layer audit:")
print(target_layer_audit.to_string(index=False))

print("\nXAI smoke-test cases:")
print(smoke_cases_df.to_string(index=False))

print("\nXAI smoke-test results:")
print(
    smoke_results[
        [
            "model_id",
            "sample_id",
            "patient_id",
            "y_true",
            "probability",
            "status",
            "error",
        ]
    ].to_string(index=False)
)

print(f"\nRuntime registry CSV: {XAI_PRIORITY_MODEL_RUNTIME_REGISTRY_CSV}")
print(f"Runtime registry JSON: {XAI_PRIORITY_MODEL_RUNTIME_REGISTRY_JSON}")
print(f"Smoke-test cases CSV: {XAI_SMOKE_TEST_CASES_CSV}")
print(f"Smoke-test results CSV: {XAI_SMOKE_TEST_RESULTS_CSV}")
print(f"Target layer audit CSV: {XAI_TARGET_LAYER_AUDIT_CSV}")
print(f"Runtime config JSON: {XAI_RUNTIME_CONFIG_JSON}")

required_outputs = [
    XAI_PRIORITY_MODEL_RUNTIME_REGISTRY_CSV,
    XAI_PRIORITY_MODEL_RUNTIME_REGISTRY_JSON,
    XAI_SMOKE_TEST_CASES_CSV,
    XAI_SMOKE_TEST_RESULTS_CSV,
    XAI_TARGET_LAYER_AUDIT_CSV,
    XAI_RUNTIME_CONFIG_JSON,
]

missing_outputs = [
    str(path)
    for path in required_outputs
    if not Path(path).exists()
]

if missing_outputs:
    raise RuntimeError(f"Missing Cell 4 outputs: {missing_outputs}")

if len(smoke_results) != XAI_SMOKE_CASES_PER_MODEL * 3:
    raise RuntimeError("Unexpected number of XAI smoke-test results.")

if not np.isfinite(smoke_results["probability"].astype(float)).all():
    raise RuntimeError("Non-finite probabilities found in XAI smoke test.")

print("Status: PASS")

In [ ]:
# Cell 5: Build branch-specific anatomical mask manifest for priority XAI models

import json
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

XAI_RUNTIME_ROOT = RESULTS_12_DIR / "xai_runtime_priority_models"
XAI_MASK_ROOT = XAI_RUNTIME_ROOT / "anatomical_masks_384"
XAI_MASK_ROOT.mkdir(parents=True, exist_ok=True)

XAI_RUNTIME_CONFIG_JSON = (
    XAI_RUNTIME_ROOT
    / "notebook12_xai_runtime_config.json"
)

HYBRID_PAIRED_INPUT_MANIFEST_CSV = (
    PROJECT_DATA_DIR
    / "hygd_hybrid_anatomy_aware_classifier_inputs"
    / "notebook11_hybrid_full_image_od25x_paired_input_manifest.csv"
)

ANATOMY_STRICT_INPUT_MANIFEST_CSV = (
    PROJECT_DATA_DIR
    / "hygd_anatomy_constrained_classifier_inputs"
    / "notebook10_strict_anatomy_constrained_input_manifest.csv"
)

OUTSIDE_OD_INPUT_MANIFEST_CSV = (
    PROJECT_DATA_DIR
    / "hygd_anatomy_constrained_classifier_inputs"
    / "notebook10_outside_od_only_trained_control_input_manifest.csv"
)

SPLIT_MANIFEST_CSV = (
    PROJECT_DATA_DIR
    / "hygd_glaucoma_classifier_splits"
    / "hygd_image_level_split_manifest_seed42.csv"
)

ANALYSIS_MANIFEST_CSV = (
    PROJECT_DATA_DIR
    / "hygd_final_od_oc_proxy"
    / "hygd_final_od_oc_proxy_analysis_manifest.csv"
)

LOCAL_HYGD_IMAGE_CACHE = Path("/content/HYGD_local/Images")

XAI_MASK_MANIFEST_CSV = (
    XAI_RUNTIME_ROOT
    / "notebook12_xai_priority_anatomical_mask_manifest.csv"
)

XAI_MASK_SUMMARY_CSV = (
    XAI_RUNTIME_ROOT
    / "notebook12_xai_priority_anatomical_mask_summary.csv"
)

XAI_MASK_AUDIT_CSV = (
    XAI_RUNTIME_ROOT
    / "notebook12_xai_priority_anatomical_mask_audit.csv"
)

XAI_MASK_VISUAL_CASES_CSV = (
    XAI_RUNTIME_ROOT
    / "notebook12_xai_priority_anatomical_mask_visual_cases.csv"
)

XAI_MASK_VISUAL_PANEL_PNG = (
    XAI_RUNTIME_ROOT
    / "notebook12_xai_priority_anatomical_mask_visual_panel.png"
)

XAI_MASK_VISUAL_PANEL_PDF = (
    XAI_RUNTIME_ROOT
    / "notebook12_xai_priority_anatomical_mask_visual_panel.pdf"
)

XAI_MASK_BUILD_SUMMARY_JSON = (
    XAI_RUNTIME_ROOT
    / "notebook12_xai_priority_anatomical_mask_build_summary.json"
)

XAI_INPUT_SIZE = 384
OD_CENTERED_CROP_FACTOR = 2.5

required_inputs = [
    XAI_RUNTIME_CONFIG_JSON,
    HYBRID_PAIRED_INPUT_MANIFEST_CSV,
    ANATOMY_STRICT_INPUT_MANIFEST_CSV,
    OUTSIDE_OD_INPUT_MANIFEST_CSV,
    SPLIT_MANIFEST_CSV,
    ANALYSIS_MANIFEST_CSV,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not Path(path).exists()
]

if missing_inputs:
    raise FileNotFoundError(f"Missing Cell 5 inputs: {missing_inputs}")

with open(XAI_RUNTIME_CONFIG_JSON, "r", encoding="utf-8") as f:
    xai_runtime_config = json.load(f)

hybrid_manifest = pd.read_csv(HYBRID_PAIRED_INPUT_MANIFEST_CSV)
anatomy_manifest = pd.read_csv(ANATOMY_STRICT_INPUT_MANIFEST_CSV)
outside_manifest = pd.read_csv(OUTSIDE_OD_INPUT_MANIFEST_CSV)
split_manifest = pd.read_csv(SPLIT_MANIFEST_CSV)
analysis_manifest = pd.read_csv(ANALYSIS_MANIFEST_CSV)

for df in [
    hybrid_manifest,
    anatomy_manifest,
    outside_manifest,
    split_manifest,
    analysis_manifest,
]:
    if "sample_id" in df.columns:
        df["sample_id"] = df["sample_id"].astype(str)
    if "patient_id" in df.columns:
        df["patient_id"] = df["patient_id"].astype(str)

split_manifest["glaucoma_split"] = split_manifest["glaucoma_split"].astype(str)

required_analysis_cols = [
    "sample_id",
    "patient_id",
    "mask_512_path",
    "od_centroid_x_512",
    "od_centroid_y_512",
    "od_area_512",
]

missing_analysis_cols = [
    col for col in required_analysis_cols
    if col not in analysis_manifest.columns
]

if missing_analysis_cols:
    raise ValueError(f"Missing required analysis manifest columns: {missing_analysis_cols}")

test_samples = split_manifest[
    split_manifest["glaucoma_split"] == "test"
][
    [
        "sample_id",
        "patient_id",
        "glaucoma_split",
    ]
].drop_duplicates().copy()


def get_merged_value(row, base_name, default=np.nan):
    candidate_columns = [
        base_name,
        f"{base_name}_x",
        f"{base_name}_y",
        f"{base_name}_metadata",
        f"{base_name}_manifest",
    ]

    for col in candidate_columns:
        if col in row.index:
            value = row[col]
            if pd.notna(value):
                return value

    return default


def get_binary_label_from_row(row):
    value = get_merged_value(row, "gon_binary", default=np.nan)

    if pd.notna(value):
        return int(float(value))

    label_value = get_merged_value(row, "gon_label", default="")
    label_value = str(label_value).strip().upper()

    if label_value in ["GON+", "1", "POSITIVE", "GLAUCOMA"]:
        return 1

    if label_value in ["GON-", "0", "NEGATIVE", "NORMAL"]:
        return 0

    raise KeyError(
        "Could not resolve binary label from merged row. "
        f"Available columns: {list(row.index)}"
    )


def candidate_full_image_paths(image_path):
    image_path = str(image_path)

    candidates = [
        str(LOCAL_HYGD_IMAGE_CACHE / Path(image_path).name),
        image_path,
    ]

    unique = []
    seen = set()

    for candidate in candidates:
        if candidate not in seen:
            unique.append(candidate)
            seen.add(candidate)

    return unique


def read_rgb_image_robust(image_path, use_local_candidate=False, n_attempts=3, sleep_sec=0.20):
    if use_local_candidate:
        candidates = candidate_full_image_paths(image_path)
    else:
        candidates = [str(image_path)]

    last_error = ""

    for candidate in candidates:
        for _ in range(n_attempts):
            try:
                image_bgr = cv2.imread(str(candidate), cv2.IMREAD_COLOR)

                if image_bgr is not None:
                    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
                    return image_rgb, str(candidate)

                last_error = f"cv2.imread returned None for {candidate}"

            except Exception as exc:
                last_error = repr(exc)

            time.sleep(sleep_sec)

    raise FileNotFoundError(
        f"Could not read RGB image from candidates for {image_path}. Last error: {last_error}"
    )


def read_joint_od_oc_mask(mask_path, n_attempts=3, sleep_sec=0.20):
    last_error = ""

    for _ in range(n_attempts):
        try:
            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

            if mask is not None:
                unique_values = sorted([int(v) for v in np.unique(mask)])
                nonzero_values = [value for value in unique_values if value > 0]

                if len(nonzero_values) == 0:
                    raise RuntimeError(f"Joint mask is empty: {mask_path}")

                if len(nonzero_values) >= 2:
                    oc_value = max(nonzero_values)
                    od_mask = (mask > 0).astype(np.uint8)
                    oc_mask = (mask == oc_value).astype(np.uint8)
                    decode_mode = f"multi_value_joint_mask_oc_is_max_value_{oc_value}"
                else:
                    od_mask = (mask > 0).astype(np.uint8)
                    oc_mask = np.zeros_like(od_mask, dtype=np.uint8)
                    decode_mode = "single_value_joint_mask_od_only_oc_unavailable"

                oc_mask = (oc_mask & od_mask).astype(np.uint8)

                return od_mask, oc_mask, unique_values, decode_mode

            last_error = f"cv2.imread returned None for {mask_path}"

        except Exception as exc:
            last_error = repr(exc)

        time.sleep(sleep_sec)

    raise FileNotFoundError(
        f"Could not read joint OD/OC mask: {mask_path}. Last error: {last_error}"
    )


def read_binary_mask(mask_path, n_attempts=3, sleep_sec=0.20):
    last_error = ""

    for _ in range(n_attempts):
        try:
            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

            if mask is not None:
                return (mask > 0).astype(np.uint8)

            last_error = f"cv2.imread returned None for {mask_path}"

        except Exception as exc:
            last_error = repr(exc)

        time.sleep(sleep_sec)

    raise FileNotFoundError(
        f"Could not read binary mask: {mask_path}. Last error: {last_error}"
    )


def resize_binary_mask(mask, output_size=XAI_INPUT_SIZE):
    resized = cv2.resize(
        mask.astype(np.uint8),
        (output_size, output_size),
        interpolation=cv2.INTER_NEAREST,
    )

    return (resized > 0).astype(np.uint8)


def infer_fov_mask_from_rgb(image_rgb):
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    fov = (gray > 8).astype(np.uint8)

    kernel = np.ones((7, 7), dtype=np.uint8)
    fov = cv2.morphologyEx(fov, cv2.MORPH_CLOSE, kernel)
    fov = cv2.morphologyEx(fov, cv2.MORPH_OPEN, kernel)

    return fov.astype(np.uint8)


def crop_mask_with_box(mask, crop_box):
    h, w = mask.shape[:2]

    x0 = int(round(float(crop_box["x0"])))
    y0 = int(round(float(crop_box["y0"])))
    x1 = int(round(float(crop_box["x1"])))
    y1 = int(round(float(crop_box["y1"])))

    crop_w = max(1, x1 - x0)
    crop_h = max(1, y1 - y0)

    canvas = np.zeros((crop_h, crop_w), dtype=np.uint8)

    sx0 = max(0, x0)
    sy0 = max(0, y0)
    sx1 = min(w, x1)
    sy1 = min(h, y1)

    if sx1 <= sx0 or sy1 <= sy0:
        return canvas

    dx0 = sx0 - x0
    dy0 = sy0 - y0

    canvas[
        dy0:dy0 + (sy1 - sy0),
        dx0:dx0 + (sx1 - sx0),
    ] = mask[sy0:sy1, sx0:sx1]

    return canvas


def infer_od_centered_crop_box_512(row, crop_factor=OD_CENTERED_CROP_FACTOR):
    cx = float(get_merged_value(row, "od_centroid_x_512"))
    cy = float(get_merged_value(row, "od_centroid_y_512"))
    od_area = float(get_merged_value(row, "od_area_512"))

    if not np.isfinite(cx) or not np.isfinite(cy) or not np.isfinite(od_area) or od_area <= 0:
        raise RuntimeError(
            "Invalid OD centroid/area for inferred crop box: "
            f"cx={cx}, cy={cy}, od_area={od_area}"
        )

    od_equivalent_diameter = 2.0 * np.sqrt(od_area / np.pi)
    crop_side = crop_factor * od_equivalent_diameter

    return {
        "x0": cx - crop_side / 2.0,
        "y0": cy - crop_side / 2.0,
        "x1": cx + crop_side / 2.0,
        "y1": cy + crop_side / 2.0,
        "crop_side_512": crop_side,
        "crop_factor": crop_factor,
    }


def save_binary_mask(mask, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    mask_u8 = (mask.astype(np.uint8) * 255)
    ok = cv2.imwrite(str(output_path), mask_u8)

    if not ok:
        raise RuntimeError(f"Could not save mask: {output_path}")

    return str(output_path)


def prepare_masks_for_input(
    joint_mask_path,
    input_image_path,
    branch_space,
    crop_box=None,
    use_local_candidate_for_input=False,
):
    input_rgb, resolved_input_path = read_rgb_image_robust(
        input_image_path,
        use_local_candidate=use_local_candidate_for_input,
    )

    od_mask, oc_mask, joint_unique_values, joint_decode_mode = read_joint_od_oc_mask(
        joint_mask_path
    )

    if branch_space in [
        "hybrid_od_centered_branch",
        "od_centered_crop_2p5x_single_model",
    ]:
        if crop_box is None:
            raise RuntimeError(f"Missing inferred crop box for branch_space={branch_space}")

        od_crop = crop_mask_with_box(od_mask, crop_box)
        oc_crop = crop_mask_with_box(oc_mask, crop_box)

        od_input = resize_binary_mask(od_crop, XAI_INPUT_SIZE)
        oc_input = resize_binary_mask(oc_crop, XAI_INPUT_SIZE)
        mask_source_mode = "joint_mask_512_inferred_od_centered_crop_2p5x"

    else:
        od_input = resize_binary_mask(od_mask, XAI_INPUT_SIZE)
        oc_input = resize_binary_mask(oc_mask, XAI_INPUT_SIZE)
        mask_source_mode = "joint_mask_512_resized_to_input"

    oc_input = (oc_input & od_input).astype(np.uint8)

    fov_input = resize_binary_mask(
        infer_fov_mask_from_rgb(input_rgb),
        XAI_INPUT_SIZE,
    )

    rim = (od_input & (1 - oc_input)).astype(np.uint8)
    outside_od = (fov_input & (1 - od_input)).astype(np.uint8)

    regions = {
        "OD": od_input,
        "OC": oc_input,
        "Rim": rim,
        "OD_plus_OC": od_input.copy(),
        "Outside_OD": outside_od,
        "FOV": fov_input,
    }

    return regions, resolved_input_path, mask_source_mode, joint_unique_values, joint_decode_mode


metadata_cols = [
    "sample_id",
    "patient_id",
    "gon_binary",
    "gon_label",
    "quality_score",
    "area_cdr_proxy",
    "mask_512_path",
    "od_centroid_x_512",
    "od_centroid_y_512",
    "od_area_512",
]

metadata_cols = [
    col for col in metadata_cols
    if col in analysis_manifest.columns
]

test_metadata = test_samples.merge(
    analysis_manifest[metadata_cols],
    on=["sample_id", "patient_id"],
    how="left",
    validate="one_to_one",
)

if test_metadata["mask_512_path"].isna().any():
    missing_mask = int(test_metadata["mask_512_path"].isna().sum())
    raise RuntimeError(f"Missing joint OD/OC mask paths in test metadata: {missing_mask}")

hybrid_test = hybrid_manifest.merge(
    test_metadata,
    on=["sample_id", "patient_id"],
    how="inner",
    validate="one_to_one",
)

split_columns = [
    col for col in hybrid_test.columns
    if col.startswith("glaucoma_split")
]

if len(split_columns) > 0:
    hybrid_test = hybrid_test[
        hybrid_test[split_columns].astype(str).eq("test").any(axis=1)
    ].copy()

od25_test = anatomy_manifest[
    anatomy_manifest["experiment_id"].astype(str).eq("od_centered_crop_2p5x")
].copy()

od25_test = od25_test.merge(
    test_metadata,
    on=["sample_id", "patient_id"],
    how="inner",
    validate="one_to_one",
)

outside_test = outside_manifest.merge(
    test_metadata,
    on=["sample_id", "patient_id"],
    how="inner",
    validate="one_to_one",
)

if len(hybrid_test) == 0:
    raise RuntimeError("Hybrid test manifest is empty.")

if len(od25_test) == 0:
    raise RuntimeError("OD-centered 2.5x test manifest is empty.")

if len(outside_test) == 0:
    raise RuntimeError("Outside-OD control test manifest is empty.")

od25_path_col = xai_runtime_config["detected_path_columns"]["od_centered_crop_2p5x"]
outside_path_col = xai_runtime_config["detected_path_columns"]["outside_od_only_trained_control"]

tasks = []

for _, row in hybrid_test.iterrows():
    sample_id = str(row["sample_id"])

    tasks.append(
        {
            "model_id": "hybrid__full_image_plus_od_centered_2p5x_effb0",
            "branch_space": "hybrid_full_image_branch",
            "sample_id": sample_id,
            "patient_id": str(row["patient_id"]),
            "y_true": get_binary_label_from_row(row),
            "input_image_path": str(row["full_image_path"]),
            "use_local_candidate_for_input": True,
            "joint_mask_path": str(row["mask_512_path"]),
            "crop_box": None,
        }
    )

    od25_match = od25_test[
        od25_test["sample_id"].astype(str).eq(sample_id)
    ]

    if len(od25_match) != 1:
        raise RuntimeError(f"Could not find unique OD-centered row for sample_id={sample_id}")

    od25_row = od25_match.iloc[0]
    crop_box = infer_od_centered_crop_box_512(od25_row)

    tasks.append(
        {
            "model_id": "hybrid__full_image_plus_od_centered_2p5x_effb0",
            "branch_space": "hybrid_od_centered_branch",
            "sample_id": sample_id,
            "patient_id": str(row["patient_id"]),
            "y_true": get_binary_label_from_row(row),
            "input_image_path": str(row["od_centered_2p5x_image_path"]),
            "use_local_candidate_for_input": False,
            "joint_mask_path": str(row["mask_512_path"]),
            "crop_box": crop_box,
        }
    )

for _, row in od25_test.iterrows():
    crop_box = infer_od_centered_crop_box_512(row)

    tasks.append(
        {
            "model_id": "anatomy__od_centered_crop_2p5x",
            "branch_space": "od_centered_crop_2p5x_single_model",
            "sample_id": str(row["sample_id"]),
            "patient_id": str(row["patient_id"]),
            "y_true": get_binary_label_from_row(row),
            "input_image_path": str(row[od25_path_col]),
            "use_local_candidate_for_input": False,
            "joint_mask_path": str(row["mask_512_path"]),
            "crop_box": crop_box,
        }
    )

for _, row in outside_test.iterrows():
    tasks.append(
        {
            "model_id": "anatomy__outside_od_only_trained_control",
            "branch_space": "outside_od_only_trained_control_single_model",
            "sample_id": str(row["sample_id"]),
            "patient_id": str(row["patient_id"]),
            "y_true": get_binary_label_from_row(row),
            "input_image_path": str(row[outside_path_col]),
            "use_local_candidate_for_input": False,
            "joint_mask_path": str(row["mask_512_path"]),
            "crop_box": None,
        }
    )

mask_manifest_rows = []
mask_audit_rows = []

for task in tqdm(tasks, desc="Building XAI anatomical masks"):
    try:
        regions, resolved_input_path, mask_source_mode, joint_unique_values, joint_decode_mode = prepare_masks_for_input(
            joint_mask_path=task["joint_mask_path"],
            input_image_path=task["input_image_path"],
            branch_space=task["branch_space"],
            crop_box=task["crop_box"],
            use_local_candidate_for_input=task["use_local_candidate_for_input"],
        )

        region_paths = {}

        for region_name, region_mask in regions.items():
            output_path = (
                XAI_MASK_ROOT
                / task["model_id"]
                / task["branch_space"]
                / f"{task['sample_id']}_{region_name}_mask_384.png"
            )

            region_paths[region_name] = save_binary_mask(
                region_mask,
                output_path,
            )

        crop_box = task["crop_box"]

        row = {
            "model_id": task["model_id"],
            "branch_space": task["branch_space"],
            "sample_id": task["sample_id"],
            "patient_id": task["patient_id"],
            "y_true": int(task["y_true"]),
            "input_image_path": task["input_image_path"],
            "resolved_input_path": resolved_input_path,
            "joint_mask_source_path": task["joint_mask_path"],
            "joint_mask_unique_values": str(joint_unique_values),
            "joint_mask_decode_mode": joint_decode_mode,
            "mask_source_mode": mask_source_mode,
            "crop_box_used": crop_box is not None,
            "crop_x0_512": crop_box["x0"] if crop_box is not None else np.nan,
            "crop_y0_512": crop_box["y0"] if crop_box is not None else np.nan,
            "crop_x1_512": crop_box["x1"] if crop_box is not None else np.nan,
            "crop_y1_512": crop_box["y1"] if crop_box is not None else np.nan,
            "crop_side_512": crop_box["crop_side_512"] if crop_box is not None else np.nan,
            "OD_mask_path": region_paths["OD"],
            "OC_mask_path": region_paths["OC"],
            "Rim_mask_path": region_paths["Rim"],
            "OD_plus_OC_mask_path": region_paths["OD_plus_OC"],
            "Outside_OD_mask_path": region_paths["Outside_OD"],
            "FOV_mask_path": region_paths["FOV"],
            "OD_pixels": int(regions["OD"].sum()),
            "OC_pixels": int(regions["OC"].sum()),
            "Rim_pixels": int(regions["Rim"].sum()),
            "OD_plus_OC_pixels": int(regions["OD_plus_OC"].sum()),
            "Outside_OD_pixels": int(regions["Outside_OD"].sum()),
            "FOV_pixels": int(regions["FOV"].sum()),
            "status": "pass",
            "error": "",
        }

        mask_manifest_rows.append(row)

    except Exception as exc:
        mask_audit_rows.append(
            {
                "model_id": task["model_id"],
                "branch_space": task["branch_space"],
                "sample_id": task["sample_id"],
                "patient_id": task["patient_id"],
                "input_image_path": task["input_image_path"],
                "status": "fail",
                "error": repr(exc),
            }
        )

mask_manifest = pd.DataFrame(mask_manifest_rows)
mask_audit = pd.DataFrame(mask_audit_rows)

if len(mask_audit) > 0:
    mask_audit.to_csv(XAI_MASK_AUDIT_CSV, index=False)
    print(mask_audit.head(20).to_string(index=False))
    raise RuntimeError(f"Mask construction failed for {len(mask_audit)} tasks.")

mask_audit = pd.DataFrame(
    [
        {
            "status": "pass",
            "n_tasks": len(tasks),
            "n_completed": len(mask_manifest),
            "n_failed": 0,
        }
    ]
)

mask_manifest.to_csv(XAI_MASK_MANIFEST_CSV, index=False)
mask_audit.to_csv(XAI_MASK_AUDIT_CSV, index=False)

mask_summary = (
    mask_manifest
    .groupby(["model_id", "branch_space", "mask_source_mode", "joint_mask_decode_mode", "crop_box_used"])
    .agg(
        n_images=("sample_id", "nunique"),
        n_patients=("patient_id", "nunique"),
        mean_OD_pixels=("OD_pixels", "mean"),
        mean_OC_pixels=("OC_pixels", "mean"),
        mean_Rim_pixels=("Rim_pixels", "mean"),
        mean_Outside_OD_pixels=("Outside_OD_pixels", "mean"),
        mean_FOV_pixels=("FOV_pixels", "mean"),
    )
    .reset_index()
)

mask_summary.to_csv(XAI_MASK_SUMMARY_CSV, index=False)

visual_cases = (
    mask_manifest
    .sort_values(["model_id", "branch_space", "sample_id"])
    .groupby(["model_id", "branch_space"])
    .head(1)
    .reset_index(drop=True)
)

visual_cases.to_csv(XAI_MASK_VISUAL_CASES_CSV, index=False)

n_rows = len(visual_cases)
fig, axes = plt.subplots(
    n_rows,
    4,
    figsize=(13, max(3.2, 3.0 * n_rows)),
)

if n_rows == 1:
    axes = np.expand_dims(axes, axis=0)

for row_idx, (_, row) in enumerate(visual_cases.iterrows()):
    image_rgb, _ = read_rgb_image_robust(
        row["resolved_input_path"],
        use_local_candidate=False,
    )

    image_rgb = cv2.resize(
        image_rgb,
        (XAI_INPUT_SIZE, XAI_INPUT_SIZE),
        interpolation=cv2.INTER_AREA,
    )

    od = read_binary_mask(row["OD_mask_path"])
    oc = read_binary_mask(row["OC_mask_path"])
    outside = read_binary_mask(row["Outside_OD_mask_path"])

    overlay = image_rgb.copy()

    overlay[od.astype(bool)] = (
        0.65 * overlay[od.astype(bool)] + 0.35 * np.array([255, 170, 0])
    ).astype(np.uint8)

    overlay[oc.astype(bool)] = (
        0.55 * overlay[oc.astype(bool)] + 0.45 * np.array([0, 180, 255])
    ).astype(np.uint8)

    axes[row_idx, 0].imshow(image_rgb)
    axes[row_idx, 0].set_title("Input")

    axes[row_idx, 1].imshow(overlay)
    axes[row_idx, 1].set_title("OD/OC overlay")

    axes[row_idx, 2].imshow(od, cmap="gray")
    axes[row_idx, 2].set_title("OD mask")

    axes[row_idx, 3].imshow(outside, cmap="gray")
    axes[row_idx, 3].set_title("Outside OD")

    ylabel = (
        f"{row['model_id']}\n"
        f"{row['branch_space']}\n"
        f"{row['sample_id']}"
    )

    axes[row_idx, 0].set_ylabel(ylabel, fontsize=8)

    for col_idx in range(4):
        axes[row_idx, col_idx].axis("off")

plt.suptitle(
    "Notebook 12 XAI anatomical mask audit for priority models",
    fontsize=13,
    fontweight="bold",
)

plt.tight_layout()
plt.savefig(XAI_MASK_VISUAL_PANEL_PNG, dpi=220, bbox_inches="tight")
plt.savefig(XAI_MASK_VISUAL_PANEL_PDF, bbox_inches="tight")
plt.show()

build_summary = {
    "notebook": "12_integrated_internal_model_audit_and_freezing",
    "cell": "05_build_branch_specific_anatomical_mask_manifest",
    "status": "PASS",
    "input_size": int(XAI_INPUT_SIZE),
    "n_tasks": int(len(tasks)),
    "n_completed": int(len(mask_manifest)),
    "n_failed": 0,
    "n_unique_models": int(mask_manifest["model_id"].nunique()),
    "n_unique_branch_spaces": int(mask_manifest["branch_space"].nunique()),
    "joint_mask_column": "mask_512_path",
    "od_centered_crop_factor": float(OD_CENTERED_CROP_FACTOR),
    "crop_alignment_method": "inferred_from_od_centroid_and_od_equivalent_diameter_in_512_space",
    "outputs": {
        "mask_manifest_csv": str(XAI_MASK_MANIFEST_CSV),
        "mask_summary_csv": str(XAI_MASK_SUMMARY_CSV),
        "mask_audit_csv": str(XAI_MASK_AUDIT_CSV),
        "visual_cases_csv": str(XAI_MASK_VISUAL_CASES_CSV),
        "visual_panel_png": str(XAI_MASK_VISUAL_PANEL_PNG),
        "visual_panel_pdf": str(XAI_MASK_VISUAL_PANEL_PDF),
    },
}

with open(XAI_MASK_BUILD_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(build_summary, f, indent=2)

print("XAI anatomical mask summary:")
print(mask_summary.to_string(index=False))

print("\nXAI anatomical mask audit:")
print(mask_audit.to_string(index=False))

print("\nVisual audit cases:")
print(
    visual_cases[
        [
            "model_id",
            "branch_space",
            "sample_id",
            "patient_id",
            "OD_pixels",
            "OC_pixels",
            "Rim_pixels",
            "Outside_OD_pixels",
            "joint_mask_decode_mode",
            "mask_source_mode",
        ]
    ].to_string(index=False)
)

print(f"\nMask manifest CSV: {XAI_MASK_MANIFEST_CSV}")
print(f"Mask summary CSV: {XAI_MASK_SUMMARY_CSV}")
print(f"Mask audit CSV: {XAI_MASK_AUDIT_CSV}")
print(f"Visual cases CSV: {XAI_MASK_VISUAL_CASES_CSV}")
print(f"Visual panel PNG: {XAI_MASK_VISUAL_PANEL_PNG}")
print(f"Visual panel PDF: {XAI_MASK_VISUAL_PANEL_PDF}")
print(f"Build summary JSON: {XAI_MASK_BUILD_SUMMARY_JSON}")

required_outputs = [
    XAI_MASK_MANIFEST_CSV,
    XAI_MASK_SUMMARY_CSV,
    XAI_MASK_AUDIT_CSV,
    XAI_MASK_VISUAL_CASES_CSV,
    XAI_MASK_VISUAL_PANEL_PNG,
    XAI_MASK_VISUAL_PANEL_PDF,
    XAI_MASK_BUILD_SUMMARY_JSON,
]

missing_outputs = [
    str(path)
    for path in required_outputs
    if not Path(path).exists()
]

if missing_outputs:
    raise RuntimeError(f"Missing Cell 5 outputs: {missing_outputs}")

expected_branch_spaces = {
    "hybrid_full_image_branch",
    "hybrid_od_centered_branch",
    "od_centered_crop_2p5x_single_model",
    "outside_od_only_trained_control_single_model",
}

observed_branch_spaces = set(mask_manifest["branch_space"].astype(str).unique())

if not expected_branch_spaces.issubset(observed_branch_spaces):
    raise RuntimeError(
        f"Missing expected branch spaces. Observed: {sorted(observed_branch_spaces)}"
    )

if (mask_manifest["OD_pixels"].astype(int) <= 0).any():
    empty_rows = mask_manifest[
        mask_manifest["OD_pixels"].astype(int) <= 0
    ][
        [
            "model_id",
            "branch_space",
            "sample_id",
            "OD_pixels",
            "input_image_path",
            "joint_mask_source_path",
            "mask_source_mode",
        ]
    ].head(20)

    print(empty_rows.to_string(index=False))
    raise RuntimeError("At least one generated OD mask is empty.")

if (mask_manifest["FOV_pixels"].astype(int) <= 0).any():
    raise RuntimeError("At least one generated FOV mask is empty.")

print("Status: PASS")

In [ ]:
# Cell 6: Reusable Multi-XAI utilities for priority models

import json
import time
import warnings
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
import timm

warnings.filterwarnings("ignore")

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

XAI_INPUT_SIZE = 384
RANDOM_CONTROLS_PER_REGION = 64

XAI_RUNTIME_ROOT = RESULTS_12_DIR / "xai_runtime_priority_models"
XAI_RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)

XAI_RUNTIME_CONFIG_JSON = (
    XAI_RUNTIME_ROOT
    / "notebook12_xai_runtime_config.json"
)

XAI_MASK_MANIFEST_CSV = (
    XAI_RUNTIME_ROOT
    / "notebook12_xai_priority_anatomical_mask_manifest.csv"
)

HYBRID_SELECTION_JSON = (
    RESULTS_11_DIR
    / "notebook11_validation_selected_hybrid_model.json"
)

HYBRID_PAIRED_INPUT_MANIFEST_CSV = (
    PROJECT_DATA_DIR
    / "hygd_hybrid_anatomy_aware_classifier_inputs"
    / "notebook11_hybrid_full_image_od25x_paired_input_manifest.csv"
)

LOCAL_HYGD_IMAGE_CACHE = Path("/content/HYGD_local/Images")

MULTIXAI_UTILITIES_CONFIG_JSON = (
    XAI_RUNTIME_ROOT
    / "notebook12_multixai_utilities_config.json"
)

MULTIXAI_TARGET_LAYER_AUDIT_CSV = (
    XAI_RUNTIME_ROOT
    / "notebook12_multixai_target_layer_audit.csv"
)

required_inputs = [
    XAI_RUNTIME_CONFIG_JSON,
    XAI_MASK_MANIFEST_CSV,
    HYBRID_SELECTION_JSON,
    HYBRID_PAIRED_INPUT_MANIFEST_CSV,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not Path(path).exists()
]

if missing_inputs:
    raise FileNotFoundError(f"Missing Cell 6 inputs: {missing_inputs}")

with open(XAI_RUNTIME_CONFIG_JSON, "r", encoding="utf-8") as f:
    xai_runtime_config = json.load(f)

with open(HYBRID_SELECTION_JSON, "r", encoding="utf-8") as f:
    hybrid_selection = json.load(f)

mask_manifest = pd.read_csv(XAI_MASK_MANIFEST_CSV)
hybrid_manifest = pd.read_csv(HYBRID_PAIRED_INPUT_MANIFEST_CSV)

for df in [mask_manifest, hybrid_manifest]:
    if "sample_id" in df.columns:
        df["sample_id"] = df["sample_id"].astype(str)
    if "patient_id" in df.columns:
        df["patient_id"] = df["patient_id"].astype(str)

priority_branch_spaces = [
    "hybrid_full_image_branch",
    "hybrid_od_centered_branch",
    "od_centered_crop_2p5x_single_model",
    "outside_od_only_trained_control_single_model",
]

REGIONS_FOR_DIRECT_ENERGY = [
    "OD",
    "OC",
    "Rim",
    "OD_plus_OC",
    "Outside_OD",
]

REGIONS_FOR_RANDOM_CONTROL = [
    "OD",
    "OC",
    "Rim",
    "OD_plus_OC",
]

REGION_MASK_COLUMNS = {
    "OD": "OD_mask_path",
    "OC": "OC_mask_path",
    "Rim": "Rim_mask_path",
    "OD_plus_OC": "OD_plus_OC_mask_path",
    "Outside_OD": "Outside_OD_mask_path",
    "FOV": "FOV_mask_path",
}

hybrid_checkpoint = Path(hybrid_selection["selected_best_checkpoint"])

od25_checkpoint = (
    MODELS_DIR
    / "hygd_anatomy_constrained_classifiers"
    / "training_runs"
    / "od_centered_crop_2p5x"
    / "od_centered_crop_2p5x_seed42"
    / "best_checkpoint.pt"
)

outside_od_checkpoint = (
    MODELS_DIR
    / "hygd_anatomy_constrained_classifiers"
    / "training_runs"
    / "outside_od_only_trained_control"
    / "outside_od_only_trained_control_seed42"
    / "best_checkpoint.pt"
)

checkpoint_registry = pd.DataFrame(
    [
        {
            "model_id": "hybrid__full_image_plus_od_centered_2p5x_effb0",
            "checkpoint_path": str(hybrid_checkpoint),
            "checkpoint_exists": hybrid_checkpoint.exists(),
        },
        {
            "model_id": "anatomy__od_centered_crop_2p5x",
            "checkpoint_path": str(od25_checkpoint),
            "checkpoint_exists": od25_checkpoint.exists(),
        },
        {
            "model_id": "anatomy__outside_od_only_trained_control",
            "checkpoint_path": str(outside_od_checkpoint),
            "checkpoint_exists": outside_od_checkpoint.exists(),
        },
    ]
)

if not checkpoint_registry["checkpoint_exists"].astype(bool).all():
    print(checkpoint_registry.to_string(index=False))
    missing_checkpoints = checkpoint_registry.loc[
        ~checkpoint_registry["checkpoint_exists"].astype(bool),
        "checkpoint_path",
    ].tolist()
    raise FileNotFoundError(f"Missing Multi-XAI checkpoints: {missing_checkpoints}")

hybrid_lookup = (
    hybrid_manifest
    .drop_duplicates(subset=["sample_id", "patient_id"])
    .set_index(["sample_id", "patient_id"])
)

xai_transform = transforms.Compose(
    [
        transforms.Resize((XAI_INPUT_SIZE, XAI_INPUT_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ]
)


def candidate_full_image_paths(image_path):
    image_path = str(image_path)

    candidates = [
        str(LOCAL_HYGD_IMAGE_CACHE / Path(image_path).name),
        image_path,
    ]

    unique = []
    seen = set()

    for candidate in candidates:
        if candidate not in seen:
            unique.append(candidate)
            seen.add(candidate)

    return unique


def read_rgb_image_robust(
    image_path,
    use_local_candidate=False,
    n_attempts=3,
    sleep_sec=0.20,
):
    if use_local_candidate:
        candidates = candidate_full_image_paths(image_path)
    else:
        candidates = [str(image_path)]

    last_error = ""

    for candidate in candidates:
        for _ in range(n_attempts):
            try:
                image_bgr = cv2.imread(str(candidate), cv2.IMREAD_COLOR)

                if image_bgr is not None:
                    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
                    return image_rgb, str(candidate)

                last_error = f"cv2.imread returned None for {candidate}"

            except Exception as exc:
                last_error = repr(exc)

            time.sleep(sleep_sec)

    raise FileNotFoundError(
        f"Could not read RGB image from candidates for {image_path}. "
        f"Last error: {last_error}"
    )


def read_binary_mask(mask_path):
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

    if mask is None:
        raise FileNotFoundError(f"Could not read mask: {mask_path}")

    return (mask > 0).astype(np.uint8)


def image_to_tensor(image_path, use_local_candidate=False):
    image_rgb, resolved_path = read_rgb_image_robust(
        image_path,
        use_local_candidate=use_local_candidate,
    )

    pil_image = Image.fromarray(image_rgb)
    tensor = xai_transform(pil_image).unsqueeze(0)

    return tensor, image_rgb, resolved_path


def save_heatmap_png(heatmap, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    heatmap_u8 = np.clip(heatmap * 255.0, 0, 255).astype(np.uint8)
    ok = cv2.imwrite(str(output_path), heatmap_u8)

    if not ok:
        raise RuntimeError(f"Could not save heatmap: {output_path}")

    return str(output_path)


def save_overlay_png(image_rgb, heatmap, output_path, alpha=0.40):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    image_rgb = cv2.resize(
        image_rgb,
        (XAI_INPUT_SIZE, XAI_INPUT_SIZE),
        interpolation=cv2.INTER_AREA,
    )

    heatmap_u8 = np.clip(heatmap * 255.0, 0, 255).astype(np.uint8)
    colored_bgr = cv2.applyColorMap(heatmap_u8, cv2.COLORMAP_JET)
    colored_rgb = cv2.cvtColor(colored_bgr, cv2.COLOR_BGR2RGB)

    overlay = (
        (1.0 - alpha) * image_rgb.astype(np.float32)
        + alpha * colored_rgb.astype(np.float32)
    )

    overlay = np.clip(overlay, 0, 255).astype(np.uint8)
    overlay_bgr = cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR)

    ok = cv2.imwrite(str(output_path), overlay_bgr)

    if not ok:
        raise RuntimeError(f"Could not save overlay: {output_path}")

    return str(output_path)


def normalize_heatmap(heatmap):
    heatmap = np.asarray(heatmap, dtype=np.float32)

    heatmap = np.nan_to_num(
        heatmap,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )

    heatmap_min = float(np.min(heatmap))
    heatmap_max = float(np.max(heatmap))

    if heatmap_max > heatmap_min:
        heatmap = (heatmap - heatmap_min) / (heatmap_max - heatmap_min)
    else:
        heatmap = np.zeros_like(heatmap, dtype=np.float32)

    return heatmap.astype(np.float32)


def strip_state_dict_prefixes(state_dict):
    cleaned = {}

    for key, value in state_dict.items():
        new_key = str(key)

        for prefix in ["module.", "model."]:
            if new_key.startswith(prefix):
                new_key = new_key[len(prefix):]

        cleaned[new_key] = value

    return cleaned


def extract_state_dict(checkpoint):
    if isinstance(checkpoint, dict):
        for key in ["model_state_dict", "state_dict", "model"]:
            if key in checkpoint and isinstance(checkpoint[key], dict):
                return strip_state_dict_prefixes(checkpoint[key])

    if isinstance(checkpoint, dict):
        return strip_state_dict_prefixes(checkpoint)

    raise RuntimeError("Unsupported checkpoint format.")


class HybridEfficientNetB0Classifier(nn.Module):
    def __init__(
        self,
        full_backbone_name="efficientnet_b0",
        anatomy_backbone_name="efficientnet_b0",
        pretrained=False,
        dropout=0.35,
        hidden_dim=512,
    ):
        super().__init__()

        self.full_branch = timm.create_model(
            full_backbone_name,
            pretrained=pretrained,
            num_classes=0,
            global_pool="avg",
        )

        self.anatomy_branch = timm.create_model(
            anatomy_backbone_name,
            pretrained=pretrained,
            num_classes=0,
            global_pool="avg",
        )

        full_dim = int(self.full_branch.num_features)
        anatomy_dim = int(self.anatomy_branch.num_features)
        fusion_dim = full_dim + anatomy_dim

        self.full_dim = full_dim
        self.anatomy_dim = anatomy_dim
        self.fusion_dim = fusion_dim

        self.fusion_head = nn.Sequential(
            nn.LayerNorm(fusion_dim),
            nn.Dropout(dropout),
            nn.Linear(fusion_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, full_image, anatomy_image, return_features=False):
        full_features = self.full_branch(full_image)
        anatomy_features = self.anatomy_branch(anatomy_image)

        fused_features = torch.cat(
            [
                full_features,
                anatomy_features,
            ],
            dim=1,
        )

        logits = self.fusion_head(fused_features).squeeze(1)

        if return_features:
            return {
                "logits": logits,
                "full_features": full_features,
                "anatomy_features": anatomy_features,
                "fused_features": fused_features,
            }

        return logits


def create_binary_efficientnet_b0():
    model = timm.create_model(
        "efficientnet_b0",
        pretrained=False,
        num_classes=1,
    )

    return model


def load_hybrid_model(checkpoint_path):
    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=False,
    )

    state_dict = extract_state_dict(checkpoint)

    model = HybridEfficientNetB0Classifier(pretrained=False)
    model.load_state_dict(state_dict, strict=False)
    model = model.to(DEVICE)
    model.eval()

    return model


def load_binary_model(checkpoint_path):
    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=False,
    )

    state_dict = extract_state_dict(checkpoint)

    model = create_binary_efficientnet_b0()
    model.load_state_dict(state_dict, strict=False)
    model = model.to(DEVICE)
    model.eval()

    return model


def find_last_conv2d_layer(model):
    last_name = None
    last_module = None

    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            last_name = name
            last_module = module

    if last_module is None:
        raise RuntimeError("No Conv2d layer found for CAM target.")

    return last_name, last_module


def get_hybrid_input_paths(sample_id, patient_id):
    key = (str(sample_id), str(patient_id))

    if key not in hybrid_lookup.index:
        raise KeyError(
            f"Missing hybrid lookup row for sample_id={sample_id}, "
            f"patient_id={patient_id}"
        )

    row = hybrid_lookup.loc[key]

    if isinstance(row, pd.DataFrame):
        row = row.iloc[0]

    return str(row["full_image_path"]), str(row["od_centered_2p5x_image_path"])


def load_region_masks(mask_row):
    masks = {}

    for region_name, column_name in REGION_MASK_COLUMNS.items():
        masks[region_name] = read_binary_mask(mask_row[column_name])

    return masks


def stable_rng_seed(*parts, modulo=2**32 - 1):
    text = "|".join([str(part) for part in parts])
    value = abs(hash(text)) % modulo

    return value


def random_disk_mask(target_area, fov_mask, rng):
    target_area = int(target_area)

    if target_area <= 0:
        return np.zeros_like(fov_mask, dtype=np.uint8)

    radius = max(1, int(round(np.sqrt(target_area / np.pi))))

    coords = np.argwhere(fov_mask > 0)

    if len(coords) == 0:
        return np.zeros_like(fov_mask, dtype=np.uint8)

    for _ in range(64):
        cy, cx = coords[rng.integers(0, len(coords))]

        mask = np.zeros_like(fov_mask, dtype=np.uint8)
        cv2.circle(mask, (int(cx), int(cy)), radius, 1, -1)
        mask = (mask & fov_mask).astype(np.uint8)

        if mask.sum() > 0:
            return mask

    return np.zeros_like(fov_mask, dtype=np.uint8)


def heatmap_energy_fraction(heatmap, region_mask, fov_mask):
    heatmap = np.asarray(heatmap, dtype=np.float32)
    region_mask = np.asarray(region_mask, dtype=np.uint8)
    fov_mask = np.asarray(fov_mask, dtype=np.uint8)

    denominator = float((heatmap * fov_mask).sum())

    if denominator <= 1e-12:
        return np.nan

    numerator = float((heatmap * region_mask).sum())

    return numerator / denominator


def compute_region_enrichment_metrics(
    heatmap,
    region_mask,
    fov_mask,
    model_id,
    branch_space,
    sample_id,
    region_name,
    method_name,
    n_random_controls=RANDOM_CONTROLS_PER_REGION,
):
    observed = heatmap_energy_fraction(
        heatmap=heatmap,
        region_mask=region_mask,
        fov_mask=fov_mask,
    )

    region_area = int(region_mask.sum())
    fov_area = int(fov_mask.sum())

    if region_name not in REGIONS_FOR_RANDOM_CONTROL or region_area <= 0 or fov_area <= 0:
        return {
            "observed_energy_fraction": observed,
            "random_energy_fraction_mean": np.nan,
            "random_energy_fraction_median": np.nan,
            "observed_minus_random_mean": np.nan,
            "observed_div_random_mean": np.nan,
            "empirical_p_random_ge_observed": np.nan,
            "n_random_controls": 0,
            "region_area_pixels": region_area,
            "fov_area_pixels": fov_area,
        }

    rng = np.random.default_rng(
        stable_rng_seed(
            method_name,
            model_id,
            branch_space,
            sample_id,
            region_name,
            SEED,
        )
    )

    random_values = []

    for _ in range(n_random_controls):
        random_mask = random_disk_mask(
            target_area=region_area,
            fov_mask=fov_mask,
            rng=rng,
        )

        value = heatmap_energy_fraction(
            heatmap=heatmap,
            region_mask=random_mask,
            fov_mask=fov_mask,
        )

        if np.isfinite(value):
            random_values.append(float(value))

    if len(random_values) == 0 or not np.isfinite(observed):
        random_mean = np.nan
        random_median = np.nan
        observed_minus_random = np.nan
        observed_div_random = np.nan
        empirical_p = np.nan
    else:
        random_values = np.asarray(random_values, dtype=float)
        random_mean = float(np.mean(random_values))
        random_median = float(np.median(random_values))
        observed_minus_random = float(observed - random_mean)
        observed_div_random = float(observed / random_mean) if random_mean > 1e-12 else np.nan
        empirical_p = float(
            (np.sum(random_values >= observed) + 1)
            / (len(random_values) + 1)
        )

    return {
        "observed_energy_fraction": observed,
        "random_energy_fraction_mean": random_mean,
        "random_energy_fraction_median": random_median,
        "observed_minus_random_mean": observed_minus_random,
        "observed_div_random_mean": observed_div_random,
        "empirical_p_random_ge_observed": empirical_p,
        "n_random_controls": int(len(random_values)),
        "region_area_pixels": region_area,
        "fov_area_pixels": fov_area,
    }


class GradCAMPlusPlus:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None

        self.forward_handle = self.target_layer.register_forward_hook(
            self._forward_hook
        )

        self.backward_handle = self.target_layer.register_full_backward_hook(
            self._backward_hook
        )

    def _forward_hook(self, module, inputs, output):
        self.activations = output

    def _backward_hook(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def remove_hooks(self):
        self.forward_handle.remove()
        self.backward_handle.remove()

    def generate(self, forward_function):
        self.model.zero_grad(set_to_none=True)
        self.activations = None
        self.gradients = None

        logits = forward_function()

        if logits.ndim > 0:
            score = logits.reshape(-1)[0]
        else:
            score = logits

        score.backward()

        if self.activations is None or self.gradients is None:
            raise RuntimeError(
                "Grad-CAM++ hooks did not capture activations/gradients."
            )

        activations = self.activations.detach()
        gradients = self.gradients.detach()

        gradients_power_2 = gradients ** 2
        gradients_power_3 = gradients_power_2 * gradients

        sum_activations_gradients_power_3 = torch.sum(
            activations * gradients_power_3,
            dim=(2, 3),
            keepdim=True,
        )

        eps = 1e-8

        alphas = gradients_power_2 / (
            2.0 * gradients_power_2
            + sum_activations_gradients_power_3
            + eps
        )

        positive_gradients = torch.relu(gradients)

        weights = torch.sum(
            alphas * positive_gradients,
            dim=(2, 3),
            keepdim=True,
        )

        cam = torch.sum(weights * activations, dim=1, keepdim=True)
        cam = torch.relu(cam)

        cam = F.interpolate(
            cam,
            size=(XAI_INPUT_SIZE, XAI_INPUT_SIZE),
            mode="bilinear",
            align_corners=False,
        )

        heatmap = cam.detach().cpu().numpy()[0, 0]
        heatmap = normalize_heatmap(heatmap)

        probability = float(
            torch.sigmoid(logits.detach()).cpu().numpy().reshape(-1)[0]
        )

        logit = float(
            logits.detach().cpu().numpy().reshape(-1)[0]
        )

        return heatmap, logit, probability


def integrated_gradients_single_input(
    model_forward_function,
    input_tensor,
    baseline_tensor=None,
    steps=32,
):
    if baseline_tensor is None:
        baseline_tensor = torch.zeros_like(input_tensor)

    scaled_inputs = [
        baseline_tensor + (float(step) / steps) * (input_tensor - baseline_tensor)
        for step in range(1, steps + 1)
    ]

    gradients = []

    for scaled_input in scaled_inputs:
        scaled_input = scaled_input.detach().clone().requires_grad_(True)

        output = model_forward_function(scaled_input)

        if output.ndim > 0:
            score = output.reshape(-1)[0]
        else:
            score = output

        model_output = output.detach()

        score.backward()

        gradients.append(scaled_input.grad.detach().clone())

        model_forward_function.zero_grad_if_available()

    avg_gradients = torch.mean(torch.stack(gradients, dim=0), dim=0)
    integrated_gradients = (input_tensor - baseline_tensor) * avg_gradients

    attribution = integrated_gradients.detach().abs().sum(dim=1)[0].cpu().numpy()
    attribution = normalize_heatmap(attribution)

    probability = float(
        torch.sigmoid(model_output).cpu().numpy().reshape(-1)[0]
    )

    logit = float(
        model_output.cpu().numpy().reshape(-1)[0]
    )

    return attribution, logit, probability


def build_region_metric_rows(
    heatmap,
    mask_row,
    model_id,
    branch_space,
    method_name,
    sample_id,
    patient_id,
    y_true,
    logit,
    probability,
    heatmap_path,
    overlay_path,
    resolved_input_path,
):
    masks = load_region_masks(mask_row)

    metric_rows = []

    for region_name in REGIONS_FOR_DIRECT_ENERGY:
        region_metrics = compute_region_enrichment_metrics(
            heatmap=heatmap,
            region_mask=masks[region_name],
            fov_mask=masks["FOV"],
            model_id=model_id,
            branch_space=branch_space,
            sample_id=sample_id,
            region_name=region_name,
            method_name=method_name,
            n_random_controls=RANDOM_CONTROLS_PER_REGION,
        )

        metric_rows.append(
            {
                "model_id": model_id,
                "branch_space": branch_space,
                "method_name": method_name,
                "sample_id": sample_id,
                "patient_id": patient_id,
                "y_true": int(y_true),
                "region_name": region_name,
                "logit": float(logit),
                "probability": float(probability),
                "heatmap_path": str(heatmap_path),
                "overlay_path": str(overlay_path),
                "resolved_input_path": str(resolved_input_path),
                **region_metrics,
            }
        )

    return metric_rows


target_layer_rows = []

hybrid_model_for_audit = load_hybrid_model(hybrid_checkpoint)
od25_model_for_audit = load_binary_model(od25_checkpoint)
outside_model_for_audit = load_binary_model(outside_od_checkpoint)

hybrid_full_layer_name, _ = find_last_conv2d_layer(
    hybrid_model_for_audit.full_branch
)

hybrid_od_layer_name, _ = find_last_conv2d_layer(
    hybrid_model_for_audit.anatomy_branch
)

od25_layer_name, _ = find_last_conv2d_layer(od25_model_for_audit)
outside_layer_name, _ = find_last_conv2d_layer(outside_model_for_audit)

target_layer_rows.extend(
    [
        {
            "model_id": "hybrid__full_image_plus_od_centered_2p5x_effb0",
            "branch_space": "hybrid_full_image_branch",
            "target_layer_name": f"full_branch.{hybrid_full_layer_name}",
            "target_layer_type": "Conv2d",
        },
        {
            "model_id": "hybrid__full_image_plus_od_centered_2p5x_effb0",
            "branch_space": "hybrid_od_centered_branch",
            "target_layer_name": f"anatomy_branch.{hybrid_od_layer_name}",
            "target_layer_type": "Conv2d",
        },
        {
            "model_id": "anatomy__od_centered_crop_2p5x",
            "branch_space": "od_centered_crop_2p5x_single_model",
            "target_layer_name": od25_layer_name,
            "target_layer_type": "Conv2d",
        },
        {
            "model_id": "anatomy__outside_od_only_trained_control",
            "branch_space": "outside_od_only_trained_control_single_model",
            "target_layer_name": outside_layer_name,
            "target_layer_type": "Conv2d",
        },
    ]
)

target_layer_audit = pd.DataFrame(target_layer_rows)
target_layer_audit.to_csv(MULTIXAI_TARGET_LAYER_AUDIT_CSV, index=False)

del hybrid_model_for_audit, od25_model_for_audit, outside_model_for_audit

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

utilities_config = {
    "notebook": "12_integrated_internal_model_audit_and_freezing",
    "cell": "06_reusable_multixai_utilities",
    "status": "PASS",
    "device": str(DEVICE),
    "input_size": int(XAI_INPUT_SIZE),
    "random_controls_per_region": int(RANDOM_CONTROLS_PER_REGION),
    "regions_for_direct_energy": REGIONS_FOR_DIRECT_ENERGY,
    "regions_for_random_control": REGIONS_FOR_RANDOM_CONTROL,
    "priority_branch_spaces": priority_branch_spaces,
    "checkpoint_registry": checkpoint_registry.to_dict(orient="records"),
    "target_layer_audit_csv": str(MULTIXAI_TARGET_LAYER_AUDIT_CSV),
    "next_cell": "Grad-CAM++ execution for priority models and branches.",
}

with open(MULTIXAI_UTILITIES_CONFIG_JSON, "w", encoding="utf-8") as f:
    json.dump(utilities_config, f, indent=2)

print("Multi-XAI utilities loaded")
print(f"Device: {DEVICE}")
print(f"Input size: {XAI_INPUT_SIZE}")
print(f"Random controls per region: {RANDOM_CONTROLS_PER_REGION}")

print("\nCheckpoint registry:")
print(checkpoint_registry.to_string(index=False))

print("\nTarget layer audit:")
print(target_layer_audit.to_string(index=False))

print("\nPriority branch spaces:")
for branch_space in priority_branch_spaces:
    n_rows = int((mask_manifest["branch_space"].astype(str) == branch_space).sum())
    print(f"{branch_space}: {n_rows} rows")

print(f"\nTarget layer audit CSV: {MULTIXAI_TARGET_LAYER_AUDIT_CSV}")
print(f"Utilities config JSON: {MULTIXAI_UTILITIES_CONFIG_JSON}")

required_outputs = [
    MULTIXAI_TARGET_LAYER_AUDIT_CSV,
    MULTIXAI_UTILITIES_CONFIG_JSON,
]

missing_outputs = [
    str(path)
    for path in required_outputs
    if not Path(path).exists()
]

if missing_outputs:
    raise RuntimeError(f"Missing Cell 6 outputs: {missing_outputs}")

if len(target_layer_audit) != 4:
    raise RuntimeError("Unexpected number of target-layer audit rows.")

if not set(priority_branch_spaces).issubset(
    set(mask_manifest["branch_space"].astype(str).unique())
):
    raise RuntimeError("Mask manifest is missing one or more priority branch spaces.")

print("Status: PASS")

In [ ]:
# Cell 7: Grad-CAM++ anatomical enrichment for priority Multi-XAI targets

import json
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

REQUIRE_GPU_FOR_FULL_XAI = True

if REQUIRE_GPU_FOR_FULL_XAI and DEVICE.type != "cuda":
    raise RuntimeError(
        "Grad-CAM++ full priority XAI should not be executed on CPU. "
        "Switch Colab runtime to GPU, rerun Cell 1 and Cell 6, then rerun this Cell 7."
    )

GRADCAM_ROOT = XAI_RUNTIME_ROOT / "gradcamplusplus_priority_models"
GRADCAM_HEATMAP_ROOT = GRADCAM_ROOT / "heatmaps"
GRADCAM_OVERLAY_ROOT = GRADCAM_ROOT / "overlays"

GRADCAM_ROOT.mkdir(parents=True, exist_ok=True)
GRADCAM_HEATMAP_ROOT.mkdir(parents=True, exist_ok=True)
GRADCAM_OVERLAY_ROOT.mkdir(parents=True, exist_ok=True)

GRADCAM_METRICS_CSV = (
    GRADCAM_ROOT
    / "notebook12_gradcamplusplus_priority_anatomical_metrics.csv"
)

GRADCAM_IMAGE_SUMMARY_CSV = (
    GRADCAM_ROOT
    / "notebook12_gradcamplusplus_priority_image_summary.csv"
)

GRADCAM_PATIENT_METRICS_CSV = (
    GRADCAM_ROOT
    / "notebook12_gradcamplusplus_priority_patient_level_metrics.csv"
)

GRADCAM_PATIENT_STATS_CSV = (
    GRADCAM_ROOT
    / "notebook12_gradcamplusplus_priority_patient_level_statistics.csv"
)

GRADCAM_REGION_SUMMARY_CSV = (
    GRADCAM_ROOT
    / "notebook12_gradcamplusplus_priority_region_summary.csv"
)

GRADCAM_HEATMAP_MANIFEST_CSV = (
    GRADCAM_ROOT
    / "notebook12_gradcamplusplus_priority_heatmap_manifest.csv"
)

GRADCAM_FAILURE_LOG_CSV = (
    GRADCAM_ROOT
    / "notebook12_gradcamplusplus_priority_failure_log.csv"
)

GRADCAM_VISUAL_CASES_CSV = (
    GRADCAM_ROOT
    / "notebook12_gradcamplusplus_priority_visual_cases.csv"
)

GRADCAM_VISUAL_PANEL_PNG = (
    GRADCAM_ROOT
    / "notebook12_gradcamplusplus_priority_visual_panel.png"
)

GRADCAM_VISUAL_PANEL_PDF = (
    GRADCAM_ROOT
    / "notebook12_gradcamplusplus_priority_visual_panel.pdf"
)

GRADCAM_SUMMARY_JSON = (
    GRADCAM_ROOT
    / "notebook12_gradcamplusplus_priority_summary.json"
)

SAVE_ALL_HEATMAPS = True

FAILURE_LOG_COLUMNS = [
    "model_id",
    "branch_space",
    "sample_id",
    "patient_id",
    "status",
    "error",
]

priority_xai_manifest = mask_manifest[
    mask_manifest["branch_space"].astype(str).isin(priority_branch_spaces)
].copy()

priority_xai_manifest = priority_xai_manifest.sort_values(
    [
        "model_id",
        "branch_space",
        "sample_id",
    ]
).reset_index(drop=True)

if len(priority_xai_manifest) == 0:
    raise RuntimeError("Priority XAI manifest is empty.")

expected_rows = 95 * 4

if len(priority_xai_manifest) != expected_rows:
    raise RuntimeError(
        f"Unexpected number of priority XAI rows: {len(priority_xai_manifest)} != {expected_rows}"
    )


def run_gradcam_for_mask_row(
    row,
    hybrid_model,
    od25_model,
    outside_model,
    hybrid_full_cam,
    hybrid_od_cam,
    od25_cam,
    outside_cam,
):
    model_id = str(row["model_id"])
    branch_space = str(row["branch_space"])
    sample_id = str(row["sample_id"])
    patient_id = str(row["patient_id"])
    y_true = int(row["y_true"])

    if model_id == "hybrid__full_image_plus_od_centered_2p5x_effb0":
        full_path, od_path = get_hybrid_input_paths(
            sample_id=sample_id,
            patient_id=patient_id,
        )

        full_tensor, full_rgb, full_resolved_path = image_to_tensor(
            full_path,
            use_local_candidate=True,
        )

        od_tensor, od_rgb, od_resolved_path = image_to_tensor(
            od_path,
            use_local_candidate=False,
        )

        full_tensor = full_tensor.to(DEVICE)
        od_tensor = od_tensor.to(DEVICE)

        if branch_space == "hybrid_full_image_branch":
            cam_runner = hybrid_full_cam
            target_rgb = full_rgb
            resolved_input_path = full_resolved_path

        elif branch_space == "hybrid_od_centered_branch":
            cam_runner = hybrid_od_cam
            target_rgb = od_rgb
            resolved_input_path = od_resolved_path

        else:
            raise ValueError(f"Unexpected hybrid branch_space: {branch_space}")

        heatmap, logit, probability = cam_runner.generate(
            forward_function=lambda: hybrid_model(full_tensor, od_tensor)
        )

    elif model_id == "anatomy__od_centered_crop_2p5x":
        tensor, image_rgb, resolved_input_path = image_to_tensor(
            row["resolved_input_path"],
            use_local_candidate=False,
        )

        tensor = tensor.to(DEVICE)

        heatmap, logit, probability = od25_cam.generate(
            forward_function=lambda: od25_model(tensor).squeeze(1)
        )

        target_rgb = image_rgb

    elif model_id == "anatomy__outside_od_only_trained_control":
        tensor, image_rgb, resolved_input_path = image_to_tensor(
            row["resolved_input_path"],
            use_local_candidate=False,
        )

        tensor = tensor.to(DEVICE)

        heatmap, logit, probability = outside_cam.generate(
            forward_function=lambda: outside_model(tensor).squeeze(1)
        )

        target_rgb = image_rgb

    else:
        raise ValueError(f"Unknown model_id: {model_id}")

    heatmap_path = (
        GRADCAM_HEATMAP_ROOT
        / model_id
        / branch_space
        / f"{sample_id}_gradcamplusplus_heatmap.png"
    )

    overlay_path = (
        GRADCAM_OVERLAY_ROOT
        / model_id
        / branch_space
        / f"{sample_id}_gradcamplusplus_overlay.png"
    )

    if SAVE_ALL_HEATMAPS:
        heatmap_path = save_heatmap_png(
            heatmap,
            heatmap_path,
        )

        overlay_path = save_overlay_png(
            target_rgb,
            heatmap,
            overlay_path,
        )
    else:
        heatmap_path = ""
        overlay_path = ""

    metric_rows = build_region_metric_rows(
        heatmap=heatmap,
        mask_row=row,
        model_id=model_id,
        branch_space=branch_space,
        method_name="GradCAMPlusPlus",
        sample_id=sample_id,
        patient_id=patient_id,
        y_true=y_true,
        logit=logit,
        probability=probability,
        heatmap_path=heatmap_path,
        overlay_path=overlay_path,
        resolved_input_path=resolved_input_path,
    )

    heatmap_manifest_row = {
        "model_id": model_id,
        "branch_space": branch_space,
        "method_name": "GradCAMPlusPlus",
        "sample_id": sample_id,
        "patient_id": patient_id,
        "y_true": y_true,
        "logit": float(logit),
        "probability": float(probability),
        "heatmap_path": str(heatmap_path),
        "overlay_path": str(overlay_path),
        "resolved_input_path": str(resolved_input_path),
        "status": "pass",
        "error": "",
    }

    return metric_rows, heatmap_manifest_row


def build_patient_level_statistics(patient_metrics_df):
    try:
        from scipy.stats import wilcoxon
        scipy_available = True

    except Exception:
        scipy_available = False

    stat_rows = []

    grouped = patient_metrics_df.groupby(
        [
            "model_id",
            "branch_space",
            "method_name",
            "region_name",
        ]
    )

    for (model_id, branch_space, method_name, region_name), block in grouped:
        endpoint = (
            block["observed_minus_random_mean"]
            .astype(float)
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
        )

        values = endpoint.values.astype(float)

        if len(values) >= 5:
            mean_value = float(np.mean(values))
            median_value = float(np.median(values))
            std_value = float(np.std(values, ddof=1)) if len(values) > 1 else np.nan
            ci95_low = float(np.percentile(values, 2.5))
            ci95_high = float(np.percentile(values, 97.5))
        else:
            mean_value = np.nan
            median_value = np.nan
            std_value = np.nan
            ci95_low = np.nan
            ci95_high = np.nan

        if scipy_available and len(values) >= 5 and not np.allclose(values, 0):
            p_two_sided = float(wilcoxon(values, alternative="two-sided").pvalue)
            p_greater = float(wilcoxon(values, alternative="greater").pvalue)
            p_less = float(wilcoxon(values, alternative="less").pvalue)
        else:
            p_two_sided = np.nan
            p_greater = np.nan
            p_less = np.nan

        positive = int(np.sum(values > 0))
        negative = int(np.sum(values < 0))
        denominator = positive + negative

        if denominator > 0:
            signed_direction_effect_proxy = float((positive - negative) / denominator)
        else:
            signed_direction_effect_proxy = np.nan

        stat_rows.append(
            {
                "model_id": model_id,
                "branch_space": branch_space,
                "method_name": method_name,
                "region_name": region_name,
                "endpoint_name": "observed_minus_random_mean",
                "n_patients": int(len(values)),
                "mean": mean_value,
                "median": median_value,
                "std": std_value,
                "ci95_low_percentile": ci95_low,
                "ci95_high_percentile": ci95_high,
                "wilcoxon_p_two_sided": p_two_sided,
                "wilcoxon_p_greater": p_greater,
                "wilcoxon_p_less": p_less,
                "signed_direction_effect_proxy": signed_direction_effect_proxy,
            }
        )

    return pd.DataFrame(stat_rows)


print("Grad-CAM++ priority Multi-XAI started")
print(f"Device: {DEVICE}")
print(f"Rows to process: {len(priority_xai_manifest)}")
print(f"Random controls per region: {RANDOM_CONTROLS_PER_REGION}")
print(f"Save heatmaps: {SAVE_ALL_HEATMAPS}")

start_time = time.time()

hybrid_model = load_hybrid_model(hybrid_checkpoint)
od25_model = load_binary_model(od25_checkpoint)
outside_model = load_binary_model(outside_od_checkpoint)

_, hybrid_full_layer = find_last_conv2d_layer(hybrid_model.full_branch)
_, hybrid_od_layer = find_last_conv2d_layer(hybrid_model.anatomy_branch)
_, od25_layer = find_last_conv2d_layer(od25_model)
_, outside_layer = find_last_conv2d_layer(outside_model)

hybrid_full_cam = GradCAMPlusPlus(
    model=hybrid_model,
    target_layer=hybrid_full_layer,
)

hybrid_od_cam = GradCAMPlusPlus(
    model=hybrid_model,
    target_layer=hybrid_od_layer,
)

od25_cam = GradCAMPlusPlus(
    model=od25_model,
    target_layer=od25_layer,
)

outside_cam = GradCAMPlusPlus(
    model=outside_model,
    target_layer=outside_layer,
)

all_metric_rows = []
heatmap_manifest_rows = []
failure_rows = []

for _, row in tqdm(
    priority_xai_manifest.iterrows(),
    total=len(priority_xai_manifest),
    desc="Grad-CAM++ priority XAI",
):
    try:
        metric_rows, heatmap_manifest_row = run_gradcam_for_mask_row(
            row=row,
            hybrid_model=hybrid_model,
            od25_model=od25_model,
            outside_model=outside_model,
            hybrid_full_cam=hybrid_full_cam,
            hybrid_od_cam=hybrid_od_cam,
            od25_cam=od25_cam,
            outside_cam=outside_cam,
        )

        all_metric_rows.extend(metric_rows)
        heatmap_manifest_rows.append(heatmap_manifest_row)

    except Exception as exc:
        failure_row = {
            "model_id": str(row.get("model_id", "")),
            "branch_space": str(row.get("branch_space", "")),
            "sample_id": str(row.get("sample_id", "")),
            "patient_id": str(row.get("patient_id", "")),
            "status": "fail",
            "error": repr(exc),
        }

        failure_rows.append(failure_row)

        heatmap_manifest_rows.append(
            {
                "model_id": str(row.get("model_id", "")),
                "branch_space": str(row.get("branch_space", "")),
                "method_name": "GradCAMPlusPlus",
                "sample_id": str(row.get("sample_id", "")),
                "patient_id": str(row.get("patient_id", "")),
                "y_true": int(row.get("y_true", -1)),
                "logit": np.nan,
                "probability": np.nan,
                "heatmap_path": "",
                "overlay_path": "",
                "resolved_input_path": "",
                "status": "fail",
                "error": repr(exc),
            }
        )

hybrid_full_cam.remove_hooks()
hybrid_od_cam.remove_hooks()
od25_cam.remove_hooks()
outside_cam.remove_hooks()

del hybrid_model, od25_model, outside_model

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

metrics_df = pd.DataFrame(all_metric_rows)
heatmap_manifest_df = pd.DataFrame(heatmap_manifest_rows)
failure_df = pd.DataFrame(failure_rows, columns=FAILURE_LOG_COLUMNS)

metrics_df.to_csv(GRADCAM_METRICS_CSV, index=False)
heatmap_manifest_df.to_csv(GRADCAM_HEATMAP_MANIFEST_CSV, index=False)
failure_df.to_csv(GRADCAM_FAILURE_LOG_CSV, index=False)

if len(failure_df) > 0:
    print(failure_df.head(20).to_string(index=False))
    raise RuntimeError(f"Grad-CAM++ failed for {len(failure_df)} rows.")

image_summary = (
    metrics_df
    .pivot_table(
        index=[
            "model_id",
            "branch_space",
            "method_name",
            "sample_id",
            "patient_id",
            "y_true",
            "probability",
        ],
        columns="region_name",
        values="observed_energy_fraction",
        aggfunc="first",
    )
    .reset_index()
)

image_summary.columns.name = None
image_summary.to_csv(GRADCAM_IMAGE_SUMMARY_CSV, index=False)

patient_metrics = (
    metrics_df
    .groupby(
        [
            "model_id",
            "branch_space",
            "method_name",
            "patient_id",
            "region_name",
        ]
    )
    .agg(
        n_images=("sample_id", "nunique"),
        y_true=("y_true", "max"),
        probability_mean=("probability", "mean"),
        observed_energy_fraction_mean=("observed_energy_fraction", "mean"),
        random_energy_fraction_mean=("random_energy_fraction_mean", "mean"),
        observed_minus_random_mean=("observed_minus_random_mean", "mean"),
        observed_div_random_mean=("observed_div_random_mean", "median"),
        empirical_p_random_ge_observed_mean=("empirical_p_random_ge_observed", "mean"),
        region_area_pixels_mean=("region_area_pixels", "mean"),
        fov_area_pixels_mean=("fov_area_pixels", "mean"),
    )
    .reset_index()
)

patient_metrics.to_csv(GRADCAM_PATIENT_METRICS_CSV, index=False)

patient_stats = build_patient_level_statistics(patient_metrics)
patient_stats.to_csv(GRADCAM_PATIENT_STATS_CSV, index=False)

region_summary = (
    metrics_df
    .groupby(
        [
            "model_id",
            "branch_space",
            "method_name",
            "region_name",
        ]
    )
    .agg(
        n_rows=("sample_id", "count"),
        n_images=("sample_id", "nunique"),
        n_patients=("patient_id", "nunique"),
        observed_energy_fraction_mean=("observed_energy_fraction", "mean"),
        observed_energy_fraction_median=("observed_energy_fraction", "median"),
        random_energy_fraction_mean_mean=("random_energy_fraction_mean", "mean"),
        observed_minus_random_mean_mean=("observed_minus_random_mean", "mean"),
        observed_div_random_mean_median=("observed_div_random_mean", "median"),
        empirical_p_random_ge_observed_mean=("empirical_p_random_ge_observed", "mean"),
        region_area_pixels_mean=("region_area_pixels", "mean"),
        fov_area_pixels_mean=("fov_area_pixels", "mean"),
    )
    .reset_index()
)

region_summary.to_csv(GRADCAM_REGION_SUMMARY_CSV, index=False)

visual_cases = (
    heatmap_manifest_df
    .sort_values(
        [
            "model_id",
            "branch_space",
            "sample_id",
        ]
    )
    .groupby(["model_id", "branch_space"])
    .head(1)
    .reset_index(drop=True)
)

visual_cases.to_csv(GRADCAM_VISUAL_CASES_CSV, index=False)

n_rows = len(visual_cases)

fig, axes = plt.subplots(
    n_rows,
    4,
    figsize=(14, max(3.2, 3.0 * n_rows)),
)

if n_rows == 1:
    axes = np.expand_dims(axes, axis=0)

for row_idx, (_, row) in enumerate(visual_cases.iterrows()):
    image_rgb, _ = read_rgb_image_robust(
        row["resolved_input_path"],
        use_local_candidate=False,
    )

    image_rgb = cv2.resize(
        image_rgb,
        (XAI_INPUT_SIZE, XAI_INPUT_SIZE),
        interpolation=cv2.INTER_AREA,
    )

    heatmap_gray = cv2.imread(
        str(row["heatmap_path"]),
        cv2.IMREAD_GRAYSCALE,
    )

    if heatmap_gray is None:
        heatmap_gray = np.zeros(
            (XAI_INPUT_SIZE, XAI_INPUT_SIZE),
            dtype=np.uint8,
        )

    heatmap_float = heatmap_gray.astype(np.float32) / 255.0
    heatmap_u8 = np.clip(heatmap_float * 255.0, 0, 255).astype(np.uint8)

    heatmap_color = cv2.cvtColor(
        cv2.applyColorMap(heatmap_u8, cv2.COLORMAP_JET),
        cv2.COLOR_BGR2RGB,
    )

    heatmap_overlay = (
        0.60 * image_rgb.astype(np.float32)
        + 0.40 * heatmap_color.astype(np.float32)
    )
    heatmap_overlay = np.clip(heatmap_overlay, 0, 255).astype(np.uint8)

    mask_row = priority_xai_manifest[
        (priority_xai_manifest["model_id"].astype(str) == str(row["model_id"]))
        & (priority_xai_manifest["branch_space"].astype(str) == str(row["branch_space"]))
        & (priority_xai_manifest["sample_id"].astype(str) == str(row["sample_id"]))
    ].iloc[0]

    od = read_binary_mask(mask_row["OD_mask_path"])
    oc = read_binary_mask(mask_row["OC_mask_path"])

    mask_overlay = image_rgb.copy()

    mask_overlay[od.astype(bool)] = (
        0.65 * mask_overlay[od.astype(bool)]
        + 0.35 * np.array([255, 170, 0])
    ).astype(np.uint8)

    mask_overlay[oc.astype(bool)] = (
        0.55 * mask_overlay[oc.astype(bool)]
        + 0.45 * np.array([0, 180, 255])
    ).astype(np.uint8)

    axes[row_idx, 0].imshow(image_rgb)
    axes[row_idx, 0].set_title("Input")

    axes[row_idx, 1].imshow(heatmap_color)
    axes[row_idx, 1].set_title("Grad-CAM++")

    axes[row_idx, 2].imshow(heatmap_overlay)
    axes[row_idx, 2].set_title("Heatmap overlay")

    axes[row_idx, 3].imshow(mask_overlay)
    axes[row_idx, 3].set_title("OD/OC mask")

    ylabel = (
        f"{row['model_id']}\n"
        f"{row['branch_space']}\n"
        f"{row['sample_id']}"
    )

    axes[row_idx, 0].set_ylabel(ylabel, fontsize=8)

    for col_idx in range(4):
        axes[row_idx, col_idx].axis("off")

plt.suptitle(
    "Notebook 12 Grad-CAM++ visual audit for priority Multi-XAI targets",
    fontsize=13,
    fontweight="bold",
)

plt.tight_layout()
plt.savefig(GRADCAM_VISUAL_PANEL_PNG, dpi=220, bbox_inches="tight")
plt.savefig(GRADCAM_VISUAL_PANEL_PDF, bbox_inches="tight")
plt.show()

elapsed_min = (time.time() - start_time) / 60.0

summary_payload = {
    "notebook": "12_integrated_internal_model_audit_and_freezing",
    "cell": "07_gradcamplusplus_priority_multixai",
    "status": "PASS",
    "method_name": "GradCAMPlusPlus",
    "device": str(DEVICE),
    "input_size": int(XAI_INPUT_SIZE),
    "random_controls_per_region": int(RANDOM_CONTROLS_PER_REGION),
    "n_priority_rows": int(len(priority_xai_manifest)),
    "n_metric_rows": int(len(metrics_df)),
    "n_heatmaps": int(len(heatmap_manifest_df)),
    "n_failures": int(len(failure_df)),
    "elapsed_min": float(elapsed_min),
    "outputs": {
        "metrics_csv": str(GRADCAM_METRICS_CSV),
        "image_summary_csv": str(GRADCAM_IMAGE_SUMMARY_CSV),
        "patient_metrics_csv": str(GRADCAM_PATIENT_METRICS_CSV),
        "patient_stats_csv": str(GRADCAM_PATIENT_STATS_CSV),
        "region_summary_csv": str(GRADCAM_REGION_SUMMARY_CSV),
        "heatmap_manifest_csv": str(GRADCAM_HEATMAP_MANIFEST_CSV),
        "failure_log_csv": str(GRADCAM_FAILURE_LOG_CSV),
        "visual_cases_csv": str(GRADCAM_VISUAL_CASES_CSV),
        "visual_panel_png": str(GRADCAM_VISUAL_PANEL_PNG),
        "visual_panel_pdf": str(GRADCAM_VISUAL_PANEL_PDF),
    },
}

with open(GRADCAM_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary_payload, f, indent=2)

print("Grad-CAM++ region summary:")
print(
    region_summary[
        [
            "model_id",
            "branch_space",
            "region_name",
            "n_images",
            "n_patients",
            "observed_energy_fraction_mean",
            "observed_minus_random_mean_mean",
            "observed_div_random_mean_median",
            "empirical_p_random_ge_observed_mean",
        ]
    ].to_string(index=False)
)

print("\nGrad-CAM++ patient-level statistics:")
print(
    patient_stats[
        patient_stats["region_name"].isin(
            [
                "OD",
                "OC",
                "Rim",
                "OD_plus_OC",
            ]
        )
    ][
        [
            "model_id",
            "branch_space",
            "region_name",
            "n_patients",
            "mean",
            "median",
            "ci95_low_percentile",
            "ci95_high_percentile",
            "wilcoxon_p_two_sided",
            "wilcoxon_p_greater",
            "wilcoxon_p_less",
        ]
    ].to_string(index=False)
)

print(f"\nGrad-CAM++ metrics CSV: {GRADCAM_METRICS_CSV}")
print(f"Grad-CAM++ image summary CSV: {GRADCAM_IMAGE_SUMMARY_CSV}")
print(f"Grad-CAM++ patient metrics CSV: {GRADCAM_PATIENT_METRICS_CSV}")
print(f"Grad-CAM++ patient stats CSV: {GRADCAM_PATIENT_STATS_CSV}")
print(f"Grad-CAM++ region summary CSV: {GRADCAM_REGION_SUMMARY_CSV}")
print(f"Grad-CAM++ heatmap manifest CSV: {GRADCAM_HEATMAP_MANIFEST_CSV}")
print(f"Grad-CAM++ failure log CSV: {GRADCAM_FAILURE_LOG_CSV}")
print(f"Grad-CAM++ visual cases CSV: {GRADCAM_VISUAL_CASES_CSV}")
print(f"Grad-CAM++ visual panel PNG: {GRADCAM_VISUAL_PANEL_PNG}")
print(f"Grad-CAM++ visual panel PDF: {GRADCAM_VISUAL_PANEL_PDF}")
print(f"Grad-CAM++ summary JSON: {GRADCAM_SUMMARY_JSON}")

required_outputs = [
    GRADCAM_METRICS_CSV,
    GRADCAM_IMAGE_SUMMARY_CSV,
    GRADCAM_PATIENT_METRICS_CSV,
    GRADCAM_PATIENT_STATS_CSV,
    GRADCAM_REGION_SUMMARY_CSV,
    GRADCAM_HEATMAP_MANIFEST_CSV,
    GRADCAM_FAILURE_LOG_CSV,
    GRADCAM_VISUAL_CASES_CSV,
    GRADCAM_VISUAL_PANEL_PNG,
    GRADCAM_VISUAL_PANEL_PDF,
    GRADCAM_SUMMARY_JSON,
]

missing_outputs = [
    str(path)
    for path in required_outputs
    if not Path(path).exists()
]

if missing_outputs:
    raise RuntimeError(f"Missing Cell 7 outputs: {missing_outputs}")

if len(failure_df) > 0:
    raise RuntimeError("Grad-CAM++ completed with failures.")

if len(heatmap_manifest_df) != len(priority_xai_manifest):
    raise RuntimeError(
        f"Unexpected number of Grad-CAM++ heatmaps: "
        f"{len(heatmap_manifest_df)} != {len(priority_xai_manifest)}"
    )

if not heatmap_manifest_df["status"].eq("pass").all():
    raise RuntimeError("At least one Grad-CAM++ heatmap failed.")

if len(metrics_df) != len(priority_xai_manifest) * len(REGIONS_FOR_DIRECT_ENERGY):
    raise RuntimeError(
        f"Unexpected number of Grad-CAM++ metric rows: "
        f"{len(metrics_df)} != {len(priority_xai_manifest) * len(REGIONS_FOR_DIRECT_ENERGY)}"
    )

print("Status: PASS")

In [ ]:
# Cell 8: Integrated Gradients anatomical enrichment for priority Multi-XAI targets

import json
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

REQUIRE_GPU_FOR_FULL_XAI = True

if REQUIRE_GPU_FOR_FULL_XAI and DEVICE.type != "cuda":
    raise RuntimeError(
        "Integrated Gradients full priority XAI should not be executed on CPU. "
        "Switch Colab runtime to GPU, rerun Cell 1 and Cell 6, then rerun this Cell 8."
    )

IG_STEPS = 24
IG_BASELINE_MODE = "zero_normalized_tensor"
SAVE_ALL_HEATMAPS = True

IG_ROOT = XAI_RUNTIME_ROOT / "integrated_gradients_priority_models"
IG_HEATMAP_ROOT = IG_ROOT / "heatmaps"
IG_OVERLAY_ROOT = IG_ROOT / "overlays"

IG_ROOT.mkdir(parents=True, exist_ok=True)
IG_HEATMAP_ROOT.mkdir(parents=True, exist_ok=True)
IG_OVERLAY_ROOT.mkdir(parents=True, exist_ok=True)

IG_METRICS_CSV = (
    IG_ROOT
    / "notebook12_integratedgradients_priority_anatomical_metrics.csv"
)

IG_IMAGE_SUMMARY_CSV = (
    IG_ROOT
    / "notebook12_integratedgradients_priority_image_summary.csv"
)

IG_PATIENT_METRICS_CSV = (
    IG_ROOT
    / "notebook12_integratedgradients_priority_patient_level_metrics.csv"
)

IG_PATIENT_STATS_CSV = (
    IG_ROOT
    / "notebook12_integratedgradients_priority_patient_level_statistics.csv"
)

IG_REGION_SUMMARY_CSV = (
    IG_ROOT
    / "notebook12_integratedgradients_priority_region_summary.csv"
)

IG_HEATMAP_MANIFEST_CSV = (
    IG_ROOT
    / "notebook12_integratedgradients_priority_heatmap_manifest.csv"
)

IG_FAILURE_LOG_CSV = (
    IG_ROOT
    / "notebook12_integratedgradients_priority_failure_log.csv"
)

IG_VISUAL_CASES_CSV = (
    IG_ROOT
    / "notebook12_integratedgradients_priority_visual_cases.csv"
)

IG_VISUAL_PANEL_PNG = (
    IG_ROOT
    / "notebook12_integratedgradients_priority_visual_panel.png"
)

IG_VISUAL_PANEL_PDF = (
    IG_ROOT
    / "notebook12_integratedgradients_priority_visual_panel.pdf"
)

IG_SUMMARY_JSON = (
    IG_ROOT
    / "notebook12_integratedgradients_priority_summary.json"
)

FAILURE_LOG_COLUMNS = [
    "model_id",
    "branch_space",
    "sample_id",
    "patient_id",
    "status",
    "error",
]

priority_xai_manifest = mask_manifest[
    mask_manifest["branch_space"].astype(str).isin(priority_branch_spaces)
].copy()

priority_xai_manifest = priority_xai_manifest.sort_values(
    [
        "model_id",
        "branch_space",
        "sample_id",
    ]
).reset_index(drop=True)

if len(priority_xai_manifest) == 0:
    raise RuntimeError("Priority XAI manifest is empty.")

expected_rows = 95 * 4

if len(priority_xai_manifest) != expected_rows:
    raise RuntimeError(
        f"Unexpected number of priority XAI rows: {len(priority_xai_manifest)} != {expected_rows}"
    )


def compute_integrated_gradients_heatmap(
    forward_function,
    input_tensor,
    steps=IG_STEPS,
    baseline_tensor=None,
):
    if baseline_tensor is None:
        baseline_tensor = torch.zeros_like(input_tensor)

    accumulated_gradients = torch.zeros_like(input_tensor)

    last_logit = None

    for step_idx in range(1, steps + 1):
        alpha = float(step_idx) / float(steps)

        scaled_input = baseline_tensor + alpha * (input_tensor - baseline_tensor)
        scaled_input = scaled_input.detach().clone().requires_grad_(True)

        output = forward_function(scaled_input)

        if output.ndim > 0:
            score = output.reshape(-1)[0]
        else:
            score = output

        score.backward()

        if scaled_input.grad is None:
            raise RuntimeError("Integrated Gradients did not produce input gradients.")

        accumulated_gradients += scaled_input.grad.detach()
        last_logit = output.detach()

    average_gradients = accumulated_gradients / float(steps)
    integrated_gradients = (input_tensor - baseline_tensor) * average_gradients

    attribution = integrated_gradients.detach().abs().sum(dim=1)[0].cpu().numpy()
    heatmap = normalize_heatmap(attribution)

    logit = float(last_logit.cpu().numpy().reshape(-1)[0])
    probability = float(torch.sigmoid(last_logit).cpu().numpy().reshape(-1)[0])

    return heatmap, logit, probability


def run_integrated_gradients_for_mask_row(
    row,
    hybrid_model,
    od25_model,
    outside_model,
):
    model_id = str(row["model_id"])
    branch_space = str(row["branch_space"])
    sample_id = str(row["sample_id"])
    patient_id = str(row["patient_id"])
    y_true = int(row["y_true"])

    if model_id == "hybrid__full_image_plus_od_centered_2p5x_effb0":
        full_path, od_path = get_hybrid_input_paths(
            sample_id=sample_id,
            patient_id=patient_id,
        )

        full_tensor, full_rgb, full_resolved_path = image_to_tensor(
            full_path,
            use_local_candidate=True,
        )

        od_tensor, od_rgb, od_resolved_path = image_to_tensor(
            od_path,
            use_local_candidate=False,
        )

        full_tensor = full_tensor.to(DEVICE)
        od_tensor = od_tensor.to(DEVICE)

        if branch_space == "hybrid_full_image_branch":
            target_tensor = full_tensor
            target_rgb = full_rgb
            resolved_input_path = full_resolved_path

            def forward_function(variable_full_tensor):
                hybrid_model.zero_grad(set_to_none=True)
                return hybrid_model(variable_full_tensor, od_tensor)

        elif branch_space == "hybrid_od_centered_branch":
            target_tensor = od_tensor
            target_rgb = od_rgb
            resolved_input_path = od_resolved_path

            def forward_function(variable_od_tensor):
                hybrid_model.zero_grad(set_to_none=True)
                return hybrid_model(full_tensor, variable_od_tensor)

        else:
            raise ValueError(f"Unexpected hybrid branch_space: {branch_space}")

        baseline_tensor = torch.zeros_like(target_tensor)

        heatmap, logit, probability = compute_integrated_gradients_heatmap(
            forward_function=forward_function,
            input_tensor=target_tensor,
            steps=IG_STEPS,
            baseline_tensor=baseline_tensor,
        )

    elif model_id == "anatomy__od_centered_crop_2p5x":
        tensor, image_rgb, resolved_input_path = image_to_tensor(
            row["resolved_input_path"],
            use_local_candidate=False,
        )

        tensor = tensor.to(DEVICE)
        target_rgb = image_rgb

        def forward_function(variable_tensor):
            od25_model.zero_grad(set_to_none=True)
            return od25_model(variable_tensor).squeeze(1)

        baseline_tensor = torch.zeros_like(tensor)

        heatmap, logit, probability = compute_integrated_gradients_heatmap(
            forward_function=forward_function,
            input_tensor=tensor,
            steps=IG_STEPS,
            baseline_tensor=baseline_tensor,
        )

    elif model_id == "anatomy__outside_od_only_trained_control":
        tensor, image_rgb, resolved_input_path = image_to_tensor(
            row["resolved_input_path"],
            use_local_candidate=False,
        )

        tensor = tensor.to(DEVICE)
        target_rgb = image_rgb

        def forward_function(variable_tensor):
            outside_model.zero_grad(set_to_none=True)
            return outside_model(variable_tensor).squeeze(1)

        baseline_tensor = torch.zeros_like(tensor)

        heatmap, logit, probability = compute_integrated_gradients_heatmap(
            forward_function=forward_function,
            input_tensor=tensor,
            steps=IG_STEPS,
            baseline_tensor=baseline_tensor,
        )

    else:
        raise ValueError(f"Unknown model_id: {model_id}")

    heatmap_path = (
        IG_HEATMAP_ROOT
        / model_id
        / branch_space
        / f"{sample_id}_integratedgradients_heatmap.png"
    )

    overlay_path = (
        IG_OVERLAY_ROOT
        / model_id
        / branch_space
        / f"{sample_id}_integratedgradients_overlay.png"
    )

    if SAVE_ALL_HEATMAPS:
        heatmap_path = save_heatmap_png(
            heatmap,
            heatmap_path,
        )

        overlay_path = save_overlay_png(
            target_rgb,
            heatmap,
            overlay_path,
        )
    else:
        heatmap_path = ""
        overlay_path = ""

    metric_rows = build_region_metric_rows(
        heatmap=heatmap,
        mask_row=row,
        model_id=model_id,
        branch_space=branch_space,
        method_name="IntegratedGradients",
        sample_id=sample_id,
        patient_id=patient_id,
        y_true=y_true,
        logit=logit,
        probability=probability,
        heatmap_path=heatmap_path,
        overlay_path=overlay_path,
        resolved_input_path=resolved_input_path,
    )

    heatmap_manifest_row = {
        "model_id": model_id,
        "branch_space": branch_space,
        "method_name": "IntegratedGradients",
        "sample_id": sample_id,
        "patient_id": patient_id,
        "y_true": y_true,
        "logit": float(logit),
        "probability": float(probability),
        "heatmap_path": str(heatmap_path),
        "overlay_path": str(overlay_path),
        "resolved_input_path": str(resolved_input_path),
        "ig_steps": int(IG_STEPS),
        "ig_baseline_mode": IG_BASELINE_MODE,
        "status": "pass",
        "error": "",
    }

    return metric_rows, heatmap_manifest_row


def build_patient_level_statistics(patient_metrics_df):
    try:
        from scipy.stats import wilcoxon
        scipy_available = True

    except Exception:
        scipy_available = False

    stat_rows = []

    grouped = patient_metrics_df.groupby(
        [
            "model_id",
            "branch_space",
            "method_name",
            "region_name",
        ]
    )

    for (model_id, branch_space, method_name, region_name), block in grouped:
        endpoint = (
            block["observed_minus_random_mean"]
            .astype(float)
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
        )

        values = endpoint.values.astype(float)

        if len(values) >= 5:
            mean_value = float(np.mean(values))
            median_value = float(np.median(values))
            std_value = float(np.std(values, ddof=1)) if len(values) > 1 else np.nan
            ci95_low = float(np.percentile(values, 2.5))
            ci95_high = float(np.percentile(values, 97.5))
        else:
            mean_value = np.nan
            median_value = np.nan
            std_value = np.nan
            ci95_low = np.nan
            ci95_high = np.nan

        if scipy_available and len(values) >= 5 and not np.allclose(values, 0):
            p_two_sided = float(wilcoxon(values, alternative="two-sided").pvalue)
            p_greater = float(wilcoxon(values, alternative="greater").pvalue)
            p_less = float(wilcoxon(values, alternative="less").pvalue)
        else:
            p_two_sided = np.nan
            p_greater = np.nan
            p_less = np.nan

        positive = int(np.sum(values > 0))
        negative = int(np.sum(values < 0))
        denominator = positive + negative

        if denominator > 0:
            signed_direction_effect_proxy = float((positive - negative) / denominator)
        else:
            signed_direction_effect_proxy = np.nan

        stat_rows.append(
            {
                "model_id": model_id,
                "branch_space": branch_space,
                "method_name": method_name,
                "region_name": region_name,
                "endpoint_name": "observed_minus_random_mean",
                "n_patients": int(len(values)),
                "mean": mean_value,
                "median": median_value,
                "std": std_value,
                "ci95_low_percentile": ci95_low,
                "ci95_high_percentile": ci95_high,
                "wilcoxon_p_two_sided": p_two_sided,
                "wilcoxon_p_greater": p_greater,
                "wilcoxon_p_less": p_less,
                "signed_direction_effect_proxy": signed_direction_effect_proxy,
            }
        )

    return pd.DataFrame(stat_rows)


print("Integrated Gradients priority Multi-XAI started")
print(f"Device: {DEVICE}")
print(f"Rows to process: {len(priority_xai_manifest)}")
print(f"IG steps: {IG_STEPS}")
print(f"Baseline mode: {IG_BASELINE_MODE}")
print(f"Random controls per region: {RANDOM_CONTROLS_PER_REGION}")
print(f"Save heatmaps: {SAVE_ALL_HEATMAPS}")

start_time = time.time()

hybrid_model = load_hybrid_model(hybrid_checkpoint)
od25_model = load_binary_model(od25_checkpoint)
outside_model = load_binary_model(outside_od_checkpoint)

hybrid_model.eval()
od25_model.eval()
outside_model.eval()

all_metric_rows = []
heatmap_manifest_rows = []
failure_rows = []

for _, row in tqdm(
    priority_xai_manifest.iterrows(),
    total=len(priority_xai_manifest),
    desc="Integrated Gradients priority XAI",
):
    try:
        metric_rows, heatmap_manifest_row = run_integrated_gradients_for_mask_row(
            row=row,
            hybrid_model=hybrid_model,
            od25_model=od25_model,
            outside_model=outside_model,
        )

        all_metric_rows.extend(metric_rows)
        heatmap_manifest_rows.append(heatmap_manifest_row)

    except Exception as exc:
        failure_row = {
            "model_id": str(row.get("model_id", "")),
            "branch_space": str(row.get("branch_space", "")),
            "sample_id": str(row.get("sample_id", "")),
            "patient_id": str(row.get("patient_id", "")),
            "status": "fail",
            "error": repr(exc),
        }

        failure_rows.append(failure_row)

        heatmap_manifest_rows.append(
            {
                "model_id": str(row.get("model_id", "")),
                "branch_space": str(row.get("branch_space", "")),
                "method_name": "IntegratedGradients",
                "sample_id": str(row.get("sample_id", "")),
                "patient_id": str(row.get("patient_id", "")),
                "y_true": int(row.get("y_true", -1)),
                "logit": np.nan,
                "probability": np.nan,
                "heatmap_path": "",
                "overlay_path": "",
                "resolved_input_path": "",
                "ig_steps": int(IG_STEPS),
                "ig_baseline_mode": IG_BASELINE_MODE,
                "status": "fail",
                "error": repr(exc),
            }
        )

del hybrid_model, od25_model, outside_model

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

metrics_df = pd.DataFrame(all_metric_rows)
heatmap_manifest_df = pd.DataFrame(heatmap_manifest_rows)
failure_df = pd.DataFrame(failure_rows, columns=FAILURE_LOG_COLUMNS)

metrics_df.to_csv(IG_METRICS_CSV, index=False)
heatmap_manifest_df.to_csv(IG_HEATMAP_MANIFEST_CSV, index=False)
failure_df.to_csv(IG_FAILURE_LOG_CSV, index=False)

if len(failure_df) > 0:
    print(failure_df.head(20).to_string(index=False))
    raise RuntimeError(f"Integrated Gradients failed for {len(failure_df)} rows.")

image_summary = (
    metrics_df
    .pivot_table(
        index=[
            "model_id",
            "branch_space",
            "method_name",
            "sample_id",
            "patient_id",
            "y_true",
            "probability",
        ],
        columns="region_name",
        values="observed_energy_fraction",
        aggfunc="first",
    )
    .reset_index()
)

image_summary.columns.name = None
image_summary.to_csv(IG_IMAGE_SUMMARY_CSV, index=False)

patient_metrics = (
    metrics_df
    .groupby(
        [
            "model_id",
            "branch_space",
            "method_name",
            "patient_id",
            "region_name",
        ]
    )
    .agg(
        n_images=("sample_id", "nunique"),
        y_true=("y_true", "max"),
        probability_mean=("probability", "mean"),
        observed_energy_fraction_mean=("observed_energy_fraction", "mean"),
        random_energy_fraction_mean=("random_energy_fraction_mean", "mean"),
        observed_minus_random_mean=("observed_minus_random_mean", "mean"),
        observed_div_random_mean=("observed_div_random_mean", "median"),
        empirical_p_random_ge_observed_mean=("empirical_p_random_ge_observed", "mean"),
        region_area_pixels_mean=("region_area_pixels", "mean"),
        fov_area_pixels_mean=("fov_area_pixels", "mean"),
    )
    .reset_index()
)

patient_metrics.to_csv(IG_PATIENT_METRICS_CSV, index=False)

patient_stats = build_patient_level_statistics(patient_metrics)
patient_stats.to_csv(IG_PATIENT_STATS_CSV, index=False)

region_summary = (
    metrics_df
    .groupby(
        [
            "model_id",
            "branch_space",
            "method_name",
            "region_name",
        ]
    )
    .agg(
        n_rows=("sample_id", "count"),
        n_images=("sample_id", "nunique"),
        n_patients=("patient_id", "nunique"),
        observed_energy_fraction_mean=("observed_energy_fraction", "mean"),
        observed_energy_fraction_median=("observed_energy_fraction", "median"),
        random_energy_fraction_mean_mean=("random_energy_fraction_mean", "mean"),
        observed_minus_random_mean_mean=("observed_minus_random_mean", "mean"),
        observed_div_random_mean_median=("observed_div_random_mean", "median"),
        empirical_p_random_ge_observed_mean=("empirical_p_random_ge_observed", "mean"),
        region_area_pixels_mean=("region_area_pixels", "mean"),
        fov_area_pixels_mean=("fov_area_pixels", "mean"),
    )
    .reset_index()
)

region_summary.to_csv(IG_REGION_SUMMARY_CSV, index=False)

visual_cases = (
    heatmap_manifest_df
    .sort_values(
        [
            "model_id",
            "branch_space",
            "sample_id",
        ]
    )
    .groupby(["model_id", "branch_space"])
    .head(1)
    .reset_index(drop=True)
)

visual_cases.to_csv(IG_VISUAL_CASES_CSV, index=False)

n_rows = len(visual_cases)

fig, axes = plt.subplots(
    n_rows,
    4,
    figsize=(14, max(3.2, 3.0 * n_rows)),
)

if n_rows == 1:
    axes = np.expand_dims(axes, axis=0)

for row_idx, (_, row) in enumerate(visual_cases.iterrows()):
    image_rgb, _ = read_rgb_image_robust(
        row["resolved_input_path"],
        use_local_candidate=False,
    )

    image_rgb = cv2.resize(
        image_rgb,
        (XAI_INPUT_SIZE, XAI_INPUT_SIZE),
        interpolation=cv2.INTER_AREA,
    )

    heatmap_gray = cv2.imread(
        str(row["heatmap_path"]),
        cv2.IMREAD_GRAYSCALE,
    )

    if heatmap_gray is None:
        heatmap_gray = np.zeros(
            (XAI_INPUT_SIZE, XAI_INPUT_SIZE),
            dtype=np.uint8,
        )

    heatmap_float = heatmap_gray.astype(np.float32) / 255.0
    heatmap_u8 = np.clip(heatmap_float * 255.0, 0, 255).astype(np.uint8)

    heatmap_color = cv2.cvtColor(
        cv2.applyColorMap(heatmap_u8, cv2.COLORMAP_JET),
        cv2.COLOR_BGR2RGB,
    )

    heatmap_overlay = (
        0.60 * image_rgb.astype(np.float32)
        + 0.40 * heatmap_color.astype(np.float32)
    )
    heatmap_overlay = np.clip(heatmap_overlay, 0, 255).astype(np.uint8)

    mask_row = priority_xai_manifest[
        (priority_xai_manifest["model_id"].astype(str) == str(row["model_id"]))
        & (priority_xai_manifest["branch_space"].astype(str) == str(row["branch_space"]))
        & (priority_xai_manifest["sample_id"].astype(str) == str(row["sample_id"]))
    ].iloc[0]

    od = read_binary_mask(mask_row["OD_mask_path"])
    oc = read_binary_mask(mask_row["OC_mask_path"])

    mask_overlay = image_rgb.copy()

    mask_overlay[od.astype(bool)] = (
        0.65 * mask_overlay[od.astype(bool)]
        + 0.35 * np.array([255, 170, 0])
    ).astype(np.uint8)

    mask_overlay[oc.astype(bool)] = (
        0.55 * mask_overlay[oc.astype(bool)]
        + 0.45 * np.array([0, 180, 255])
    ).astype(np.uint8)

    axes[row_idx, 0].imshow(image_rgb)
    axes[row_idx, 0].set_title("Input")

    axes[row_idx, 1].imshow(heatmap_color)
    axes[row_idx, 1].set_title("Integrated Gradients")

    axes[row_idx, 2].imshow(heatmap_overlay)
    axes[row_idx, 2].set_title("Attribution overlay")

    axes[row_idx, 3].imshow(mask_overlay)
    axes[row_idx, 3].set_title("OD/OC mask")

    ylabel = (
        f"{row['model_id']}\n"
        f"{row['branch_space']}\n"
        f"{row['sample_id']}"
    )

    axes[row_idx, 0].set_ylabel(ylabel, fontsize=8)

    for col_idx in range(4):
        axes[row_idx, col_idx].axis("off")

plt.suptitle(
    "Notebook 12 Integrated Gradients visual audit for priority Multi-XAI targets",
    fontsize=13,
    fontweight="bold",
)

plt.tight_layout()
plt.savefig(IG_VISUAL_PANEL_PNG, dpi=220, bbox_inches="tight")
plt.savefig(IG_VISUAL_PANEL_PDF, bbox_inches="tight")
plt.show()

elapsed_min = (time.time() - start_time) / 60.0

summary_payload = {
    "notebook": "12_integrated_internal_model_audit_and_freezing",
    "cell": "08_integratedgradients_priority_multixai",
    "status": "PASS",
    "method_name": "IntegratedGradients",
    "device": str(DEVICE),
    "input_size": int(XAI_INPUT_SIZE),
    "ig_steps": int(IG_STEPS),
    "ig_baseline_mode": IG_BASELINE_MODE,
    "random_controls_per_region": int(RANDOM_CONTROLS_PER_REGION),
    "n_priority_rows": int(len(priority_xai_manifest)),
    "n_metric_rows": int(len(metrics_df)),
    "n_heatmaps": int(len(heatmap_manifest_df)),
    "n_failures": int(len(failure_df)),
    "elapsed_min": float(elapsed_min),
    "outputs": {
        "metrics_csv": str(IG_METRICS_CSV),
        "image_summary_csv": str(IG_IMAGE_SUMMARY_CSV),
        "patient_metrics_csv": str(IG_PATIENT_METRICS_CSV),
        "patient_stats_csv": str(IG_PATIENT_STATS_CSV),
        "region_summary_csv": str(IG_REGION_SUMMARY_CSV),
        "heatmap_manifest_csv": str(IG_HEATMAP_MANIFEST_CSV),
        "failure_log_csv": str(IG_FAILURE_LOG_CSV),
        "visual_cases_csv": str(IG_VISUAL_CASES_CSV),
        "visual_panel_png": str(IG_VISUAL_PANEL_PNG),
        "visual_panel_pdf": str(IG_VISUAL_PANEL_PDF),
    },
}

with open(IG_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary_payload, f, indent=2)

print("Integrated Gradients region summary:")
print(
    region_summary[
        [
            "model_id",
            "branch_space",
            "region_name",
            "n_images",
            "n_patients",
            "observed_energy_fraction_mean",
            "observed_minus_random_mean_mean",
            "observed_div_random_mean_median",
            "empirical_p_random_ge_observed_mean",
        ]
    ].to_string(index=False)
)

print("\nIntegrated Gradients patient-level statistics:")
print(
    patient_stats[
        patient_stats["region_name"].isin(
            [
                "OD",
                "OC",
                "Rim",
                "OD_plus_OC",
            ]
        )
    ][
        [
            "model_id",
            "branch_space",
            "region_name",
            "n_patients",
            "mean",
            "median",
            "ci95_low_percentile",
            "ci95_high_percentile",
            "wilcoxon_p_two_sided",
            "wilcoxon_p_greater",
            "wilcoxon_p_less",
        ]
    ].to_string(index=False)
)

print(f"\nIntegrated Gradients metrics CSV: {IG_METRICS_CSV}")
print(f"Integrated Gradients image summary CSV: {IG_IMAGE_SUMMARY_CSV}")
print(f"Integrated Gradients patient metrics CSV: {IG_PATIENT_METRICS_CSV}")
print(f"Integrated Gradients patient stats CSV: {IG_PATIENT_STATS_CSV}")
print(f"Integrated Gradients region summary CSV: {IG_REGION_SUMMARY_CSV}")
print(f"Integrated Gradients heatmap manifest CSV: {IG_HEATMAP_MANIFEST_CSV}")
print(f"Integrated Gradients failure log CSV: {IG_FAILURE_LOG_CSV}")
print(f"Integrated Gradients visual cases CSV: {IG_VISUAL_CASES_CSV}")
print(f"Integrated Gradients visual panel PNG: {IG_VISUAL_PANEL_PNG}")
print(f"Integrated Gradients visual panel PDF: {IG_VISUAL_PANEL_PDF}")
print(f"Integrated Gradients summary JSON: {IG_SUMMARY_JSON}")

required_outputs = [
    IG_METRICS_CSV,
    IG_IMAGE_SUMMARY_CSV,
    IG_PATIENT_METRICS_CSV,
    IG_PATIENT_STATS_CSV,
    IG_REGION_SUMMARY_CSV,
    IG_HEATMAP_MANIFEST_CSV,
    IG_FAILURE_LOG_CSV,
    IG_VISUAL_CASES_CSV,
    IG_VISUAL_PANEL_PNG,
    IG_VISUAL_PANEL_PDF,
    IG_SUMMARY_JSON,
]

missing_outputs = [
    str(path)
    for path in required_outputs
    if not Path(path).exists()
]

if missing_outputs:
    raise RuntimeError(f"Missing Cell 8 outputs: {missing_outputs}")

if len(failure_df) > 0:
    raise RuntimeError("Integrated Gradients completed with failures.")

if len(heatmap_manifest_df) != len(priority_xai_manifest):
    raise RuntimeError(
        f"Unexpected number of Integrated Gradients heatmaps: "
        f"{len(heatmap_manifest_df)} != {len(priority_xai_manifest)}"
    )

if not heatmap_manifest_df["status"].eq("pass").all():
    raise RuntimeError("At least one Integrated Gradients heatmap failed.")

if len(metrics_df) != len(priority_xai_manifest) * len(REGIONS_FOR_DIRECT_ENERGY):
    raise RuntimeError(
        f"Unexpected number of Integrated Gradients metric rows: "
        f"{len(metrics_df)} != {len(priority_xai_manifest) * len(REGIONS_FOR_DIRECT_ENERGY)}"
    )

print("Status: PASS")

In [ ]:
# Cell 9: Anatomical occlusion for priority Multi-XAI targets

from tqdm.auto import tqdm
import matplotlib.pyplot as plt

REQUIRE_GPU_FOR_FULL_XAI = True

if REQUIRE_GPU_FOR_FULL_XAI and DEVICE.type != "cuda":
    raise RuntimeError(
        "Anatomical occlusion should not be executed on CPU. "
        "Switch Colab runtime to GPU, rerun Cell 1 and Cell 6, then rerun this Cell 9."
    )

OCCLUSION_RANDOM_CONTROLS_PER_REGION = 24
SAVE_VISUAL_PANEL = True

OCCLUSION_ROOT = XAI_RUNTIME_ROOT / "anatomical_occlusion_priority_models"
OCCLUSION_ROOT.mkdir(parents=True, exist_ok=True)

OCCLUSION_METRICS_CSV = (
    OCCLUSION_ROOT
    / "notebook12_anatomical_occlusion_priority_metrics.csv"
)

OCCLUSION_PATIENT_METRICS_CSV = (
    OCCLUSION_ROOT
    / "notebook12_anatomical_occlusion_priority_patient_level_metrics.csv"
)

OCCLUSION_PATIENT_STATS_CSV = (
    OCCLUSION_ROOT
    / "notebook12_anatomical_occlusion_priority_patient_level_statistics.csv"
)

OCCLUSION_REGION_SUMMARY_CSV = (
    OCCLUSION_ROOT
    / "notebook12_anatomical_occlusion_priority_region_summary.csv"
)

OCCLUSION_FAILURE_LOG_CSV = (
    OCCLUSION_ROOT
    / "notebook12_anatomical_occlusion_priority_failure_log.csv"
)

OCCLUSION_VISUAL_CASES_CSV = (
    OCCLUSION_ROOT
    / "notebook12_anatomical_occlusion_priority_visual_cases.csv"
)

OCCLUSION_VISUAL_PANEL_PNG = (
    OCCLUSION_ROOT
    / "notebook12_anatomical_occlusion_priority_visual_panel.png"
)

OCCLUSION_VISUAL_PANEL_PDF = (
    OCCLUSION_ROOT
    / "notebook12_anatomical_occlusion_priority_visual_panel.pdf"
)

OCCLUSION_SUMMARY_JSON = (
    OCCLUSION_ROOT
    / "notebook12_anatomical_occlusion_priority_summary.json"
)

FAILURE_LOG_COLUMNS = [
    "model_id",
    "branch_space",
    "sample_id",
    "patient_id",
    "status",
    "error",
]

OCCLUSION_REGIONS = [
    "OD",
    "OC",
    "Rim",
    "OD_plus_OC",
    "Outside_OD",
]

RANDOM_CONTROL_REGIONS_FOR_OCCLUSION = [
    "OD",
    "OC",
    "Rim",
    "OD_plus_OC",
]

priority_xai_manifest = mask_manifest[
    mask_manifest["branch_space"].astype(str).isin(priority_branch_spaces)
].copy()

priority_xai_manifest = priority_xai_manifest.sort_values(
    [
        "model_id",
        "branch_space",
        "sample_id",
    ]
).reset_index(drop=True)

if len(priority_xai_manifest) == 0:
    raise RuntimeError("Priority XAI manifest is empty.")

expected_rows = 95 * 4

if len(priority_xai_manifest) != expected_rows:
    raise RuntimeError(
        f"Unexpected number of priority XAI rows: {len(priority_xai_manifest)} != {expected_rows}"
    )


def resize_rgb_to_xai(image_rgb):
    return cv2.resize(
        image_rgb,
        (XAI_INPUT_SIZE, XAI_INPUT_SIZE),
        interpolation=cv2.INTER_AREA,
    )


def rgb_to_model_tensor(image_rgb_384):
    pil_image = Image.fromarray(image_rgb_384.astype(np.uint8))
    tensor = xai_transform(pil_image).unsqueeze(0).to(DEVICE)

    return tensor


def neutralize_region_rgb(image_rgb_384, region_mask, fov_mask):
    image_rgb_384 = image_rgb_384.copy().astype(np.uint8)
    region_mask = region_mask.astype(bool)
    fov_mask = fov_mask.astype(bool)

    if region_mask.sum() == 0:
        return image_rgb_384

    source_mask = fov_mask & (~region_mask)

    if source_mask.sum() < 10:
        source_mask = fov_mask

    if source_mask.sum() < 10:
        source_mask = np.ones(region_mask.shape, dtype=bool)

    replacement_color = np.median(
        image_rgb_384[source_mask],
        axis=0,
    )

    neutralized = image_rgb_384.copy()
    neutralized[region_mask] = replacement_color.astype(np.uint8)

    return neutralized


def predict_hybrid_probability(model, full_tensor, od_tensor):
    with torch.no_grad():
        logits = model(full_tensor, od_tensor)
        probability = torch.sigmoid(logits).detach().cpu().numpy().reshape(-1)[0]

    return float(logits.detach().cpu().numpy().reshape(-1)[0]), float(probability)


def predict_single_probability(model, tensor):
    with torch.no_grad():
        logits = model(tensor).squeeze(1)
        probability = torch.sigmoid(logits).detach().cpu().numpy().reshape(-1)[0]

    return float(logits.detach().cpu().numpy().reshape(-1)[0]), float(probability)


def load_branch_inputs_for_occlusion(row):
    model_id = str(row["model_id"])
    branch_space = str(row["branch_space"])
    sample_id = str(row["sample_id"])
    patient_id = str(row["patient_id"])

    if model_id == "hybrid__full_image_plus_od_centered_2p5x_effb0":
        full_path, od_path = get_hybrid_input_paths(
            sample_id=sample_id,
            patient_id=patient_id,
        )

        full_rgb_raw, full_resolved_path = read_rgb_image_robust(
            full_path,
            use_local_candidate=True,
        )

        od_rgb_raw, od_resolved_path = read_rgb_image_robust(
            od_path,
            use_local_candidate=False,
        )

        full_rgb = resize_rgb_to_xai(full_rgb_raw)
        od_rgb = resize_rgb_to_xai(od_rgb_raw)

        full_tensor = rgb_to_model_tensor(full_rgb)
        od_tensor = rgb_to_model_tensor(od_rgb)

        if branch_space == "hybrid_full_image_branch":
            target_rgb = full_rgb
            fixed_rgb = od_rgb
            target_name = "full"
            resolved_input_path = full_resolved_path

        elif branch_space == "hybrid_od_centered_branch":
            target_rgb = od_rgb
            fixed_rgb = full_rgb
            target_name = "od"
            resolved_input_path = od_resolved_path

        else:
            raise ValueError(f"Unexpected hybrid branch_space: {branch_space}")

        return {
            "model_id": model_id,
            "branch_space": branch_space,
            "target_name": target_name,
            "target_rgb": target_rgb,
            "fixed_rgb": fixed_rgb,
            "full_rgb": full_rgb,
            "od_rgb": od_rgb,
            "full_tensor": full_tensor,
            "od_tensor": od_tensor,
            "resolved_input_path": resolved_input_path,
        }

    image_rgb_raw, resolved_input_path = read_rgb_image_robust(
        row["resolved_input_path"],
        use_local_candidate=False,
    )

    image_rgb = resize_rgb_to_xai(image_rgb_raw)
    tensor = rgb_to_model_tensor(image_rgb)

    return {
        "model_id": model_id,
        "branch_space": branch_space,
        "target_name": "single",
        "target_rgb": image_rgb,
        "fixed_rgb": None,
        "single_tensor": tensor,
        "resolved_input_path": resolved_input_path,
    }


def predict_original_for_branch(branch_payload, hybrid_model, od25_model, outside_model):
    model_id = branch_payload["model_id"]

    if model_id == "hybrid__full_image_plus_od_centered_2p5x_effb0":
        return predict_hybrid_probability(
            hybrid_model,
            branch_payload["full_tensor"],
            branch_payload["od_tensor"],
        )

    if model_id == "anatomy__od_centered_crop_2p5x":
        return predict_single_probability(
            od25_model,
            branch_payload["single_tensor"],
        )

    if model_id == "anatomy__outside_od_only_trained_control":
        return predict_single_probability(
            outside_model,
            branch_payload["single_tensor"],
        )

    raise ValueError(f"Unknown model_id: {model_id}")


def predict_ablated_for_branch(
    branch_payload,
    ablated_target_rgb,
    hybrid_model,
    od25_model,
    outside_model,
):
    model_id = branch_payload["model_id"]
    target_name = branch_payload["target_name"]

    ablated_target_tensor = rgb_to_model_tensor(ablated_target_rgb)

    if model_id == "hybrid__full_image_plus_od_centered_2p5x_effb0":
        if target_name == "full":
            return predict_hybrid_probability(
                hybrid_model,
                ablated_target_tensor,
                branch_payload["od_tensor"],
            )

        if target_name == "od":
            return predict_hybrid_probability(
                hybrid_model,
                branch_payload["full_tensor"],
                ablated_target_tensor,
            )

        raise ValueError(f"Unexpected hybrid target_name: {target_name}")

    if model_id == "anatomy__od_centered_crop_2p5x":
        return predict_single_probability(
            od25_model,
            ablated_target_tensor,
        )

    if model_id == "anatomy__outside_od_only_trained_control":
        return predict_single_probability(
            outside_model,
            ablated_target_tensor,
        )

    raise ValueError(f"Unknown model_id: {model_id}")


def compute_occlusion_region_metrics(
    row,
    branch_payload,
    original_logit,
    original_probability,
    region_name,
    hybrid_model,
    od25_model,
    outside_model,
):
    sample_id = str(row["sample_id"])
    model_id = str(row["model_id"])
    branch_space = str(row["branch_space"])

    masks = load_region_masks(row)

    region_mask = masks[region_name]
    fov_mask = masks["FOV"]

    region_area = int(region_mask.sum())
    fov_area = int(fov_mask.sum())

    ablated_rgb = neutralize_region_rgb(
        branch_payload["target_rgb"],
        region_mask,
        fov_mask,
    )

    ablated_logit, ablated_probability = predict_ablated_for_branch(
        branch_payload=branch_payload,
        ablated_target_rgb=ablated_rgb,
        hybrid_model=hybrid_model,
        od25_model=od25_model,
        outside_model=outside_model,
    )

    observed_probability_drop = float(original_probability - ablated_probability)
    observed_logit_drop = float(original_logit - ablated_logit)

    random_probability_drops = []
    random_logit_drops = []

    if (
        region_name in RANDOM_CONTROL_REGIONS_FOR_OCCLUSION
        and region_area > 0
        and fov_area > 0
    ):
        rng = np.random.default_rng(
            stable_rng_seed(
                "AnatomicalOcclusion",
                model_id,
                branch_space,
                sample_id,
                region_name,
                SEED,
            )
        )

        for _ in range(OCCLUSION_RANDOM_CONTROLS_PER_REGION):
            random_mask = random_disk_mask(
                target_area=region_area,
                fov_mask=fov_mask,
                rng=rng,
            )

            random_ablated_rgb = neutralize_region_rgb(
                branch_payload["target_rgb"],
                random_mask,
                fov_mask,
            )

            random_logit, random_probability = predict_ablated_for_branch(
                branch_payload=branch_payload,
                ablated_target_rgb=random_ablated_rgb,
                hybrid_model=hybrid_model,
                od25_model=od25_model,
                outside_model=outside_model,
            )

            random_probability_drops.append(float(original_probability - random_probability))
            random_logit_drops.append(float(original_logit - random_logit))

    if len(random_probability_drops) > 0:
        random_probability_drops = np.asarray(random_probability_drops, dtype=float)
        random_logit_drops = np.asarray(random_logit_drops, dtype=float)

        random_probability_drop_mean = float(np.mean(random_probability_drops))
        random_probability_drop_median = float(np.median(random_probability_drops))
        random_logit_drop_mean = float(np.mean(random_logit_drops))
        random_logit_drop_median = float(np.median(random_logit_drops))

        anatomical_minus_random_probability_drop_mean = float(
            observed_probability_drop - random_probability_drop_mean
        )

        anatomical_minus_random_logit_drop_mean = float(
            observed_logit_drop - random_logit_drop_mean
        )

        empirical_p_random_ge_observed_probability_drop = float(
            (np.sum(random_probability_drops >= observed_probability_drop) + 1)
            / (len(random_probability_drops) + 1)
        )

    else:
        random_probability_drop_mean = np.nan
        random_probability_drop_median = np.nan
        random_logit_drop_mean = np.nan
        random_logit_drop_median = np.nan
        anatomical_minus_random_probability_drop_mean = np.nan
        anatomical_minus_random_logit_drop_mean = np.nan
        empirical_p_random_ge_observed_probability_drop = np.nan

    return {
        "region_name": region_name,
        "original_logit": float(original_logit),
        "original_probability": float(original_probability),
        "ablated_logit": float(ablated_logit),
        "ablated_probability": float(ablated_probability),
        "observed_probability_drop": observed_probability_drop,
        "observed_logit_drop": observed_logit_drop,
        "random_probability_drop_mean": random_probability_drop_mean,
        "random_probability_drop_median": random_probability_drop_median,
        "random_logit_drop_mean": random_logit_drop_mean,
        "random_logit_drop_median": random_logit_drop_median,
        "anatomical_minus_random_probability_drop_mean": anatomical_minus_random_probability_drop_mean,
        "anatomical_minus_random_logit_drop_mean": anatomical_minus_random_logit_drop_mean,
        "empirical_p_random_ge_observed_probability_drop": empirical_p_random_ge_observed_probability_drop,
        "n_random_controls": int(len(random_probability_drops)),
        "region_area_pixels": region_area,
        "fov_area_pixels": fov_area,
    }


def build_occlusion_patient_level_statistics(patient_metrics_df):
    try:
        from scipy.stats import wilcoxon
        scipy_available = True

    except Exception:
        scipy_available = False

    stat_rows = []

    grouped = patient_metrics_df.groupby(
        [
            "model_id",
            "branch_space",
            "method_name",
            "region_name",
        ]
    )

    for (model_id, branch_space, method_name, region_name), block in grouped:
        endpoint = (
            block["anatomical_minus_random_probability_drop_mean"]
            .astype(float)
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
        )

        values = endpoint.values.astype(float)

        if len(values) >= 5:
            mean_value = float(np.mean(values))
            median_value = float(np.median(values))
            std_value = float(np.std(values, ddof=1)) if len(values) > 1 else np.nan
            ci95_low = float(np.percentile(values, 2.5))
            ci95_high = float(np.percentile(values, 97.5))
        else:
            mean_value = np.nan
            median_value = np.nan
            std_value = np.nan
            ci95_low = np.nan
            ci95_high = np.nan

        if scipy_available and len(values) >= 5 and not np.allclose(values, 0):
            p_two_sided = float(wilcoxon(values, alternative="two-sided").pvalue)
            p_greater = float(wilcoxon(values, alternative="greater").pvalue)
            p_less = float(wilcoxon(values, alternative="less").pvalue)
        else:
            p_two_sided = np.nan
            p_greater = np.nan
            p_less = np.nan

        positive = int(np.sum(values > 0))
        negative = int(np.sum(values < 0))
        denominator = positive + negative

        if denominator > 0:
            signed_direction_effect_proxy = float((positive - negative) / denominator)
        else:
            signed_direction_effect_proxy = np.nan

        stat_rows.append(
            {
                "model_id": model_id,
                "branch_space": branch_space,
                "method_name": method_name,
                "region_name": region_name,
                "endpoint_name": "anatomical_minus_random_probability_drop_mean",
                "n_patients": int(len(values)),
                "mean": mean_value,
                "median": median_value,
                "std": std_value,
                "ci95_low_percentile": ci95_low,
                "ci95_high_percentile": ci95_high,
                "wilcoxon_p_two_sided": p_two_sided,
                "wilcoxon_p_greater": p_greater,
                "wilcoxon_p_less": p_less,
                "signed_direction_effect_proxy": signed_direction_effect_proxy,
            }
        )

    return pd.DataFrame(stat_rows)


print("Anatomical occlusion priority Multi-XAI started")
print(f"Device: {DEVICE}")
print(f"Rows to process: {len(priority_xai_manifest)}")
print(f"Occlusion regions: {OCCLUSION_REGIONS}")
print(f"Random controls per anatomical region: {OCCLUSION_RANDOM_CONTROLS_PER_REGION}")

start_time = time.time()

hybrid_model = load_hybrid_model(hybrid_checkpoint)
od25_model = load_binary_model(od25_checkpoint)
outside_model = load_binary_model(outside_od_checkpoint)

hybrid_model.eval()
od25_model.eval()
outside_model.eval()

all_metric_rows = []
failure_rows = []
visual_payloads = []

for _, row in tqdm(
    priority_xai_manifest.iterrows(),
    total=len(priority_xai_manifest),
    desc="Anatomical occlusion priority XAI",
):
    try:
        model_id = str(row["model_id"])
        branch_space = str(row["branch_space"])
        sample_id = str(row["sample_id"])
        patient_id = str(row["patient_id"])
        y_true = int(row["y_true"])

        branch_payload = load_branch_inputs_for_occlusion(row)

        original_logit, original_probability = predict_original_for_branch(
            branch_payload=branch_payload,
            hybrid_model=hybrid_model,
            od25_model=od25_model,
            outside_model=outside_model,
        )

        for region_name in OCCLUSION_REGIONS:
            region_metrics = compute_occlusion_region_metrics(
                row=row,
                branch_payload=branch_payload,
                original_logit=original_logit,
                original_probability=original_probability,
                region_name=region_name,
                hybrid_model=hybrid_model,
                od25_model=od25_model,
                outside_model=outside_model,
            )

            all_metric_rows.append(
                {
                    "model_id": model_id,
                    "branch_space": branch_space,
                    "method_name": "AnatomicalOcclusion",
                    "sample_id": sample_id,
                    "patient_id": patient_id,
                    "y_true": y_true,
                    "resolved_input_path": str(branch_payload["resolved_input_path"]),
                    **region_metrics,
                }
            )

        if len(visual_payloads) < len(priority_branch_spaces):
            if not any(
                payload["model_id"] == model_id and payload["branch_space"] == branch_space
                for payload in visual_payloads
            ):
                visual_payloads.append(
                    {
                        "model_id": model_id,
                        "branch_space": branch_space,
                        "sample_id": sample_id,
                        "patient_id": patient_id,
                        "resolved_input_path": str(branch_payload["resolved_input_path"]),
                    }
                )

    except Exception as exc:
        failure_rows.append(
            {
                "model_id": str(row.get("model_id", "")),
                "branch_space": str(row.get("branch_space", "")),
                "sample_id": str(row.get("sample_id", "")),
                "patient_id": str(row.get("patient_id", "")),
                "status": "fail",
                "error": repr(exc),
            }
        )

del hybrid_model, od25_model, outside_model

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

metrics_df = pd.DataFrame(all_metric_rows)
failure_df = pd.DataFrame(failure_rows, columns=FAILURE_LOG_COLUMNS)

metrics_df.to_csv(OCCLUSION_METRICS_CSV, index=False)
failure_df.to_csv(OCCLUSION_FAILURE_LOG_CSV, index=False)

if len(failure_df) > 0:
    print(failure_df.head(20).to_string(index=False))
    raise RuntimeError(f"Anatomical occlusion failed for {len(failure_df)} rows.")

patient_metrics = (
    metrics_df
    .groupby(
        [
            "model_id",
            "branch_space",
            "method_name",
            "patient_id",
            "region_name",
        ]
    )
    .agg(
        n_images=("sample_id", "nunique"),
        y_true=("y_true", "max"),
        original_probability_mean=("original_probability", "mean"),
        ablated_probability_mean=("ablated_probability", "mean"),
        observed_probability_drop_mean=("observed_probability_drop", "mean"),
        observed_logit_drop_mean=("observed_logit_drop", "mean"),
        random_probability_drop_mean=("random_probability_drop_mean", "mean"),
        random_logit_drop_mean=("random_logit_drop_mean", "mean"),
        anatomical_minus_random_probability_drop_mean=(
            "anatomical_minus_random_probability_drop_mean",
            "mean",
        ),
        anatomical_minus_random_logit_drop_mean=(
            "anatomical_minus_random_logit_drop_mean",
            "mean",
        ),
        empirical_p_random_ge_observed_probability_drop_mean=(
            "empirical_p_random_ge_observed_probability_drop",
            "mean",
        ),
        region_area_pixels_mean=("region_area_pixels", "mean"),
        fov_area_pixels_mean=("fov_area_pixels", "mean"),
    )
    .reset_index()
)

patient_metrics.to_csv(OCCLUSION_PATIENT_METRICS_CSV, index=False)

patient_stats = build_occlusion_patient_level_statistics(patient_metrics)
patient_stats.to_csv(OCCLUSION_PATIENT_STATS_CSV, index=False)

region_summary = (
    metrics_df
    .groupby(
        [
            "model_id",
            "branch_space",
            "method_name",
            "region_name",
        ]
    )
    .agg(
        n_rows=("sample_id", "count"),
        n_images=("sample_id", "nunique"),
        n_patients=("patient_id", "nunique"),
        original_probability_mean=("original_probability", "mean"),
        ablated_probability_mean=("ablated_probability", "mean"),
        observed_probability_drop_mean=("observed_probability_drop", "mean"),
        observed_logit_drop_mean=("observed_logit_drop", "mean"),
        random_probability_drop_mean_mean=("random_probability_drop_mean", "mean"),
        anatomical_minus_random_probability_drop_mean_mean=(
            "anatomical_minus_random_probability_drop_mean",
            "mean",
        ),
        empirical_p_random_ge_observed_probability_drop_mean=(
            "empirical_p_random_ge_observed_probability_drop",
            "mean",
        ),
        region_area_pixels_mean=("region_area_pixels", "mean"),
        fov_area_pixels_mean=("fov_area_pixels", "mean"),
    )
    .reset_index()
)

region_summary.to_csv(OCCLUSION_REGION_SUMMARY_CSV, index=False)

visual_cases = pd.DataFrame(visual_payloads)
visual_cases.to_csv(OCCLUSION_VISUAL_CASES_CSV, index=False)

if SAVE_VISUAL_PANEL:
    n_rows = len(visual_cases)

    fig, axes = plt.subplots(
        n_rows,
        4,
        figsize=(14, max(3.2, 3.0 * n_rows)),
    )

    if n_rows == 1:
        axes = np.expand_dims(axes, axis=0)

    for row_idx, (_, visual_row) in enumerate(visual_cases.iterrows()):
        mask_row = priority_xai_manifest[
            (priority_xai_manifest["model_id"].astype(str) == str(visual_row["model_id"]))
            & (priority_xai_manifest["branch_space"].astype(str) == str(visual_row["branch_space"]))
            & (priority_xai_manifest["sample_id"].astype(str) == str(visual_row["sample_id"]))
        ].iloc[0]

        image_rgb_raw, _ = read_rgb_image_robust(
            visual_row["resolved_input_path"],
            use_local_candidate=False,
        )

        image_rgb = resize_rgb_to_xai(image_rgb_raw)

        masks = load_region_masks(mask_row)
        fov_mask = masks["FOV"]

        od_oc_ablated = neutralize_region_rgb(
            image_rgb,
            masks["OD_plus_OC"],
            fov_mask,
        )

        outside_od_ablated = neutralize_region_rgb(
            image_rgb,
            masks["Outside_OD"],
            fov_mask,
        )

        mask_overlay = image_rgb.copy()
        od = masks["OD"]
        oc = masks["OC"]

        mask_overlay[od.astype(bool)] = (
            0.65 * mask_overlay[od.astype(bool)]
            + 0.35 * np.array([255, 170, 0])
        ).astype(np.uint8)

        mask_overlay[oc.astype(bool)] = (
            0.55 * mask_overlay[oc.astype(bool)]
            + 0.45 * np.array([0, 180, 255])
        ).astype(np.uint8)

        axes[row_idx, 0].imshow(image_rgb)
        axes[row_idx, 0].set_title("Input")

        axes[row_idx, 1].imshow(mask_overlay)
        axes[row_idx, 1].set_title("OD/OC mask")

        axes[row_idx, 2].imshow(od_oc_ablated)
        axes[row_idx, 2].set_title("OD+OC neutralized")

        axes[row_idx, 3].imshow(outside_od_ablated)
        axes[row_idx, 3].set_title("Outside-OD neutralized")

        ylabel = (
            f"{visual_row['model_id']}\n"
            f"{visual_row['branch_space']}\n"
            f"{visual_row['sample_id']}"
        )

        axes[row_idx, 0].set_ylabel(ylabel, fontsize=8)

        for col_idx in range(4):
            axes[row_idx, col_idx].axis("off")

    plt.suptitle(
        "Notebook 12 anatomical occlusion visual audit for priority Multi-XAI targets",
        fontsize=13,
        fontweight="bold",
    )

    plt.tight_layout()
    plt.savefig(OCCLUSION_VISUAL_PANEL_PNG, dpi=220, bbox_inches="tight")
    plt.savefig(OCCLUSION_VISUAL_PANEL_PDF, bbox_inches="tight")
    plt.show()

elapsed_min = (time.time() - start_time) / 60.0

summary_payload = {
    "notebook": "12_integrated_internal_model_audit_and_freezing",
    "cell": "09_anatomical_occlusion_priority_multixai",
    "status": "PASS",
    "method_name": "AnatomicalOcclusion",
    "device": str(DEVICE),
    "input_size": int(XAI_INPUT_SIZE),
    "occlusion_regions": OCCLUSION_REGIONS,
    "random_controls_per_anatomical_region": int(OCCLUSION_RANDOM_CONTROLS_PER_REGION),
    "n_priority_rows": int(len(priority_xai_manifest)),
    "n_metric_rows": int(len(metrics_df)),
    "n_failures": int(len(failure_df)),
    "elapsed_min": float(elapsed_min),
    "outputs": {
        "metrics_csv": str(OCCLUSION_METRICS_CSV),
        "patient_metrics_csv": str(OCCLUSION_PATIENT_METRICS_CSV),
        "patient_stats_csv": str(OCCLUSION_PATIENT_STATS_CSV),
        "region_summary_csv": str(OCCLUSION_REGION_SUMMARY_CSV),
        "failure_log_csv": str(OCCLUSION_FAILURE_LOG_CSV),
        "visual_cases_csv": str(OCCLUSION_VISUAL_CASES_CSV),
        "visual_panel_png": str(OCCLUSION_VISUAL_PANEL_PNG),
        "visual_panel_pdf": str(OCCLUSION_VISUAL_PANEL_PDF),
    },
}

with open(OCCLUSION_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary_payload, f, indent=2)

print("Anatomical occlusion region summary:")
print(
    region_summary[
        [
            "model_id",
            "branch_space",
            "region_name",
            "n_images",
            "n_patients",
            "observed_probability_drop_mean",
            "random_probability_drop_mean_mean",
            "anatomical_minus_random_probability_drop_mean_mean",
            "empirical_p_random_ge_observed_probability_drop_mean",
        ]
    ].to_string(index=False)
)

print("\nAnatomical occlusion patient-level statistics:")
print(
    patient_stats[
        patient_stats["region_name"].isin(
            [
                "OD",
                "OC",
                "Rim",
                "OD_plus_OC",
            ]
        )
    ][
        [
            "model_id",
            "branch_space",
            "region_name",
            "n_patients",
            "mean",
            "median",
            "ci95_low_percentile",
            "ci95_high_percentile",
            "wilcoxon_p_two_sided",
            "wilcoxon_p_greater",
            "wilcoxon_p_less",
        ]
    ].to_string(index=False)
)

print(f"\nAnatomical occlusion metrics CSV: {OCCLUSION_METRICS_CSV}")
print(f"Anatomical occlusion patient metrics CSV: {OCCLUSION_PATIENT_METRICS_CSV}")
print(f"Anatomical occlusion patient stats CSV: {OCCLUSION_PATIENT_STATS_CSV}")
print(f"Anatomical occlusion region summary CSV: {OCCLUSION_REGION_SUMMARY_CSV}")
print(f"Anatomical occlusion failure log CSV: {OCCLUSION_FAILURE_LOG_CSV}")
print(f"Anatomical occlusion visual cases CSV: {OCCLUSION_VISUAL_CASES_CSV}")
print(f"Anatomical occlusion visual panel PNG: {OCCLUSION_VISUAL_PANEL_PNG}")
print(f"Anatomical occlusion visual panel PDF: {OCCLUSION_VISUAL_PANEL_PDF}")
print(f"Anatomical occlusion summary JSON: {OCCLUSION_SUMMARY_JSON}")

required_outputs = [
    OCCLUSION_METRICS_CSV,
    OCCLUSION_PATIENT_METRICS_CSV,
    OCCLUSION_PATIENT_STATS_CSV,
    OCCLUSION_REGION_SUMMARY_CSV,
    OCCLUSION_FAILURE_LOG_CSV,
    OCCLUSION_VISUAL_CASES_CSV,
    OCCLUSION_SUMMARY_JSON,
]

if SAVE_VISUAL_PANEL:
    required_outputs.extend(
        [
            OCCLUSION_VISUAL_PANEL_PNG,
            OCCLUSION_VISUAL_PANEL_PDF,
        ]
    )

missing_outputs = [
    str(path)
    for path in required_outputs
    if not Path(path).exists()
]

if missing_outputs:
    raise RuntimeError(f"Missing Cell 9 outputs: {missing_outputs}")

if len(failure_df) > 0:
    raise RuntimeError("Anatomical occlusion completed with failures.")

if len(metrics_df) != len(priority_xai_manifest) * len(OCCLUSION_REGIONS):
    raise RuntimeError(
        f"Unexpected number of occlusion metric rows: "
        f"{len(metrics_df)} != {len(priority_xai_manifest) * len(OCCLUSION_REGIONS)}"
    )

print("Status: PASS")

In [ ]:
# Cell 10: Consolidate priority Multi-XAI evidence across Grad-CAM++, Integrated Gradients, and Anatomical Occlusion

import json
from pathlib import Path

import numpy as np
import pandas as pd

if "RESULTS_12_DIR" not in globals():
    raise RuntimeError("RESULTS_12_DIR is not defined. Rerun Cell 1 first.")

if "XAI_RUNTIME_ROOT" not in globals():
    XAI_RUNTIME_ROOT = RESULTS_12_DIR / "xai_runtime_priority_models"

MULTIXAI_CONSOLIDATION_ROOT = (
    XAI_RUNTIME_ROOT
    / "multixai_consolidated_priority_evidence"
)

MULTIXAI_CONSOLIDATION_ROOT.mkdir(parents=True, exist_ok=True)

GRADCAM_ROOT = XAI_RUNTIME_ROOT / "gradcamplusplus_priority_models"
IG_ROOT = XAI_RUNTIME_ROOT / "integrated_gradients_priority_models"
OCCLUSION_ROOT = XAI_RUNTIME_ROOT / "anatomical_occlusion_priority_models"

GRADCAM_PATIENT_STATS_CSV = (
    GRADCAM_ROOT
    / "notebook12_gradcamplusplus_priority_patient_level_statistics.csv"
)

IG_PATIENT_STATS_CSV = (
    IG_ROOT
    / "notebook12_integratedgradients_priority_patient_level_statistics.csv"
)

OCCLUSION_PATIENT_STATS_CSV = (
    OCCLUSION_ROOT
    / "notebook12_anatomical_occlusion_priority_patient_level_statistics.csv"
)

GRADCAM_REGION_SUMMARY_CSV = (
    GRADCAM_ROOT
    / "notebook12_gradcamplusplus_priority_region_summary.csv"
)

IG_REGION_SUMMARY_CSV = (
    IG_ROOT
    / "notebook12_integratedgradients_priority_region_summary.csv"
)

OCCLUSION_REGION_SUMMARY_CSV = (
    OCCLUSION_ROOT
    / "notebook12_anatomical_occlusion_priority_region_summary.csv"
)

MULTIXAI_PATIENT_STATS_CONSOLIDATED_CSV = (
    MULTIXAI_CONSOLIDATION_ROOT
    / "notebook12_multixai_priority_patient_statistics_consolidated.csv"
)

MULTIXAI_REGION_SUMMARY_CONSOLIDATED_CSV = (
    MULTIXAI_CONSOLIDATION_ROOT
    / "notebook12_multixai_priority_region_summary_consolidated.csv"
)

MULTIXAI_EVIDENCE_MATRIX_CSV = (
    MULTIXAI_CONSOLIDATION_ROOT
    / "notebook12_multixai_priority_evidence_matrix.csv"
)

MULTIXAI_BRANCH_SYNTHESIS_CSV = (
    MULTIXAI_CONSOLIDATION_ROOT
    / "notebook12_multixai_priority_branch_level_synthesis.csv"
)

MULTIXAI_MODEL_SYNTHESIS_CSV = (
    MULTIXAI_CONSOLIDATION_ROOT
    / "notebook12_multixai_priority_model_level_synthesis.csv"
)

MULTIXAI_SUMMARY_JSON = (
    MULTIXAI_CONSOLIDATION_ROOT
    / "notebook12_multixai_priority_consolidation_summary.json"
)

required_inputs = [
    GRADCAM_PATIENT_STATS_CSV,
    IG_PATIENT_STATS_CSV,
    OCCLUSION_PATIENT_STATS_CSV,
    GRADCAM_REGION_SUMMARY_CSV,
    IG_REGION_SUMMARY_CSV,
    OCCLUSION_REGION_SUMMARY_CSV,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not Path(path).exists()
]

if missing_inputs:
    raise FileNotFoundError(f"Missing Cell 10 inputs: {missing_inputs}")

FOCUS_REGIONS = [
    "OD",
    "OD_plus_OC",
    "Rim",
]

SECONDARY_REGIONS = [
    "OC",
]

ALL_ANATOMICAL_REGIONS = [
    "OD",
    "OC",
    "Rim",
    "OD_plus_OC",
]

PRIMARY_METHODS = [
    "GradCAMPlusPlus",
    "IntegratedGradients",
    "AnatomicalOcclusion",
]


def load_patient_stats(path, method_name):
    df = pd.read_csv(path)

    df["method_name"] = method_name

    required_cols = [
        "model_id",
        "branch_space",
        "method_name",
        "region_name",
        "n_patients",
        "mean",
        "median",
        "ci95_low_percentile",
        "ci95_high_percentile",
        "wilcoxon_p_two_sided",
        "wilcoxon_p_greater",
        "wilcoxon_p_less",
    ]

    missing_cols = [
        col for col in required_cols
        if col not in df.columns
    ]

    if missing_cols:
        raise ValueError(
            f"Missing columns in {path}: {missing_cols}. "
            f"Available columns: {list(df.columns)}"
        )

    return df[required_cols].copy()


def load_region_summary(path, method_name):
    df = pd.read_csv(path)
    df["method_name"] = method_name

    return df.copy()


def safe_float(value):
    try:
        if pd.isna(value):
            return np.nan

        return float(value)

    except Exception:
        return np.nan


def classify_xai_direction(row, alpha=0.05):
    mean_value = safe_float(row["mean"])
    p_greater = safe_float(row["wilcoxon_p_greater"])
    p_less = safe_float(row["wilcoxon_p_less"])

    if not np.isfinite(mean_value):
        return "not_evaluable"

    if np.isfinite(p_greater) and mean_value > 0 and p_greater < alpha:
        return "positive_anatomical_evidence"

    if np.isfinite(p_less) and mean_value < 0 and p_less < alpha:
        return "negative_anatomical_evidence"

    return "inconclusive_or_neutral"


def classify_strength(row, alpha=0.05):
    status = row["evidence_status"]

    if status not in [
        "positive_anatomical_evidence",
        "negative_anatomical_evidence",
    ]:
        return "none"

    if status == "positive_anatomical_evidence":
        p_value = safe_float(row["wilcoxon_p_greater"])
    else:
        p_value = safe_float(row["wilcoxon_p_less"])

    if not np.isfinite(p_value):
        return "none"

    if p_value < 0.001:
        return "strong"

    if p_value < 0.01:
        return "moderate"

    if p_value < alpha:
        return "weak"

    return "none"


def summarize_branch(model_id, branch_space, block):
    focus_block = block[
        block["region_name"].isin(FOCUS_REGIONS)
    ].copy()

    all_focus_tests = int(len(focus_block))

    positive_count = int(
        (
            focus_block["evidence_status"]
            == "positive_anatomical_evidence"
        ).sum()
    )

    negative_count = int(
        (
            focus_block["evidence_status"]
            == "negative_anatomical_evidence"
        ).sum()
    )

    inconclusive_count = int(
        (
            focus_block["evidence_status"]
            == "inconclusive_or_neutral"
        ).sum()
    )

    strong_positive_count = int(
        (
            (
                focus_block["evidence_status"]
                == "positive_anatomical_evidence"
            )
            & (focus_block["evidence_strength"] == "strong")
        ).sum()
    )

    methods_with_positive = sorted(
        focus_block.loc[
            focus_block["evidence_status"] == "positive_anatomical_evidence",
            "method_name",
        ].astype(str).unique().tolist()
    )

    regions_with_positive = sorted(
        focus_block.loc[
            focus_block["evidence_status"] == "positive_anatomical_evidence",
            "region_name",
        ].astype(str).unique().tolist()
    )

    positive_fraction = (
        positive_count / all_focus_tests
        if all_focus_tests > 0
        else np.nan
    )

    negative_fraction = (
        negative_count / all_focus_tests
        if all_focus_tests > 0
        else np.nan
    )

    return {
        "model_id": model_id,
        "branch_space": branch_space,
        "n_focus_method_region_tests": all_focus_tests,
        "positive_focus_evidence_count": positive_count,
        "negative_focus_evidence_count": negative_count,
        "inconclusive_focus_evidence_count": inconclusive_count,
        "strong_positive_focus_evidence_count": strong_positive_count,
        "positive_focus_evidence_fraction": positive_fraction,
        "negative_focus_evidence_fraction": negative_fraction,
        "methods_with_positive_focus_evidence": ";".join(methods_with_positive),
        "regions_with_positive_focus_evidence": ";".join(regions_with_positive),
    }


def assign_branch_role(row):
    model_id = row["model_id"]
    branch_space = row["branch_space"]

    if model_id == "anatomy__od_centered_crop_2p5x":
        return "primary_anatomically_parsimonious_candidate"

    if model_id == "anatomy__outside_od_only_trained_control":
        return "shortcut_control_not_clinical_candidate"

    if model_id == "hybrid__full_image_plus_od_centered_2p5x_effb0":
        if branch_space == "hybrid_full_image_branch":
            return "hybrid_global_context_branch"

        if branch_space == "hybrid_od_centered_branch":
            return "hybrid_anatomical_focus_branch"

    return "unspecified"


def assign_branch_interpretation(row):
    role = row["branch_role"]
    positive_fraction = safe_float(row["positive_focus_evidence_fraction"])
    negative_fraction = safe_float(row["negative_focus_evidence_fraction"])
    strong_positive_count = int(row["strong_positive_focus_evidence_count"])

    if not np.isfinite(positive_fraction):
        positive_fraction = 0.0

    if not np.isfinite(negative_fraction):
        negative_fraction = 0.0

    if role == "shortcut_control_not_clinical_candidate":
        if positive_fraction < 0.35:
            return (
                "Expected control behavior: limited OD/Rim/OD+OC evidence "
                "despite strong classifier performance, supporting shortcut-control interpretation."
            )

        return (
            "Unexpected control behavior: shortcut-control shows relevant anatomical "
            "evidence and should be manually reviewed."
        )

    if role == "primary_anatomically_parsimonious_candidate":
        if positive_fraction >= 0.55 and strong_positive_count >= 2:
            return (
                "Strongest anatomically supported candidate among current priority models."
            )

        if positive_fraction >= 0.35:
            return (
                "Partially supported anatomically; keep as candidate but interpret "
                "with method-specific caution."
            )

        return (
            "Limited anatomical support; not sufficient as final explainable model."
        )

    if role == "hybrid_anatomical_focus_branch":
        if positive_fraction >= 0.45:
            return (
                "Hybrid OD-centered branch shows meaningful anatomical support, "
                "but must be interpreted jointly with the weaker global branch."
            )

        if positive_fraction >= 0.25:
            return "Hybrid OD-centered branch shows mixed anatomical support."

        return "Hybrid OD-centered branch has limited anatomical support."

    if role == "hybrid_global_context_branch":
        if positive_fraction < 0.35:
            return (
                "Hybrid full-image branch carries performance but has limited direct "
                "OD/Rim anatomical evidence, indicating residual global-context or shortcut risk."
            )

        return (
            "Hybrid full-image branch shows some anatomical support, but remains less "
            "constrained than OD-centered inputs."
        )

    if positive_fraction >= 0.55:
        return "Anatomically supported."

    if negative_fraction >= 0.55:
        return "Negative anatomical evidence."

    return "Mixed or inconclusive anatomical evidence."


def summarize_model_from_branches(model_id, block):
    positive_total = int(block["positive_focus_evidence_count"].sum())
    negative_total = int(block["negative_focus_evidence_count"].sum())
    total_tests = int(block["n_focus_method_region_tests"].sum())

    positive_fraction = (
        positive_total / total_tests
        if total_tests > 0
        else np.nan
    )

    negative_fraction = (
        negative_total / total_tests
        if total_tests > 0
        else np.nan
    )

    branch_roles = ";".join(
        sorted(block["branch_role"].astype(str).unique().tolist())
    )

    branch_interpretations = " | ".join(
        block["branch_interpretation"].astype(str).tolist()
    )

    if model_id == "anatomy__od_centered_crop_2p5x":
        model_level_role = "primary_explainable_anatomy_constrained_candidate"

        if np.isfinite(positive_fraction) and positive_fraction >= 0.55:
            model_level_interpretation = (
                "Best current balance for anatomical plausibility: positive evidence "
                "across attribution and functional perturbation."
            )
        else:
            model_level_interpretation = (
                "Candidate remains useful, but anatomical evidence is not uniformly strong."
            )

    elif model_id == "hybrid__full_image_plus_od_centered_2p5x_effb0":
        model_level_role = "high_performance_hybrid_candidate_with_mixed_xai"

        model_level_interpretation = (
            "Hybrid preserves strong predictive performance and calibration, but Multi-XAI "
            "shows branch asymmetry: the OD-centered branch is more anatomically plausible "
            "than the full-image branch."
        )

    elif model_id == "anatomy__outside_od_only_trained_control":
        model_level_role = "shortcut_control"

        model_level_interpretation = (
            "Control model confirms that high internal performance can persist without "
            "visible OD anatomy; it must not be selected as clinical model."
        )

    else:
        model_level_role = "unclassified"
        model_level_interpretation = branch_interpretations

    return {
        "model_id": model_id,
        "model_level_role": model_level_role,
        "n_branches": int(block["branch_space"].nunique()),
        "total_focus_method_region_tests": total_tests,
        "positive_focus_evidence_total": positive_total,
        "negative_focus_evidence_total": negative_total,
        "positive_focus_evidence_fraction": positive_fraction,
        "negative_focus_evidence_fraction": negative_fraction,
        "branch_roles": branch_roles,
        "model_level_interpretation": model_level_interpretation,
    }


gradcam_stats = load_patient_stats(
    GRADCAM_PATIENT_STATS_CSV,
    "GradCAMPlusPlus",
)

ig_stats = load_patient_stats(
    IG_PATIENT_STATS_CSV,
    "IntegratedGradients",
)

occlusion_stats = load_patient_stats(
    OCCLUSION_PATIENT_STATS_CSV,
    "AnatomicalOcclusion",
)

patient_stats_all = pd.concat(
    [
        gradcam_stats,
        ig_stats,
        occlusion_stats,
    ],
    ignore_index=True,
)

patient_stats_all["region_name"] = patient_stats_all["region_name"].astype(str)
patient_stats_all["model_id"] = patient_stats_all["model_id"].astype(str)
patient_stats_all["branch_space"] = patient_stats_all["branch_space"].astype(str)
patient_stats_all["method_name"] = patient_stats_all["method_name"].astype(str)

patient_stats_all.to_csv(
    MULTIXAI_PATIENT_STATS_CONSOLIDATED_CSV,
    index=False,
)

gradcam_region = load_region_summary(
    GRADCAM_REGION_SUMMARY_CSV,
    "GradCAMPlusPlus",
)

ig_region = load_region_summary(
    IG_REGION_SUMMARY_CSV,
    "IntegratedGradients",
)

occlusion_region = load_region_summary(
    OCCLUSION_REGION_SUMMARY_CSV,
    "AnatomicalOcclusion",
)

region_summary_all = pd.concat(
    [
        gradcam_region,
        ig_region,
        occlusion_region,
    ],
    ignore_index=True,
)

region_summary_all.to_csv(
    MULTIXAI_REGION_SUMMARY_CONSOLIDATED_CSV,
    index=False,
)

evidence_matrix = patient_stats_all.copy()

evidence_matrix = evidence_matrix[
    evidence_matrix["region_name"].isin(ALL_ANATOMICAL_REGIONS)
].copy()

evidence_matrix["evidence_status"] = evidence_matrix.apply(
    classify_xai_direction,
    axis=1,
)

evidence_matrix["evidence_strength"] = evidence_matrix.apply(
    classify_strength,
    axis=1,
)

evidence_matrix["is_primary_region"] = evidence_matrix["region_name"].isin(
    FOCUS_REGIONS
)

evidence_matrix["is_secondary_region"] = evidence_matrix["region_name"].isin(
    SECONDARY_REGIONS
)

evidence_matrix.to_csv(
    MULTIXAI_EVIDENCE_MATRIX_CSV,
    index=False,
)

branch_synthesis_rows = []

for (model_id, branch_space), block in evidence_matrix.groupby(
    ["model_id", "branch_space"]
):
    branch_synthesis_rows.append(
        summarize_branch(
            model_id=model_id,
            branch_space=branch_space,
            block=block,
        )
    )

branch_synthesis = pd.DataFrame(branch_synthesis_rows)

branch_synthesis["branch_role"] = branch_synthesis.apply(
    assign_branch_role,
    axis=1,
)

branch_synthesis["branch_interpretation"] = branch_synthesis.apply(
    assign_branch_interpretation,
    axis=1,
)

branch_synthesis.to_csv(
    MULTIXAI_BRANCH_SYNTHESIS_CSV,
    index=False,
)

model_synthesis_rows = []

for model_id, block in branch_synthesis.groupby("model_id"):
    model_synthesis_rows.append(
        summarize_model_from_branches(
            model_id=model_id,
            block=block,
        )
    )

model_synthesis = pd.DataFrame(model_synthesis_rows)

model_synthesis.to_csv(
    MULTIXAI_MODEL_SYNTHESIS_CSV,
    index=False,
)

summary_payload = {
    "notebook": "12_integrated_internal_model_audit_and_freezing",
    "cell": "10_consolidate_priority_multixai_evidence",
    "status": "PASS",
    "methods_consolidated": PRIMARY_METHODS,
    "focus_regions": FOCUS_REGIONS,
    "secondary_regions": SECONDARY_REGIONS,
    "n_patient_stats_rows": int(len(patient_stats_all)),
    "n_evidence_rows": int(len(evidence_matrix)),
    "n_branch_synthesis_rows": int(len(branch_synthesis)),
    "n_model_synthesis_rows": int(len(model_synthesis)),
    "main_interpretation": {
        "od_centered_crop_2p5x": (
            "Primary anatomically parsimonious candidate after priority Multi-XAI."
        ),
        "hybrid": (
            "High-performance and calibrated candidate with mixed branch-level anatomical evidence."
        ),
        "outside_od_control": (
            "Shortcut-control showing why internal performance alone is insufficient."
        ),
    },
    "outputs": {
        "patient_stats_consolidated_csv": str(MULTIXAI_PATIENT_STATS_CONSOLIDATED_CSV),
        "region_summary_consolidated_csv": str(MULTIXAI_REGION_SUMMARY_CONSOLIDATED_CSV),
        "evidence_matrix_csv": str(MULTIXAI_EVIDENCE_MATRIX_CSV),
        "branch_synthesis_csv": str(MULTIXAI_BRANCH_SYNTHESIS_CSV),
        "model_synthesis_csv": str(MULTIXAI_MODEL_SYNTHESIS_CSV),
    },
}

with open(MULTIXAI_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary_payload, f, indent=2)

print("Priority Multi-XAI evidence matrix:")
print(
    evidence_matrix[
        evidence_matrix["region_name"].isin(FOCUS_REGIONS)
    ][
        [
            "model_id",
            "branch_space",
            "method_name",
            "region_name",
            "mean",
            "wilcoxon_p_greater",
            "wilcoxon_p_less",
            "evidence_status",
            "evidence_strength",
        ]
    ].to_string(index=False)
)

print("\nBranch-level Multi-XAI synthesis:")
print(
    branch_synthesis[
        [
            "model_id",
            "branch_space",
            "branch_role",
            "positive_focus_evidence_count",
            "negative_focus_evidence_count",
            "positive_focus_evidence_fraction",
            "methods_with_positive_focus_evidence",
            "regions_with_positive_focus_evidence",
            "branch_interpretation",
        ]
    ].to_string(index=False)
)

print("\nModel-level Multi-XAI synthesis:")
print(
    model_synthesis[
        [
            "model_id",
            "model_level_role",
            "positive_focus_evidence_total",
            "negative_focus_evidence_total",
            "positive_focus_evidence_fraction",
            "model_level_interpretation",
        ]
    ].to_string(index=False)
)

print(f"\nConsolidated patient stats CSV: {MULTIXAI_PATIENT_STATS_CONSOLIDATED_CSV}")
print(f"Consolidated region summary CSV: {MULTIXAI_REGION_SUMMARY_CONSOLIDATED_CSV}")
print(f"Evidence matrix CSV: {MULTIXAI_EVIDENCE_MATRIX_CSV}")
print(f"Branch synthesis CSV: {MULTIXAI_BRANCH_SYNTHESIS_CSV}")
print(f"Model synthesis CSV: {MULTIXAI_MODEL_SYNTHESIS_CSV}")
print(f"Summary JSON: {MULTIXAI_SUMMARY_JSON}")

required_outputs = [
    MULTIXAI_PATIENT_STATS_CONSOLIDATED_CSV,
    MULTIXAI_REGION_SUMMARY_CONSOLIDATED_CSV,
    MULTIXAI_EVIDENCE_MATRIX_CSV,
    MULTIXAI_BRANCH_SYNTHESIS_CSV,
    MULTIXAI_MODEL_SYNTHESIS_CSV,
    MULTIXAI_SUMMARY_JSON,
]

missing_outputs = [
    str(path)
    for path in required_outputs
    if not Path(path).exists()
]

if missing_outputs:
    raise RuntimeError(f"Missing Cell 10 outputs: {missing_outputs}")

if len(model_synthesis) < 3:
    raise RuntimeError("Expected at least three model-level synthesis rows.")

if not set(PRIMARY_METHODS).issubset(set(evidence_matrix["method_name"].unique())):
    raise RuntimeError("Not all expected Multi-XAI methods were consolidated.")

print("Status: PASS")

In [ ]:
# Cell 11: Final internal model-freezing decision before external validation

import json
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path(os.environ.get("AQPR_PROJECT_ROOT", "/content/drive/MyDrive/AQPR_Glaucoma"))
RESULTS_ROOT = PROJECT_ROOT / "results"
MODELS_ROOT = PROJECT_ROOT / "models"
PROJECT_DATA_ROOT = PROJECT_ROOT / "project_data"

RESULTS_10_DIR = RESULTS_ROOT / "10_anatomy_constrained_classifier_training"
RESULTS_11_DIR = RESULTS_ROOT / "11_hybrid_anatomy_aware_classifier_training"
RESULTS_12_DIR = RESULTS_ROOT / "12_integrated_internal_model_audit_and_freezing"

XAI_RUNTIME_ROOT = RESULTS_12_DIR / "xai_runtime_priority_models"

MULTIXAI_CONSOLIDATION_ROOT = (
    XAI_RUNTIME_ROOT
    / "multixai_consolidated_priority_evidence"
)

FINAL_FREEZE_ROOT = RESULTS_12_DIR / "final_internal_freezing_decision"
FINAL_FREEZE_ROOT.mkdir(parents=True, exist_ok=True)

INTERNAL_CANDIDATE_REGISTRY_CSV = (
    RESULTS_12_DIR
    / "notebook12_internal_candidate_registry.csv"
)

MULTIXAI_MODEL_SYNTHESIS_CSV = (
    MULTIXAI_CONSOLIDATION_ROOT
    / "notebook12_multixai_priority_model_level_synthesis.csv"
)

MULTIXAI_BRANCH_SYNTHESIS_CSV = (
    MULTIXAI_CONSOLIDATION_ROOT
    / "notebook12_multixai_priority_branch_level_synthesis.csv"
)

MULTIXAI_EVIDENCE_MATRIX_CSV = (
    MULTIXAI_CONSOLIDATION_ROOT
    / "notebook12_multixai_priority_evidence_matrix.csv"
)

ANATOMY_LOCKED_TEST_PROGRESS_CSV = (
    RESULTS_10_DIR
    / "anatomy_constrained_locked_test_evaluation"
    / "notebook10_anatomy_constrained_locked_test_evaluation_progress.csv"
)

HYBRID_LOCKED_TEST_PATIENT_METRICS_CSV = (
    RESULTS_11_DIR
    / "hybrid_locked_test_evaluation"
    / "notebook11_hybrid_test_patient_level_metrics.csv"
)

HYBRID_LOCKED_TEST_IMAGE_METRICS_CSV = (
    RESULTS_11_DIR
    / "hybrid_locked_test_evaluation"
    / "notebook11_hybrid_test_image_level_metrics.csv"
)

HYBRID_SELECTED_MODEL_JSON = (
    RESULTS_11_DIR
    / "notebook11_validation_selected_hybrid_model.json"
)

ANATOMY_SELECTED_MODEL_JSON = (
    RESULTS_10_DIR
    / "notebook10_validation_selected_anatomy_constrained_model.json"
)

FINAL_DECISION_CSV = (
    FINAL_FREEZE_ROOT
    / "notebook12_final_internal_model_freezing_decision.csv"
)

FROZEN_MODEL_REGISTRY_CSV = (
    FINAL_FREEZE_ROOT
    / "notebook12_frozen_model_registry.csv"
)

FROZEN_INPUT_CONTRACTS_CSV = (
    FINAL_FREEZE_ROOT
    / "notebook12_frozen_thresholds_and_input_contracts.csv"
)

EXTERNAL_VALIDATION_READINESS_CSV = (
    FINAL_FREEZE_ROOT
    / "notebook12_external_validation_readiness_checklist.csv"
)

FINAL_DECISION_SUMMARY_JSON = (
    FINAL_FREEZE_ROOT
    / "notebook12_final_internal_freezing_summary.json"
)

FINAL_DECISION_TXT = (
    FINAL_FREEZE_ROOT
    / "notebook12_FINAL_FREEZING_DECISION.txt"
)

NO_FURTHER_TUNING_POLICY_TXT = (
    FINAL_FREEZE_ROOT
    / "notebook12_NO_FURTHER_TUNING_POLICY.txt"
)

required_inputs = [
    INTERNAL_CANDIDATE_REGISTRY_CSV,
    MULTIXAI_MODEL_SYNTHESIS_CSV,
    MULTIXAI_BRANCH_SYNTHESIS_CSV,
    MULTIXAI_EVIDENCE_MATRIX_CSV,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not Path(path).exists()
]

if missing_inputs:
    raise FileNotFoundError(f"Missing Cell 11 inputs: {missing_inputs}")

candidate_registry = pd.read_csv(INTERNAL_CANDIDATE_REGISTRY_CSV)
multixai_model = pd.read_csv(MULTIXAI_MODEL_SYNTHESIS_CSV)
multixai_branch = pd.read_csv(MULTIXAI_BRANCH_SYNTHESIS_CSV)
multixai_evidence = pd.read_csv(MULTIXAI_EVIDENCE_MATRIX_CSV)

candidate_registry["model_id"] = candidate_registry["model_id"].astype(str)
multixai_model["model_id"] = multixai_model["model_id"].astype(str)

for col in [
    "patient_auroc",
    "patient_balanced_accuracy",
    "patient_brier_score",
]:
    if col not in candidate_registry.columns:
        raise ValueError(
            f"Missing required candidate metric column: {col}. "
            f"Available columns: {list(candidate_registry.columns)}"
        )

decision_df = candidate_registry.merge(
    multixai_model,
    on="model_id",
    how="left",
)

decision_df["positive_focus_evidence_fraction"] = decision_df[
    "positive_focus_evidence_fraction"
].fillna(0.0)

decision_df["negative_focus_evidence_fraction"] = decision_df[
    "negative_focus_evidence_fraction"
].fillna(0.0)

decision_df["positive_focus_evidence_total"] = decision_df[
    "positive_focus_evidence_total"
].fillna(0).astype(int)

decision_df["negative_focus_evidence_total"] = decision_df[
    "negative_focus_evidence_total"
].fillna(0).astype(int)

decision_df["model_level_role"] = decision_df["model_level_role"].fillna(
    "not_priority_multixai_model"
)

decision_df["model_level_interpretation"] = decision_df[
    "model_level_interpretation"
].fillna(
    "Model retained as internal comparator or sensitivity model; not part of priority Multi-XAI freezing set."
)

decision_df["passes_patient_auroc_threshold"] = (
    decision_df["patient_auroc"].astype(float) >= 0.95
)

decision_df["passes_patient_balanced_accuracy_threshold"] = (
    decision_df["patient_balanced_accuracy"].astype(float) >= 0.90
)

decision_df["passes_patient_brier_threshold"] = (
    decision_df["patient_brier_score"].astype(float) <= 0.06
)

decision_df["passes_internal_performance_gate"] = (
    decision_df["passes_patient_auroc_threshold"]
    & decision_df["passes_patient_balanced_accuracy_threshold"]
    & decision_df["passes_patient_brier_threshold"]
)

decision_df["passes_priority_multixai_gate"] = (
    decision_df["positive_focus_evidence_fraction"].astype(float) >= 0.55
)

decision_df["has_shortcut_control_role"] = (
    decision_df["model_level_role"].astype(str).eq("shortcut_control")
    | decision_df.get("model_family", pd.Series("", index=decision_df.index)).astype(str).eq("shortcut_control")
    | decision_df["model_id"].astype(str).str.contains("outside_od_only", regex=False)
)

decision_df["is_hybrid_model"] = decision_df["model_id"].astype(str).str.startswith(
    "hybrid__"
)

decision_df["is_od_centered_primary_candidate"] = decision_df["model_id"].astype(str).eq(
    "anatomy__od_centered_crop_2p5x"
)

decision_df["is_full_image_model"] = decision_df["model_id"].astype(str).str.startswith(
    "full_image__"
)

decision_df["is_anatomy_stress_test"] = (
    decision_df["model_id"].astype(str).str.contains("od_only_context_suppressed", regex=False)
)

decision_df["explainable_freezing_score"] = (
    0.30 * decision_df["patient_auroc"].astype(float)
    + 0.25 * decision_df["patient_balanced_accuracy"].astype(float)
    - 0.10 * decision_df["patient_brier_score"].astype(float)
    + 0.35 * decision_df["positive_focus_evidence_fraction"].astype(float)
)

decision_df.loc[
    decision_df["has_shortcut_control_role"],
    "explainable_freezing_score",
] = decision_df.loc[
    decision_df["has_shortcut_control_role"],
    "explainable_freezing_score",
] - 1.0

decision_df.loc[
    decision_df["is_full_image_model"],
    "explainable_freezing_score",
] = decision_df.loc[
    decision_df["is_full_image_model"],
    "explainable_freezing_score",
] - 0.25

decision_df.loc[
    decision_df["is_anatomy_stress_test"],
    "explainable_freezing_score",
] = decision_df.loc[
    decision_df["is_anatomy_stress_test"],
    "explainable_freezing_score",
] - 0.50


def assign_freezing_status(row):
    model_id = str(row["model_id"])

    if model_id == "anatomy__od_centered_crop_2p5x":
        if bool(row["passes_internal_performance_gate"]) and bool(row["passes_priority_multixai_gate"]):
            return "freeze_as_primary_explainable_external_validation_model"

        return "retain_but_not_primary_due_to_failed_gate"

    if model_id == "hybrid__full_image_plus_od_centered_2p5x_effb0":
        if bool(row["passes_internal_performance_gate"]):
            return "freeze_as_secondary_high_performance_hybrid_sensitivity_model"

        return "retain_hybrid_but_not_frozen_due_to_failed_performance_gate"

    if model_id == "anatomy__outside_od_only_trained_control":
        return "freeze_as_shortcut_control_only_not_clinical_candidate"

    if model_id == "anatomy__od_centered_crop_4p0x":
        return "retain_as_context_sensitivity_comparator"

    if model_id == "anatomy__od_only_context_suppressed":
        return "retain_as_anatomy_stress_test_not_primary"

    if model_id.startswith("full_image__"):
        return "retain_as_full_image_internal_comparator_not_final"

    return "retain_as_internal_reference"


decision_df["freezing_status"] = decision_df.apply(
    assign_freezing_status,
    axis=1,
)


def assign_external_validation_role(row):
    status = row["freezing_status"]

    if status == "freeze_as_primary_explainable_external_validation_model":
        return "primary_model_for_external_validation"

    if status == "freeze_as_secondary_high_performance_hybrid_sensitivity_model":
        return "secondary_sensitivity_model_for_external_validation"

    if status == "freeze_as_shortcut_control_only_not_clinical_candidate":
        return "shortcut_control_report_internally_not_primary_external_model"

    return "internal_comparator_or_sensitivity_only"


decision_df["external_validation_role"] = decision_df.apply(
    assign_external_validation_role,
    axis=1,
)

decision_df = decision_df.sort_values(
    [
        "external_validation_role",
        "explainable_freezing_score",
    ],
    ascending=[True, False],
).reset_index(drop=True)

decision_df.to_csv(FINAL_DECISION_CSV, index=False)

primary_model_id = "anatomy__od_centered_crop_2p5x"
secondary_model_id = "hybrid__full_image_plus_od_centered_2p5x_effb0"
shortcut_control_model_id = "anatomy__outside_od_only_trained_control"

if primary_model_id not in set(decision_df["model_id"]):
    raise RuntimeError(f"Primary model missing from decision table: {primary_model_id}")

if secondary_model_id not in set(decision_df["model_id"]):
    raise RuntimeError(f"Hybrid model missing from decision table: {secondary_model_id}")

if shortcut_control_model_id not in set(decision_df["model_id"]):
    raise RuntimeError(f"Shortcut-control model missing from decision table: {shortcut_control_model_id}")


def load_json_if_exists(path):
    path = Path(path)

    if not path.exists():
        return {}

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def get_anatomy_thresholds(experiment_id):
    if ANATOMY_LOCKED_TEST_PROGRESS_CSV.exists():
        progress = pd.read_csv(ANATOMY_LOCKED_TEST_PROGRESS_CSV)
        progress["experiment_id"] = progress["experiment_id"].astype(str)

        match = progress[
            progress["experiment_id"].eq(experiment_id)
        ]

        if len(match) == 1:
            row = match.iloc[0]

            return {
                "image_threshold": float(row.get("image_threshold_from_validation", np.nan)),
                "patient_threshold": float(row.get("patient_threshold_from_validation", np.nan)),
                "checkpoint_path": str(row.get("checkpoint_path", "")),
            }

    anatomy_selected = load_json_if_exists(ANATOMY_SELECTED_MODEL_JSON)

    return {
        "image_threshold": np.nan,
        "patient_threshold": float(anatomy_selected.get("selected_best_patient_val_threshold", np.nan)),
        "checkpoint_path": str(anatomy_selected.get("selected_best_checkpoint", "")),
    }


def get_hybrid_thresholds():
    hybrid_selected = load_json_if_exists(HYBRID_SELECTED_MODEL_JSON)

    image_threshold = np.nan
    patient_threshold = float(hybrid_selected.get("selected_best_patient_val_threshold", np.nan))
    checkpoint_path = str(hybrid_selected.get("selected_best_checkpoint", ""))

    if HYBRID_LOCKED_TEST_IMAGE_METRICS_CSV.exists():
        image_metrics = pd.read_csv(HYBRID_LOCKED_TEST_IMAGE_METRICS_CSV)

        if "threshold" in image_metrics.columns and len(image_metrics) > 0:
            image_threshold = float(image_metrics["threshold"].iloc[0])

    if HYBRID_LOCKED_TEST_PATIENT_METRICS_CSV.exists():
        patient_metrics = pd.read_csv(HYBRID_LOCKED_TEST_PATIENT_METRICS_CSV)

        if "threshold" in patient_metrics.columns and len(patient_metrics) > 0:
            patient_threshold = float(patient_metrics["threshold"].iloc[0])

    return {
        "image_threshold": image_threshold,
        "patient_threshold": patient_threshold,
        "checkpoint_path": checkpoint_path,
    }


primary_thresholds = get_anatomy_thresholds("od_centered_crop_2p5x")
od4_thresholds = get_anatomy_thresholds("od_centered_crop_4p0x")
outside_thresholds = get_anatomy_thresholds("outside_od_only_trained_control")
hybrid_thresholds = get_hybrid_thresholds()

frozen_registry_rows = [
    {
        "frozen_role": "primary_explainable_model",
        "model_id": primary_model_id,
        "model_family": "anatomy_constrained",
        "architecture": "efficientnet_b0",
        "input_contract_id": "od_centered_crop_2p5x_384",
        "checkpoint_path": primary_thresholds["checkpoint_path"],
        "image_threshold_from_validation": primary_thresholds["image_threshold"],
        "patient_threshold_from_validation": primary_thresholds["patient_threshold"],
        "patient_aggregation": "mean_probability",
        "external_validation_use": "primary",
        "allowed_for_clinical_candidate_claim": True,
        "selection_reason": (
            "Best internal anatomical plausibility after Multi-XAI, with positive OD/OD+OC/Rim evidence "
            "across Grad-CAM++, Integrated Gradients, and Anatomical Occlusion."
        ),
    },
    {
        "frozen_role": "secondary_hybrid_sensitivity_model",
        "model_id": secondary_model_id,
        "model_family": "hybrid_anatomy_aware",
        "architecture": "efficientnet_b0_plus_efficientnet_b0_late_feature_fusion",
        "input_contract_id": "full_image_384_plus_od_centered_crop_2p5x_384",
        "checkpoint_path": hybrid_thresholds["checkpoint_path"],
        "image_threshold_from_validation": hybrid_thresholds["image_threshold"],
        "patient_threshold_from_validation": hybrid_thresholds["patient_threshold"],
        "patient_aggregation": "mean_probability",
        "external_validation_use": "secondary_sensitivity",
        "allowed_for_clinical_candidate_claim": False,
        "selection_reason": (
            "High internal performance and calibration, but mixed branch-level Multi-XAI; retained as "
            "secondary sensitivity model, not as the primary explainable model."
        ),
    },
    {
        "frozen_role": "shortcut_control",
        "model_id": shortcut_control_model_id,
        "model_family": "shortcut_control",
        "architecture": "efficientnet_b0",
        "input_contract_id": "outside_od_only_trained_control_384",
        "checkpoint_path": outside_thresholds["checkpoint_path"],
        "image_threshold_from_validation": outside_thresholds["image_threshold"],
        "patient_threshold_from_validation": outside_thresholds["patient_threshold"],
        "patient_aggregation": "mean_probability",
        "external_validation_use": "internal_shortcut_control_only",
        "allowed_for_clinical_candidate_claim": False,
        "selection_reason": (
            "Control model demonstrates that high internal performance can persist without visible OD anatomy; "
            "used to support shortcut-risk interpretation."
        ),
    },
]

frozen_registry = pd.DataFrame(frozen_registry_rows)
frozen_registry.to_csv(FROZEN_MODEL_REGISTRY_CSV, index=False)

input_contract_rows = [
    {
        "input_contract_id": "od_centered_crop_2p5x_384",
        "model_id": primary_model_id,
        "input_size": 384,
        "input_definition": "Square crop centered on the OD proxy using 2.5x OD-equivalent diameter context.",
        "preprocessing_source": "notebook10_strict_anatomy_constrained_input_manifest.csv",
        "requires_od_proxy": True,
        "requires_oc_proxy": False,
        "requires_full_image": False,
        "patient_aggregation": "mean_probability",
        "image_threshold_from_validation": primary_thresholds["image_threshold"],
        "patient_threshold_from_validation": primary_thresholds["patient_threshold"],
        "external_policy": "No threshold tuning on external validation data.",
    },
    {
        "input_contract_id": "full_image_384_plus_od_centered_crop_2p5x_384",
        "model_id": secondary_model_id,
        "input_size": 384,
        "input_definition": "Two-branch input: full fundus image plus OD-centered 2.5x crop.",
        "preprocessing_source": "notebook11_hybrid_full_image_od25x_paired_input_manifest.csv",
        "requires_od_proxy": True,
        "requires_oc_proxy": False,
        "requires_full_image": True,
        "patient_aggregation": "mean_probability",
        "image_threshold_from_validation": hybrid_thresholds["image_threshold"],
        "patient_threshold_from_validation": hybrid_thresholds["patient_threshold"],
        "external_policy": "Secondary sensitivity only; no threshold tuning on external validation data.",
    },
    {
        "input_contract_id": "outside_od_only_trained_control_384",
        "model_id": shortcut_control_model_id,
        "input_size": 384,
        "input_definition": "Full image with OD region neutralized; shortcut-control input.",
        "preprocessing_source": "notebook10_outside_od_only_trained_control_input_manifest.csv",
        "requires_od_proxy": True,
        "requires_oc_proxy": False,
        "requires_full_image": True,
        "patient_aggregation": "mean_probability",
        "image_threshold_from_validation": outside_thresholds["image_threshold"],
        "patient_threshold_from_validation": outside_thresholds["patient_threshold"],
        "external_policy": "Internal shortcut-control only; not a clinical model.",
    },
]

input_contracts = pd.DataFrame(input_contract_rows)
input_contracts.to_csv(FROZEN_INPUT_CONTRACTS_CSV, index=False)

readiness_rows = [
    {
        "check_name": "primary_model_selected",
        "status": primary_model_id in set(frozen_registry["model_id"]),
        "details": primary_model_id,
    },
    {
        "check_name": "secondary_hybrid_sensitivity_model_selected",
        "status": secondary_model_id in set(frozen_registry["model_id"]),
        "details": secondary_model_id,
    },
    {
        "check_name": "shortcut_control_frozen_but_not_clinical",
        "status": shortcut_control_model_id in set(frozen_registry["model_id"]),
        "details": shortcut_control_model_id,
    },
    {
        "check_name": "primary_model_passes_internal_performance_gate",
        "status": bool(
            decision_df.loc[
                decision_df["model_id"].eq(primary_model_id),
                "passes_internal_performance_gate",
            ].iloc[0]
        ),
        "details": "AUROC>=0.95, balanced accuracy>=0.90, Brier<=0.06.",
    },
    {
        "check_name": "primary_model_passes_multixai_gate",
        "status": bool(
            decision_df.loc[
                decision_df["model_id"].eq(primary_model_id),
                "passes_priority_multixai_gate",
            ].iloc[0]
        ),
        "details": "Positive focus-region Multi-XAI fraction >= 0.55.",
    },
    {
        "check_name": "shortcut_control_not_selected_as_clinical_candidate",
        "status": not bool(
            frozen_registry.loc[
                frozen_registry["model_id"].eq(shortcut_control_model_id),
                "allowed_for_clinical_candidate_claim",
            ].iloc[0]
        ),
        "details": "Outside-OD model is retained only as a shortcut-control.",
    },
    {
        "check_name": "thresholds_frozen_before_external_validation",
        "status": bool(
            input_contracts["patient_threshold_from_validation"]
            .replace([np.inf, -np.inf], np.nan)
            .notna()
            .all()
        ),
        "details": "Patient thresholds are validation-selected and frozen before external testing.",
    },
    {
        "check_name": "no_external_dataset_used_for_model_selection",
        "status": True,
        "details": "External validation must be performed only after this freezing decision.",
    },
    {
        "check_name": "no_further_internal_model_selection_after_freezing",
        "status": True,
        "details": "Further training or threshold tuning invalidates this frozen decision.",
    },
]

readiness_checklist = pd.DataFrame(readiness_rows)
readiness_checklist.to_csv(EXTERNAL_VALIDATION_READINESS_CSV, index=False)

primary_row = decision_df[
    decision_df["model_id"].eq(primary_model_id)
].iloc[0]

hybrid_row = decision_df[
    decision_df["model_id"].eq(secondary_model_id)
].iloc[0]

shortcut_row = decision_df[
    decision_df["model_id"].eq(shortcut_control_model_id)
].iloc[0]

summary_payload = {
    "notebook": "12_integrated_internal_model_audit_and_freezing",
    "cell": "11_final_internal_model_freezing_decision",
    "status": "PASS",
    "primary_frozen_model": primary_model_id,
    "secondary_frozen_model": secondary_model_id,
    "shortcut_control_model": shortcut_control_model_id,
    "primary_model_internal_metrics": {
        "patient_auroc": float(primary_row["patient_auroc"]),
        "patient_balanced_accuracy": float(primary_row["patient_balanced_accuracy"]),
        "patient_brier_score": float(primary_row["patient_brier_score"]),
        "positive_focus_evidence_fraction": float(primary_row["positive_focus_evidence_fraction"]),
        "negative_focus_evidence_fraction": float(primary_row["negative_focus_evidence_fraction"]),
    },
    "hybrid_internal_metrics": {
        "patient_auroc": float(hybrid_row["patient_auroc"]),
        "patient_balanced_accuracy": float(hybrid_row["patient_balanced_accuracy"]),
        "patient_brier_score": float(hybrid_row["patient_brier_score"]),
        "positive_focus_evidence_fraction": float(hybrid_row["positive_focus_evidence_fraction"]),
        "negative_focus_evidence_fraction": float(hybrid_row["negative_focus_evidence_fraction"]),
    },
    "shortcut_control_internal_metrics": {
        "patient_auroc": float(shortcut_row["patient_auroc"]),
        "patient_balanced_accuracy": float(shortcut_row["patient_balanced_accuracy"]),
        "patient_brier_score": float(shortcut_row["patient_brier_score"]),
        "positive_focus_evidence_fraction": float(shortcut_row["positive_focus_evidence_fraction"]),
        "negative_focus_evidence_fraction": float(shortcut_row["negative_focus_evidence_fraction"]),
    },
    "freezing_policy": {
        "external_validation_dataset_use": "single_external_glaucoma_labeled_dataset_after_freezing",
        "threshold_policy": "locked_validation_selected_thresholds_only",
        "no_further_tuning": True,
        "patient_aggregation": "mean_probability",
    },
    "main_interpretation": (
        "The OD-centered 2.5x EfficientNet-B0 is frozen as the primary explainable model because it "
        "preserves internal patient-level performance while showing the most consistent anatomical "
        "evidence across Grad-CAM++, Integrated Gradients, and Anatomical Occlusion. The hybrid model "
        "is frozen only as a secondary high-performance sensitivity model because its Multi-XAI evidence "
        "is branch-asymmetric. The outside-OD model is frozen only as a shortcut-control."
    ),
    "outputs": {
        "decision_csv": str(FINAL_DECISION_CSV),
        "frozen_model_registry_csv": str(FROZEN_MODEL_REGISTRY_CSV),
        "frozen_input_contracts_csv": str(FROZEN_INPUT_CONTRACTS_CSV),
        "external_validation_readiness_csv": str(EXTERNAL_VALIDATION_READINESS_CSV),
        "final_decision_txt": str(FINAL_DECISION_TXT),
        "no_further_tuning_policy_txt": str(NO_FURTHER_TUNING_POLICY_TXT),
    },
}

with open(FINAL_DECISION_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary_payload, f, indent=2)

final_txt = f"""
FINAL INTERNAL MODEL-FREEZING DECISION
Notebook: 12_integrated_internal_model_audit_and_freezing

Primary explainable model frozen for external validation:
{primary_model_id}

Primary model rationale:
- Architecture: EfficientNet-B0.
- Input: OD-centered 2.5x crop at 384 x 384.
- Patient-level AUROC: {float(primary_row["patient_auroc"]):.6f}
- Patient-level balanced accuracy: {float(primary_row["patient_balanced_accuracy"]):.6f}
- Patient-level Brier score: {float(primary_row["patient_brier_score"]):.6f}
- Positive Multi-XAI focus evidence fraction: {float(primary_row["positive_focus_evidence_fraction"]):.6f}
- Negative Multi-XAI focus evidence fraction: {float(primary_row["negative_focus_evidence_fraction"]):.6f}

Secondary high-performance sensitivity model:
{secondary_model_id}

Secondary model rationale:
- High internal discrimination and calibration.
- Retained for sensitivity analysis.
- Not used as primary explainable model because Multi-XAI evidence is branch-asymmetric.

Shortcut-control model:
{shortcut_control_model_id}

Shortcut-control rationale:
- Demonstrates that high internal performance can persist without visible OD anatomy.
- Not allowed for clinical candidate claims.

External validation policy:
- Use only frozen checkpoints.
- Use only frozen preprocessing contracts.
- Use only validation-selected thresholds.
- Do not tune thresholds, hyperparameters, architecture, preprocessing, or model choice on the external dataset.
- Report external validation as confirmatory, not exploratory.
""".strip()

with open(FINAL_DECISION_TXT, "w", encoding="utf-8") as f:
    f.write(final_txt + "\n")

no_further_tuning_text = """
NO FURTHER TUNING POLICY

After this freezing decision, no additional model selection, threshold tuning, hyperparameter tuning,
architecture selection, preprocessing selection, or XAI-driven model switching is allowed before external
validation.

External validation must use:
1. The frozen primary model: anatomy__od_centered_crop_2p5x.
2. The frozen secondary sensitivity model: hybrid__full_image_plus_od_centered_2p5x_effb0.
3. The frozen shortcut-control model only as an internal interpretability/control reference.
4. The frozen patient-level aggregation method: mean probability.
5. The frozen validation-selected patient thresholds.

Any additional model tuning must be reported as a new exploratory experiment and cannot be mixed with
the confirmatory external validation claim.
""".strip()

with open(NO_FURTHER_TUNING_POLICY_TXT, "w", encoding="utf-8") as f:
    f.write(no_further_tuning_text + "\n")

print("Final internal model-freezing decision:")
print(f"Primary explainable model: {primary_model_id}")
print(f"Secondary hybrid sensitivity model: {secondary_model_id}")
print(f"Shortcut-control model: {shortcut_control_model_id}")

print("\nFrozen model registry:")
print(
    frozen_registry[
        [
            "frozen_role",
            "model_id",
            "architecture",
            "input_contract_id",
            "patient_threshold_from_validation",
            "external_validation_use",
            "allowed_for_clinical_candidate_claim",
        ]
    ].to_string(index=False)
)

print("\nDecision table:")
print(
    decision_df[
        [
            "model_id",
            "model_family",
            "model_role",
            "patient_auroc",
            "patient_balanced_accuracy",
            "patient_brier_score",
            "positive_focus_evidence_fraction",
            "negative_focus_evidence_fraction",
            "passes_internal_performance_gate",
            "passes_priority_multixai_gate",
            "freezing_status",
            "external_validation_role",
        ]
    ].to_string(index=False)
)

print("\nExternal validation readiness:")
print(readiness_checklist.to_string(index=False))

print(f"\nDecision CSV: {FINAL_DECISION_CSV}")
print(f"Frozen model registry CSV: {FROZEN_MODEL_REGISTRY_CSV}")
print(f"Frozen input contracts CSV: {FROZEN_INPUT_CONTRACTS_CSV}")
print(f"External validation readiness CSV: {EXTERNAL_VALIDATION_READINESS_CSV}")
print(f"Final summary JSON: {FINAL_DECISION_SUMMARY_JSON}")
print(f"Final decision TXT: {FINAL_DECISION_TXT}")
print(f"No further tuning policy TXT: {NO_FURTHER_TUNING_POLICY_TXT}")

required_outputs = [
    FINAL_DECISION_CSV,
    FROZEN_MODEL_REGISTRY_CSV,
    FROZEN_INPUT_CONTRACTS_CSV,
    EXTERNAL_VALIDATION_READINESS_CSV,
    FINAL_DECISION_SUMMARY_JSON,
    FINAL_DECISION_TXT,
    NO_FURTHER_TUNING_POLICY_TXT,
]

missing_outputs = [
    str(path)
    for path in required_outputs
    if not Path(path).exists()
]

if missing_outputs:
    raise RuntimeError(f"Missing Cell 11 outputs: {missing_outputs}")

if not bool(readiness_checklist["status"].all()):
    failing_checks = readiness_checklist[
        ~readiness_checklist["status"].astype(bool)
    ]
    print(failing_checks.to_string(index=False))
    raise RuntimeError("External validation readiness checklist has failing checks.")

primary_status = decision_df.loc[
    decision_df["model_id"].eq(primary_model_id),
    "freezing_status",
].iloc[0]

if primary_status != "freeze_as_primary_explainable_external_validation_model":
    raise RuntimeError(f"Primary model was not frozen correctly: {primary_status}")

print("Status: PASS")